# WCNC Agentic Digital Twin Master Notebook

This notebook consolidates the complete implementation used for the site-specific **Agentic Digital Twin-Assisted Predictive Handover** study. It is organized as a single executable workflow:

1. Generate and validate the KFUPM Digital Twin data;
2. Construct leakage-free leave-one-trajectory-out (LOTO) folds;
3. Train and evaluate the forecasting models;
4. Run the Forecasting, Planning, and Adaptation Agents;
5. Compare Event A3, persistence, fixed DP--TTT, and the Proposed Agentic Twin;
6. Generate publication-quality figures.

The paper equations are referenced in the explanatory cells. The main links are:

- Eq. (1): ray-traced channel construction;
- Eq. (2): valid-path indicator and path-power feature;
- Eq. (3): synchronized multimodal observation;
- Eq. (5): available handover actions;
- Eq. (6): handover objective;
- Eq. (7): Forecasting Agent model selection;
- Eq. (8): finite-horizon DP action-sequence optimization;
- Eqs. (9)--(10): execution validation and Adaptation Agent state update.

**Execution note:** the Digital Twin ray-tracing section is computationally expensive and requires the KFUPM Sionna RT scene, CUDA, Mitsuba, and Dr.Jit. If the synchronized dataset already exists, start from the LOTO section. The later sections can be rerun using saved results without regenerating the RF data.


## 0. Conceptual map of the implementation

The implementation follows the paper's runtime loop:

```text
Digital Twin observation o_t
        ↓
Forecasting Agent: select a forecaster and produce future predictions
        ↓
Planning Agent: evaluate feasible stay/handover sequences using DP--TTT
        ↓
Execute the first action
        ↓
Adaptation Agent: validate ray-path availability and update shared state
        ↓
Next observation and next decision
```

The forecasting networks are trained neural models. The three agents are task-specific logical modules that use those models together with dynamic programming, TTT persistence, and feedback.


## 1. Site-specific Digital Twin and synchronized dataset

This section reproduces the site-specific KFUPM scene, five-gNB layout, UE route bank, GPU ray-tracing pilot, dense RF extraction, dataset validation, multimodal feature construction, and leakage-free sequence preparation. It implements the physical and virtual environment used by Eqs. (1)--(3).


In [ ]:
import sys

print("Python executable:")
print(sys.executable)

In [ ]:
import os
import numpy as np
import torch
import sionna
import sionna.rt as rt
import mitsuba as mi
import drjit as dr

# Configure OptiX for WSL2
os.environ["DRJIT_LIBOPTIX_PATH"] = (
    "/home/afanali/driver-dist/libnvoptix.so.1"
)
os.environ["LD_LIBRARY_PATH"] = (
    "/home/afanali/driver-dist:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

# Select Sionna-compatible GPU variant
mi.set_variant("cuda_ad_mono_polarized")

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Sionna:", sionna.__version__)
print("Mitsuba variant:", mi.variant())
print("Environment initialized successfully.")

In [ ]:
from pathlib import Path

# Main project directory
PROJECT_ROOT = Path("/home/afanali/wcnc_project")

# Project subdirectories
SCENE_DIR = PROJECT_ROOT / "scenes"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

# KFUPM Sionna RT scene
SCENE_PATH = (
    SCENE_DIR
    / "kfupm_terrain"
    / "kfupm_terrain.xml"
)

# Create output directories if needed
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Scene path:", SCENE_PATH)
print("Scene exists:", SCENE_PATH.exists())
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIGURES_DIR)

In [ ]:
IMPROVED_ROOT = PROJECT_ROOT / "improved_experiment"

IMPROVED_DATA_DIR = IMPROVED_ROOT / "data"
IMPROVED_RESULTS_DIR = IMPROVED_ROOT / "results"
IMPROVED_FIGURES_DIR = IMPROVED_ROOT / "figures"
IMPROVED_MODELS_DIR = IMPROVED_ROOT / "models"

for folder in [
    IMPROVED_DATA_DIR,
    IMPROVED_RESULTS_DIR,
    IMPROVED_FIGURES_DIR,
    IMPROVED_MODELS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Separate output folders created:")
print("Data:", IMPROVED_DATA_DIR)
print("Results:", IMPROVED_RESULTS_DIR)
print("Figures:", IMPROVED_FIGURES_DIR)
print("Models:", IMPROVED_MODELS_DIR)

In [ ]:
# Load the KFUPM 3D digital-twin scene
scene = rt.load_scene(
    str(SCENE_PATH),
    merge_shapes=False
)

# Set carrier frequency
scene.frequency = 3.5e9

print("KFUPM scene loaded successfully.")
print("Number of scene objects:", len(scene.objects))
print("Carrier frequency:", scene.frequency / 1e9, "GHz")

In [ ]:
from math import pi

# ============================================================
# Research scenario and antenna configuration
# ============================================================

C0 = 299_792_458.0

RADIO_CONFIG = {
    "carrier_frequency_hz": 3.5e9,
    "bandwidth_hz": 100e6,
    "wavelength_m": C0 / 3.5e9,

    # Initial reproducible assumption used for all gNBs
    "gnb_transmit_power_dbm": 44.0,

    # Sionna PlanarArray spacings are expressed in wavelengths
    "tx_rows": 8,
    "tx_cols": 8,
    "tx_spacing_lambda": 0.5,

    "rx_rows": 2,
    "rx_cols": 2,
    "rx_spacing_lambda": 0.5,

    # 3GPP-like element pattern with vertical polarization
    # This preserves 64 TX elements and 4 RX elements.
    "antenna_pattern": "tr38901",
    "polarization": "V",
    "polarization_model": "tr38901_2",
}

scene.frequency = RADIO_CONFIG["carrier_frequency_hz"]

# ============================================================
# Transmitter array
# ============================================================

scene.tx_array = rt.PlanarArray(
    num_rows=RADIO_CONFIG["tx_rows"],
    num_cols=RADIO_CONFIG["tx_cols"],
    vertical_spacing=RADIO_CONFIG["tx_spacing_lambda"],
    horizontal_spacing=RADIO_CONFIG["tx_spacing_lambda"],
    pattern=RADIO_CONFIG["antenna_pattern"],
    polarization=RADIO_CONFIG["polarization"],
    polarization_model=RADIO_CONFIG["polarization_model"],
)

# ============================================================
# Receiver array
# ============================================================

scene.rx_array = rt.PlanarArray(
    num_rows=RADIO_CONFIG["rx_rows"],
    num_cols=RADIO_CONFIG["rx_cols"],
    vertical_spacing=RADIO_CONFIG["rx_spacing_lambda"],
    horizontal_spacing=RADIO_CONFIG["rx_spacing_lambda"],
    pattern=RADIO_CONFIG["antenna_pattern"],
    polarization=RADIO_CONFIG["polarization"],
    polarization_model=RADIO_CONFIG["polarization_model"],
)

# ============================================================
# KFUPM gNB configuration
# Coordinates are retained exactly from the validated scenario.
# z-coordinate represents the antenna phase-center height.
# ============================================================

GNB_CONFIG = [
    {
        "id": "gNB1_B59",
        "short_label": "gNB1",
        "site_label": "Building 59",
        "position_m": np.array([-541.7, 204.4, 69.0], dtype=np.float64),
        "color_hex": "#0072B2",
        "color_rgb": (0.000, 0.447, 0.698),
        "role": "activity-area serving site",
    },
    {
        "id": "gNB2_B21",
        "short_label": "gNB2",
        "site_label": "Building 21",
        "position_m": np.array([-513.0, -19.3, 87.0], dtype=np.float64),
        "color_hex": "#E69F00",
        "color_rgb": (0.902, 0.624, 0.000),
        "role": "activity-area serving site",
    },
    {
        "id": "gNB3_B15",
        "short_label": "gNB3",
        "site_label": "Building 15",
        "position_m": np.array([-903.3, 98.0, 91.2], dtype=np.float64),
        "color_hex": "#D55E00",
        "color_rgb": (0.835, 0.369, 0.000),
        "role": "activity-area serving site",
    },
]

# Compatibility dictionary for later Sionna and benchmark cells
GNB_POSITIONS = {
    item["id"]: item["position_m"].tolist()
    for item in GNB_CONFIG
}

GNB_NAMES = [item["id"] for item in GNB_CONFIG]

# Activity center used later for map zooming and figure framing
ACTIVITY_CENTER_XY = np.mean(
    [item["position_m"][:2] for item in GNB_CONFIG],
    axis=0
)

# ============================================================
# UE configuration
# ============================================================

UE_CONFIG = {
    "id": "UE_vehicle",
    "label": "UE vehicle",
    "initial_position_m": np.array(
        [-600.0, 150.0, 45.0],
        dtype=np.float64
    ),
    "antenna_mount": "vehicle roof",
    "color_hex": "#000000",
    "color_rgb": (0.0, 0.0, 0.0),
}

# ============================================================
# IEEE figure style
# ============================================================

IEEE_FIGURE_STYLE = {
    "font_family": "DejaVu Sans",
    "title_size": 16,
    "axis_label_size": 14,
    "tick_size": 12,
    "legend_size": 11,
    "line_width": 1.8,
    "grid_alpha": 0.25,
    "building_face_color": "#D9D9D9",
    "building_edge_color": "#555555",
    "road_color": "#BDBDBD",
    "route_color": "#111111",
    "ue_color": "#000000",
}

# Physical array dimensions for reporting
tx_aperture_y_m = (
    (RADIO_CONFIG["tx_cols"] - 1)
    * RADIO_CONFIG["tx_spacing_lambda"]
    * RADIO_CONFIG["wavelength_m"]
)

tx_aperture_z_m = (
    (RADIO_CONFIG["tx_rows"] - 1)
    * RADIO_CONFIG["tx_spacing_lambda"]
    * RADIO_CONFIG["wavelength_m"]
)

rx_aperture_y_m = (
    (RADIO_CONFIG["rx_cols"] - 1)
    * RADIO_CONFIG["rx_spacing_lambda"]
    * RADIO_CONFIG["wavelength_m"]
)

rx_aperture_z_m = (
    (RADIO_CONFIG["rx_rows"] - 1)
    * RADIO_CONFIG["rx_spacing_lambda"]
    * RADIO_CONFIG["wavelength_m"]
)

print("Radio configuration")
print("-------------------")
print(f"Carrier frequency: {RADIO_CONFIG['carrier_frequency_hz'] / 1e9:.2f} GHz")
print(f"Wavelength: {RADIO_CONFIG['wavelength_m']:.4f} m")
print(f"TX elements: {scene.tx_array.num_ant}")
print(f"RX elements: {scene.rx_array.num_ant}")
print(f"TX physical aperture: {tx_aperture_y_m:.3f} m x {tx_aperture_z_m:.3f} m")
print(f"RX physical aperture: {rx_aperture_y_m:.3f} m x {rx_aperture_z_m:.3f} m")

print("\nKFUPM gNB configuration")
print("-----------------------")
for item in GNB_CONFIG:
    print(
        f"{item['id']} | {item['site_label']} | "
        f"position={item['position_m'].tolist()}"
    )

print("\nActivity center:")
print(ACTIVITY_CENTER_XY)

In [ ]:
# ============================================================
# Dense-data configuration and expanded KFUPM coverage
# ============================================================

DATA_CONFIG = {
    # Existing notebook used approximately 20 observations.
    # We will use a much denser route for the improved study.
    "route_points_per_trajectory": 240,

    # Start with six trajectories; increase later if runtime is acceptable.
    "num_trajectories": 6,

    # Mobility sampling interval
    "time_step_s": 0.5,

    # Different UE speeds for diversity
    "speeds_mps": [5.0, 8.0, 12.0],

    # Ray-tracing configuration
    "max_depth": 5,
    "los": True,
    "specular_reflection": True,
    "diffuse_reflection": False,
    "refraction": True,
    "synthetic_array": True,

    # Process observations in manageable batches
    "receiver_batch_size": 16,

    # Reproducibility
    "random_seed": 2026,
}

# Height of an antenna above the selected building roof.
# This is an explicit modeling assumption for the two added sites.
ADDITIONAL_GNB_ROOFTOP_OFFSET_M = 8.0

print("Dense-data configuration")
print("------------------------")
print(
    "Route points per trajectory:",
    DATA_CONFIG["route_points_per_trajectory"]
)
print("Number of trajectories:", DATA_CONFIG["num_trajectories"])
print("UE speeds:", DATA_CONFIG["speeds_mps"], "m/s")
print("Receiver batch size:", DATA_CONFIG["receiver_batch_size"])

In [ ]:
# ============================================================
# Find candidate building rooftops from the actual KFUPM scene
# ============================================================

def point3_to_numpy(point):
    """Convert a Mitsuba Point3f-like object to a NumPy vector."""
    return np.array(
        [float(point[0]), float(point[1]), float(point[2])],
        dtype=np.float64
    )


# Existing validated gNB locations
existing_gnb_xy = np.array(
    [
        item["position_m"][:2]
        for item in GNB_CONFIG
    ],
    dtype=np.float64
)

building_candidates = []

for object_name, scene_object in scene.objects.items():

    if not hasattr(scene_object, "mi_mesh"):
        continue

    try:
        bbox = scene_object.mi_mesh.bbox()
        bbox_min = point3_to_numpy(bbox.min)
        bbox_max = point3_to_numpy(bbox.max)
    except Exception:
        continue

    name_lower = object_name.lower()

    # Select likely building/structure meshes.
    # This avoids treating terrain, trees, and small objects as rooftops.
    looks_like_structure = (
        "itu_" in name_lower
        or "building" in name_lower
        or "mosque" in name_lower
        or "hall" in name_lower
        or "tower" in name_lower
    )

    if not looks_like_structure:
        continue

    size_xyz = bbox_max - bbox_min
    footprint_area = size_xyz[0] * size_xyz[1]

    # Geometry filters for plausible building-like structures
    if size_xyz[0] < 8.0 or size_xyz[1] < 8.0:
        continue

    if size_xyz[2] < 4.0:
        continue

    if size_xyz[0] > 300.0 or size_xyz[1] > 300.0:
        continue

    if footprint_area < 100.0 or footprint_area > 50_000.0:
        continue

    center_xy = 0.5 * (bbox_min[:2] + bbox_max[:2])
    roof_height = bbox_max[2]

    distance_to_existing = min(
        np.linalg.norm(center_xy - existing_xy)
        for existing_xy in existing_gnb_xy
    )

    building_candidates.append(
        {
            "object_name": object_name,
            "center_xy": center_xy,
            "roof_height_m": roof_height,
            "bbox_min": bbox_min,
            "bbox_max": bbox_max,
            "footprint_area_m2": footprint_area,
            "distance_to_existing_m": distance_to_existing,
        }
    )

print("Candidate building structures found:", len(building_candidates))

# Display the most spatially separated candidates
ranked_candidates = sorted(
    building_candidates,
    key=lambda item: item["distance_to_existing_m"],
    reverse=True
)

print("\nMost spatially separated candidates:")
print("-----------------------------------")

for index, candidate in enumerate(ranked_candidates[:15], start=1):
    print(
        f"{index:02d}. {candidate['object_name'][:55]:55s} | "
        f"center=({candidate['center_xy'][0]:8.2f}, "
        f"{candidate['center_xy'][1]:8.2f}) | "
        f"roof z={candidate['roof_height_m']:6.2f} m | "
        f"distance={candidate['distance_to_existing_m']:7.2f} m"
    )

In [ ]:
# ============================================================
# Select two additional gNB sites using farthest-point sampling
# ============================================================

if "building_candidates" not in globals():
    raise NameError(
        "building_candidates is not defined. "
        "Run the candidate-rooftop search cell first."
    )

if len(building_candidates) < 2:
    raise RuntimeError(
        "Fewer than two suitable building candidates were found."
    )

# Always preserve only the original three validated sites
BASE_GNB_CONFIG = [
    dict(item)
    for item in GNB_CONFIG[:3]
]

# Start coverage anchors from the original three gNBs
coverage_anchors = [
    item["position_m"][:2].copy()
    for item in BASE_GNB_CONFIG
]

remaining_candidates = building_candidates.copy()
selected_candidates = []

# Select two candidates that maximize spatial separation
for _ in range(2):

    best_candidate = max(
        remaining_candidates,
        key=lambda candidate: min(
            np.linalg.norm(
                candidate["center_xy"] - anchor
            )
            for anchor in coverage_anchors
        )
    )

    selected_candidates.append(best_candidate)
    coverage_anchors.append(best_candidate["center_xy"])
    remaining_candidates.remove(best_candidate)

# Colorblind-friendly colors for the new sites
additional_colors = [
    ("#009E73", (0.000, 0.620, 0.451)),
    ("#CC79A7", (0.800, 0.475, 0.655)),
]

EXPANDED_GNB_CONFIG = [
    dict(item)
    for item in BASE_GNB_CONFIG
]

for index, (candidate, color_info) in enumerate(
    zip(selected_candidates, additional_colors),
    start=4
):

    color_hex, color_rgb = color_info

    roof_position = np.array(
        [
            candidate["center_xy"][0],
            candidate["center_xy"][1],
            candidate["roof_height_m"]
            + ADDITIONAL_GNB_ROOFTOP_OFFSET_M,
        ],
        dtype=np.float64,
    )

    EXPANDED_GNB_CONFIG.append(
        {
            "id": f"gNB{index}_EXPANDED",
            "short_label": f"gNB{index}",
            "site_label": f"Expanded site {index}",
            "position_m": roof_position,
            "color_hex": color_hex,
            "color_rgb": color_rgb,
            "role": "coverage-expansion site",
            "source_scene_object": candidate["object_name"],
            "roof_height_m": candidate["roof_height_m"],
            "rooftop_offset_m": ADDITIONAL_GNB_ROOFTOP_OFFSET_M,
        }
    )

# Update the active configuration
GNB_CONFIG = EXPANDED_GNB_CONFIG

GNB_POSITIONS = {
    item["id"]: item["position_m"].tolist()
    for item in GNB_CONFIG
}

GNB_NAMES = [
    item["id"]
    for item in GNB_CONFIG
]

print("Expanded gNB configuration")
print("--------------------------")

for item in GNB_CONFIG:
    print(
        f"{item['id']:18s} | "
        f"{item['site_label']:22s} | "
        f"position={np.round(item['position_m'], 2).tolist()}"
    )

print("\nTotal number of gNBs:", len(GNB_CONFIG))

In [ ]:
import matplotlib.pyplot as plt

# Display the selected scene objects and rooftop information
print("Selected expansion sites")
print("------------------------")

for item in GNB_CONFIG[3:]:
    print("Site:", item["id"])
    print("Source object:", item["source_scene_object"])
    print("Roof height:", round(item["roof_height_m"], 2), "m")
    print("Antenna position:", np.round(item["position_m"], 2))
    print()

# Candidate building centers
candidate_xy = np.array(
    [
        candidate["center_xy"]
        for candidate in building_candidates
    ]
)

# Plot validation map
plt.figure(figsize=(11, 8))

plt.scatter(
    candidate_xy[:, 0],
    candidate_xy[:, 1],
    s=8,
    color="#D0D0D0",
    alpha=0.45,
    label="Candidate building centers"
)

for item in GNB_CONFIG:
    position = item["position_m"]
    is_expanded = "EXPANDED" in item["id"]

    plt.scatter(
        position[0],
        position[1],
        s=180 if is_expanded else 220,
        color=item["color_hex"],
        edgecolor="black",
        linewidth=1.2,
        marker="^",
        zorder=5,
    )

    plt.annotate(
        f"{item['short_label']}\n{item['site_label']}",
        xy=(position[0], position[1]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor=item["color_hex"],
            alpha=0.9,
        ),
    )

# Activity center
plt.scatter(
    ACTIVITY_CENTER_XY[0],
    ACTIVITY_CENTER_XY[1],
    marker="x",
    s=160,
    color="black",
    linewidth=2.2,
    label="Main activity center",
    zorder=6,
)

plt.xlabel("KFUPM x-coordinate (m)", fontsize=13)
plt.ylabel("KFUPM y-coordinate (m)", fontsize=13)
plt.title(
    "KFUPM Candidate gNB Placement Validation",
    fontsize=16,
    fontweight="bold",
)

plt.grid(alpha=0.25)
plt.axis("equal")
plt.legend(fontsize=10)
plt.tight_layout()

validation_path = (
    IMPROVED_FIGURES_DIR
    / "gnb_placement_validation.png"
)

plt.savefig(
    validation_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Validation figure saved to:")
print(validation_path)

In [ ]:
# ============================================================
# Reset to the only currently verified gNB site
# ============================================================

b59_item = next(
    item
    for item in GNB_CONFIG
    if item["short_label"] == "gNB1"
)

b59_item["id"] = "gNB1_B59"
b59_item["short_label"] = "gNB1"
b59_item["building_name"] = (
    "Building 59 – Electrical Engineering Department"
)
b59_item["site_label"] = (
    "Building 59 – Electrical Engineering Department"
)
b59_item["role"] = "primary activity-area site"
b59_item["distance_from_b59_m"] = 0.0

GNB_CONFIG = [b59_item]

GNB_POSITIONS = {
    item["id"]: item["position_m"].tolist()
    for item in GNB_CONFIG
}

GNB_NAMES = [
    item["id"]
    for item in GNB_CONFIG
]

print("Verified gNB configuration")
print("--------------------------")

for item in GNB_CONFIG:
    print(
        item["id"],
        "|",
        item["building_name"],
        "|",
        np.round(item["position_m"], 2).tolist()
    )

In [ ]:
# ============================================================
# Discover meaningful building names in the KFUPM scene
# ============================================================

import re

def remove_material_suffix(name):
    return re.split(
        r"-itu_",
        name,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]


def readable_scene_name(name):
    name = remove_material_suffix(name)
    name = re.sub(r"[_-]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()


meaningful_buildings = {}

for candidate in building_candidates:

    raw_name = candidate["object_name"]
    readable_name = readable_scene_name(raw_name)

    # Reject generic mesh identifiers
    if re.match(
        r"^element\s+\d+$",
        readable_name,
        flags=re.IGNORECASE
    ):
        continue

    # Keep only semantically meaningful objects
    name_lower = readable_name.lower()

    meaningful_keywords = [
        "building",
        "department",
        "library",
        "physics",
        "mathematics",
        "math",
        "administration",
        "research",
        "mosque",
        "hall",
        "tower",
        "college",
        "institute",
        "center",
    ]

    if not any(
        keyword in name_lower
        for keyword in meaningful_keywords
    ):
        continue

    # Keep the largest mesh if several material meshes exist
    if (
        readable_name not in meaningful_buildings
        or candidate["footprint_area_m2"]
        > meaningful_buildings[readable_name]["footprint_area_m2"]
    ):
        candidate_copy = dict(candidate)
        candidate_copy["readable_name"] = readable_name
        meaningful_buildings[readable_name] = candidate_copy

meaningful_buildings = list(
    meaningful_buildings.values()
)

b59_xy = np.array(
    [-541.7, 204.4],
    dtype=np.float64
)

meaningful_buildings.sort(
    key=lambda item: np.linalg.norm(
        item["center_xy"] - b59_xy
    )
)

print(
    "Meaningful named scene objects found:",
    len(meaningful_buildings)
)

print("\nNamed buildings and structures:")
print("--------------------------------")

for index, candidate in enumerate(
    meaningful_buildings,
    start=1
):

    distance = np.linalg.norm(
        candidate["center_xy"] - b59_xy
    )

    print(
        f"{index:03d}. "
        f"{candidate['readable_name'][:65]:65s} | "
        f"distance from B59: {distance:8.2f} m | "
        f"center: "
        f"{np.round(candidate['center_xy'], 2).tolist()}"
    )

In [ ]:
# ============================================================
# Final verified five-gNB KFUPM layout
#
# gNB1: fixed at Building 59
# gNB2-gNB5: meaningful buildings at least 400 m away
# ============================================================

import re

B59_POSITION = np.array(
    [-541.7, 204.4, 69.0],
    dtype=np.float64
)

B59_XY = B59_POSITION[:2]
MIN_DISTANCE_FROM_B59_M = 400.0

# Exclude religious structures from the initial radio-site layout.
# They remain available for visualization.
excluded_keywords = [
    "mosque"
]

eligible_buildings = []

for candidate in meaningful_buildings:

    name_lower = candidate["readable_name"].lower()

    if any(
        keyword in name_lower
        for keyword in excluded_keywords
    ):
        continue

    distance_from_b59 = np.linalg.norm(
        candidate["center_xy"] - B59_XY
    )

    if distance_from_b59 < MIN_DISTANCE_FROM_B59_M:
        continue

    candidate_copy = dict(candidate)
    candidate_copy["distance_from_b59_m"] = distance_from_b59
    eligible_buildings.append(candidate_copy)

if len(eligible_buildings) < 4:
    raise RuntimeError(
        f"Only {len(eligible_buildings)} named buildings are "
        f"at least {MIN_DISTANCE_FROM_B59_M:.0f} m from B59."
    )

# Select four sites using farthest-point sampling.
# This spreads the sites spatially across the scene.
selected_buildings = []

coverage_anchors = [
    B59_XY.copy()
]

remaining_buildings = eligible_buildings.copy()

for _ in range(4):

    selected = max(
        remaining_buildings,
        key=lambda candidate: min(
            np.linalg.norm(
                candidate["center_xy"] - anchor
            )
            for anchor in coverage_anchors
        )
    )

    selected_buildings.append(selected)
    coverage_anchors.append(
        selected["center_xy"]
    )

    remaining_buildings.remove(selected)


def polish_building_label(label):
    """Create a clean label for the IEEE figure."""

    corrections = {
        "Building 59 Electrical Engineering Dept":
            "Building 59 – Electrical Engineering Department",

        "Building 5 Mathematics department":
            "Building 5 – Mathematics Department",

        "Building 6 Physics department":
            "Building 6 – Physics Department",

        "Building 8 Main Liberary":
            "Building 8 – Main Library",

        "Building 14 ITC":
            "Building 14 – ITC",

        "Building 15 Research Institute":
            "Building 15 – Research Institute",

        "Building 21 Administration":
            "Building 21 – Administration",

        "Building 22 Computer Science Department":
            "Building 22 – Computer Science Department",

        "Building 76 Petrolum Engineering Geosciences Department":
            "Building 76 – Petroleum Engineering and Geosciences Department",
    }

    if label in corrections:
        return corrections[label]

    # Remove mesh/subcomponent suffixes such as "Building 44 001"
    label = re.sub(
        r"^(Building\s+\d+)\s+\d+$",
        r"\1",
        label,
        flags=re.IGNORECASE
    )

    return label


def safe_identifier(label):
    """Create a clean Sionna device identifier."""
    identifier = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        label
    )
    return identifier.strip("_")[:35]


# Colorblind-friendly site colors
site_colors = [
    ("#0072B2", (0.000, 0.447, 0.698)),
    ("#E69F00", (0.902, 0.624, 0.000)),
    ("#D55E00", (0.835, 0.369, 0.000)),
    ("#009E73", (0.000, 0.620, 0.451)),
    ("#CC79A7", (0.800, 0.475, 0.655)),
]

# Fixed Building 59 site
FINAL_GNB_CONFIG = [
    {
        "id": "gNB1_B59",
        "short_label": "gNB1",
        "site_label": (
            "Building 59 – Electrical Engineering Department"
        ),
        "building_name": (
            "Building 59 – Electrical Engineering Department"
        ),
        "position_m": B59_POSITION,
        "color_hex": site_colors[0][0],
        "color_rgb": site_colors[0][1],
        "role": "primary activity-area site",
        "distance_from_b59_m": 0.0,
        "source_scene_object": (
            "Building_59_Electrical_Engineering_Dept-itu_brick"
        ),
    }
]

# Add four distant, meaningful buildings
for index, (candidate, color_info) in enumerate(
    zip(selected_buildings, site_colors[1:]),
    start=2
):

    building_label = polish_building_label(
        candidate["readable_name"]
    )

    gnb_position = np.array(
        [
            candidate["center_xy"][0],
            candidate["center_xy"][1],
            candidate["roof_height_m"]
            + ADDITIONAL_GNB_ROOFTOP_OFFSET_M,
        ],
        dtype=np.float64
    )

    FINAL_GNB_CONFIG.append(
        {
            "id": (
                f"gNB{index}_"
                f"{safe_identifier(building_label)}"
            ),
            "short_label": f"gNB{index}",
            "site_label": building_label,
            "building_name": building_label,
            "position_m": gnb_position,
            "color_hex": color_info[0],
            "color_rgb": color_info[1],
            "role": "campus-wide coverage site",
            "distance_from_b59": candidate[
                "distance_from_b59_m"
            ],
            "distance_from_b59_m": candidate[
                "distance_from_b59_m"
            ],
            "roof_height_m": candidate[
                "roof_height_m"
            ],
            "source_scene_object": candidate[
                "object_name"
            ],
        }
    )

# Activate final configuration
GNB_CONFIG = FINAL_GNB_CONFIG

GNB_POSITIONS = {
    item["id"]: item["position_m"].tolist()
    for item in GNB_CONFIG
}

GNB_NAMES = [
    item["id"]
    for item in GNB_CONFIG
]

print("Final verified KFUPM gNB layout")
print("--------------------------------")

for item in GNB_CONFIG:
    print(
        f"{item['short_label']:5s} | "
        f"{item['building_name'][:65]:65s} | "
        f"distance from B59: "
        f"{item['distance_from_b59_m']:8.2f} m | "
        f"position: "
        f"{np.round(item['position_m'], 2).tolist()}"
    )

print("\nTotal gNBs:", len(GNB_CONFIG))

In [ ]:
# ============================================================
# Add final gNB transmitters and UE receiver to the scene
# ============================================================

# Remove previously added radio devices if this cell is rerun
for tx_name in list(scene.transmitters.keys()):
    scene.remove(tx_name)

for rx_name in list(scene.receivers.keys()):
    scene.remove(rx_name)

# Common boresight target for the activity area
sector_target = (
    float(ACTIVITY_CENTER_XY[0]),
    float(ACTIVITY_CENTER_XY[1]),
    45.0,
)

# Add all gNB transmitters
for item in GNB_CONFIG:

    tx_position = tuple(
        float(value)
        for value in item["position_m"]
    )

    transmitter = rt.Transmitter(
        name=item["id"],
        position=tx_position,
        look_at=sector_target,
        power_dbm=RADIO_CONFIG[
            "gnb_transmit_power_dbm"
        ],
        color=item["color_rgb"],
        display_radius=10.0,
    )

    scene.add(transmitter)

# Add the initial UE receiver
ue_position = tuple(
    float(value)
    for value in UE_CONFIG["initial_position_m"]
)

receiver = rt.Receiver(
    name=UE_CONFIG["id"],
    position=ue_position,
    orientation=(0.0, 0.0, 0.0),
    color=UE_CONFIG["color_rgb"],
    display_radius=2.5,
)

scene.add(receiver)

print("Radio devices added successfully.")
print("Transmitters:", len(scene.transmitters))
print("Receivers:", len(scene.receivers))

print("\nTransmitters:")
for item in GNB_CONFIG:
    print(
        f"{item['short_label']} | "
        f"{item['building_name']} | "
        f"{np.round(item['position_m'], 2).tolist()}"
    )

print("\nUE position:")
print(np.round(UE_CONFIG["initial_position_m"], 2))

In [ ]:
# ============================================================
# Interactive KFUPM Digital-Twin Visualization
# ============================================================

import numpy as np
import plotly.graph_objects as go
import re

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def point3_to_numpy_local(point):
    return np.array(
        [
            float(point[0]),
            float(point[1]),
            float(point[2]),
        ],
        dtype=np.float64,
    )


def get_scene_bbox(scene_object):
    try:
        bbox = scene_object.mi_mesh.bbox()

        bbox_min = point3_to_numpy_local(bbox.min)
        bbox_max = point3_to_numpy_local(bbox.max)

        return bbox_min, bbox_max

    except Exception:
        return None, None


def add_building_box(
    figure,
    bbox_min,
    bbox_max,
    label,
    color="#BDBDBD",
    opacity=0.65,
    show_legend=False,
):
    """
    Add an extruded building box using the actual scene bounding box.
    """

    xmin, ymin, zmin = bbox_min
    xmax, ymax, zmax = bbox_max

    x = [
        xmin, xmax, xmax, xmin,
        xmin, xmax, xmax, xmin,
    ]

    y = [
        ymin, ymin, ymax, ymax,
        ymin, ymin, ymax, ymax,
    ]

    z = [
        zmin, zmin, zmin, zmin,
        zmax, zmax, zmax, zmax,
    ]

    # Cuboid triangular faces
    i = [
        0, 0, 4, 4,
        0, 0, 1, 1,
        2, 2, 3, 3,
    ]

    j = [
        1, 2, 5, 6,
        4, 5, 5, 6,
        6, 7, 7, 4,
    ]

    k = [
        2, 3, 6, 7,
        5, 1, 6, 2,
        7, 3, 4, 0,
    ]

    figure.add_trace(
        go.Mesh3d(
            x=x,
            y=y,
            z=z,
            i=i,
            j=j,
            k=k,
            color=color,
            opacity=opacity,
            flatshading=True,
            name="Buildings",
            showlegend=show_legend,
            hovertemplate=(
                f"<b>{label}</b><br>"
                "x: %{x:.1f} m<br>"
                "y: %{y:.1f} m<br>"
                "z: %{z:.1f} m"
                "<extra></extra>"
            ),
            lighting=dict(
                ambient=0.65,
                diffuse=0.75,
                specular=0.15,
            ),
        )
    )


def add_flat_rectangle(
    figure,
    bbox_min,
    bbox_max,
    label,
    color,
    opacity=0.45,
    show_legend=False,
):
    """
    Add a flat surface for roads, paths, or terrain.
    """

    xmin, ymin, zmin = bbox_min
    xmax, ymax, zmax = bbox_max

    z_surface = zmin + 0.5

    figure.add_trace(
        go.Mesh3d(
            x=[xmin, xmax, xmax, xmin],
            y=[ymin, ymin, ymax, ymax],
            z=[
                z_surface,
                z_surface,
                z_surface,
                z_surface,
            ],
            i=[0, 0],
            j=[1, 2],
            k=[2, 3],
            color=color,
            opacity=opacity,
            name=label,
            showlegend=show_legend,
            hovertemplate=(
                f"<b>{label}</b><br>"
                "x: %{x:.1f} m<br>"
                "y: %{y:.1f} m"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Detect roads and vegetation from the actual scene
# ------------------------------------------------------------

ROAD_TOKENS = (
    "road",
    "street",
    "avenue",
    "asphalt",
    "pavement",
    "sidewalk",
    "parking",
    "path",
    "driveway",
)

VEGETATION_TOKENS = (
    "tree",
    "palm",
    "grass",
    "shrub",
    "bush",
    "plant",
    "vegetation",
    "lawn",
    "date",
)

road_objects = []
vegetation_objects = []

for object_name, scene_object in scene.objects.items():

    bbox_min, bbox_max = get_scene_bbox(scene_object)

    if bbox_min is None:
        continue

    name_lower = object_name.lower()
    object_size = bbox_max - bbox_min

    if any(token in name_lower for token in ROAD_TOKENS):

        # Ignore enormous terrain-like objects
        if object_size[0] < 1000 and object_size[1] < 1000:
            road_objects.append(
                {
                    "name": object_name,
                    "bbox_min": bbox_min,
                    "bbox_max": bbox_max,
                    "center_xy": 0.5 * (
                        bbox_min[:2] + bbox_max[:2]
                    ),
                }
            )

    if any(
        token in name_lower
        for token in VEGETATION_TOKENS
    ):

        vegetation_objects.append(
            {
                "name": object_name,
                "bbox_min": bbox_min,
                "bbox_max": bbox_max,
                "center_xy": 0.5 * (
                    bbox_min[:2] + bbox_max[:2]
                ),
            }
        )

print("Detected road-like scene objects:", len(road_objects))
print(
    "Detected vegetation-like scene objects:",
    len(vegetation_objects),
)


# ------------------------------------------------------------
# Route, if already available
# ------------------------------------------------------------

route_array = None

if "ue_route" in globals():

    try:
        route_array = np.asarray(ue_route, dtype=np.float64)

        if route_array.ndim != 2 or route_array.shape[1] < 3:
            route_array = None

    except Exception:
        route_array = None


# ------------------------------------------------------------
# Figure generator
# ------------------------------------------------------------

def make_digital_twin_figure(
    radius_m=None,
    title="KFUPM Digital Twin",
    output_filename="kfupm_digital_twin.html",
):

    figure = go.Figure()

    center_xy = np.asarray(
        ACTIVITY_CENTER_XY,
        dtype=np.float64,
    )

    def visible_xy(xy):
        if radius_m is None:
            return True

        return (
            np.linalg.norm(
                np.asarray(xy) - center_xy
            )
            <= radius_m
        )

    # --------------------------------------------------------
    # Select named buildings
    # --------------------------------------------------------

    visible_buildings = [
        candidate
        for candidate in meaningful_buildings
        if visible_xy(candidate["center_xy"])
    ]

    # --------------------------------------------------------
    # Select roads and vegetation
    # --------------------------------------------------------

    visible_roads = [
        road
        for road in road_objects
        if visible_xy(road["center_xy"])
    ]

    visible_vegetation = [
        vegetation
        for vegetation in vegetation_objects
        if visible_xy(vegetation["center_xy"])
    ]

    # Limit vegetation markers for browser performance
    visible_vegetation = sorted(
        visible_vegetation,
        key=lambda item: np.linalg.norm(
            item["center_xy"] - center_xy
        ),
    )[:500]

    # --------------------------------------------------------
    # Select gNBs
    # --------------------------------------------------------

    visible_gnbs = [
        item
        for item in GNB_CONFIG
        if visible_xy(item["position_m"][:2])
    ]

    # --------------------------------------------------------
    # Determine visualization bounds
    # --------------------------------------------------------

    xy_points = []

    for candidate in visible_buildings:
        xy_points.extend(
            [
                candidate["bbox_min"][:2],
                candidate["bbox_max"][:2],
            ]
        )

    for road in visible_roads:
        xy_points.extend(
            [
                road["bbox_min"][:2],
                road["bbox_max"][:2],
            ]
        )

    for item in visible_gnbs:
        xy_points.append(item["position_m"][:2])

    if route_array is not None:
        for point in route_array:
            if visible_xy(point[:2]):
                xy_points.append(point[:2])

    if len(xy_points) == 0:
        xy_points = [
            center_xy - np.array([300.0, 300.0]),
            center_xy + np.array([300.0, 300.0]),
        ]

    xy_points = np.asarray(xy_points)

    xmin = xy_points[:, 0].min() - 50.0
    xmax = xy_points[:, 0].max() + 50.0
    ymin = xy_points[:, 1].min() - 50.0
    ymax = xy_points[:, 1].max() + 50.0

    building_z_values = []

    for candidate in visible_buildings:
        building_z_values.extend(
            [
                candidate["bbox_min"][2],
                candidate["bbox_max"][2],
            ]
        )

    if building_z_values:
        ground_z = min(building_z_values) - 2.0
    else:
        ground_z = 0.0

    # --------------------------------------------------------
    # Ground plane
    # --------------------------------------------------------

    add_flat_rectangle(
        figure,
        np.array([xmin, ymin, ground_z]),
        np.array([xmax, ymax, ground_z + 0.1]),
        label="KFUPM terrain reference",
        color="#E8E3D3",
        opacity=0.30,
        show_legend=True,
    )

    # --------------------------------------------------------
    # Roads
    # --------------------------------------------------------

    for road_index, road in enumerate(visible_roads):

        add_flat_rectangle(
            figure,
            road["bbox_min"],
            road["bbox_max"],
            label="Roads and paved areas",
            color="#777777",
            opacity=0.52,
            show_legend=(road_index == 0),
        )

    # --------------------------------------------------------
    # Buildings
    # --------------------------------------------------------

    gnb_building_centers = [
        item["position_m"][:2]
        for item in GNB_CONFIG
    ]

    for building_index, candidate in enumerate(
        visible_buildings
    ):

        label = candidate.get(
            "readable_name",
            candidate["object_name"],
        )

        building_center = candidate["center_xy"]

        is_gnb_building = any(
            np.linalg.norm(
                building_center - gnb_center
            ) < 60.0
            for gnb_center in gnb_building_centers
        )

        building_color = (
            "#AFC6D8"
            if is_gnb_building
            else "#C9C9C9"
        )

        add_building_box(
            figure,
            candidate["bbox_min"],
            candidate["bbox_max"],
            label=label,
            color=building_color,
            opacity=0.72,
            show_legend=(
                building_index == 0
            ),
        )

    # --------------------------------------------------------
    # Vegetation
    # --------------------------------------------------------

    if visible_vegetation:

        vegetation_x = []
        vegetation_y = []
        vegetation_z = []
        vegetation_sizes = []
        vegetation_hover = []

        for vegetation in visible_vegetation:

            bbox_min = vegetation["bbox_min"]
            bbox_max = vegetation["bbox_max"]
            center = 0.5 * (
                bbox_min + bbox_max
            )

            object_width = max(
                bbox_max[0] - bbox_min[0],
                bbox_max[1] - bbox_min[1],
                1.0,
            )

            vegetation_x.append(center[0])
            vegetation_y.append(center[1])
            vegetation_z.append(bbox_max[2])
            vegetation_sizes.append(
                float(np.clip(
                    3.0 + object_width * 0.4,
                    3.0,
                    11.0,
                ))
            )
            vegetation_hover.append(
                vegetation["name"]
            )

        figure.add_trace(
            go.Scatter3d(
                x=vegetation_x,
                y=vegetation_y,
                z=vegetation_z,
                mode="markers",
                name="Vegetation",
                showlegend=True,
                marker=dict(
                    size=vegetation_sizes,
                    color="#2E7D32",
                    opacity=0.75,
                    symbol="circle",
                ),
                text=vegetation_hover,
                hovertemplate=(
                    "<b>Vegetation</b><br>"
                    "%{text}<br>"
                    "x: %{x:.1f} m<br>"
                    "y: %{y:.1f} m"
                    "<extra></extra>"
                ),
            )
        )

    # --------------------------------------------------------
    # UE route
    # --------------------------------------------------------

    if route_array is not None:

        route_visible = np.array(
            [
                point
                for point in route_array
                if visible_xy(point[:2])
            ]
        )

        if len(route_visible) > 1:

            figure.add_trace(
                go.Scatter3d(
                    x=route_visible[:, 0],
                    y=route_visible[:, 1],
                    z=route_visible[:, 2] + 2.0,
                    mode="lines",
                    name="UE route",
                    line=dict(
                        color="#111111",
                        width=7,
                    ),
                    hovertemplate=(
                        "UE route<br>"
                        "x: %{x:.1f} m<br>"
                        "y: %{y:.1f} m"
                        "<extra></extra>"
                    ),
                )
            )

    # --------------------------------------------------------
    # gNB markers
    # --------------------------------------------------------

    if visible_gnbs:

        figure.add_trace(
            go.Scatter3d(
                x=[
                    item["position_m"][0]
                    for item in visible_gnbs
                ],
                y=[
                    item["position_m"][1]
                    for item in visible_gnbs
                ],
                z=[
                    item["position_m"][2]
                    for item in visible_gnbs
                ],
                mode="markers+text",
                name="gNBs",
                text=[
                    item["short_label"]
                    for item in visible_gnbs
                ],
                textposition="top center",
                marker=dict(
                    size=12,
                    symbol="diamond",
                    color=[
                        item["color_hex"]
                        for item in visible_gnbs
                    ],
                    line=dict(
                        color="black",
                        width=1.5,
                    ),
                ),
                customdata=[
                    [
                        item["building_name"],
                        item["position_m"][2],
                    ]
                    for item in visible_gnbs
                ],
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Site: %{customdata[0]}<br>"
                    "x: %{x:.1f} m<br>"
                    "y: %{y:.1f} m<br>"
                    "antenna height: %{customdata[1]:.1f} m"
                    "<extra></extra>"
                ),
            )
        )

    # --------------------------------------------------------
    # UE marker
    # --------------------------------------------------------

    ue_position = np.asarray(
        UE_CONFIG["initial_position_m"],
        dtype=np.float64,
    )

    if visible_xy(ue_position[:2]):

        figure.add_trace(
            go.Scatter3d(
                x=[ue_position[0]],
                y=[ue_position[1]],
                z=[ue_position[2]],
                mode="markers+text",
                name="UE vehicle",
                text=["UE"],
                textposition="top center",
                marker=dict(
                    size=10,
                    symbol="circle",
                    color="#000000",
                    line=dict(
                        color="white",
                        width=2,
                    ),
                ),
                hovertemplate=(
                    "<b>UE vehicle</b><br>"
                    "x: %{x:.1f} m<br>"
                    "y: %{y:.1f} m<br>"
                    "z: %{z:.1f} m"
                    "<extra></extra>"
                ),
            )
        )

    # --------------------------------------------------------
    # Professional interactive layout
    # --------------------------------------------------------

    figure.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(
                size=20,
                family="Arial",
            ),
        ),
        template="plotly_white",
        width=1250,
        height=850,
        margin=dict(
            l=0,
            r=0,
            t=70,
            b=0,
        ),
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#777777",
            borderwidth=1,
        ),
        scene=dict(
            xaxis_title="KFUPM x-coordinate (m)",
            yaxis_title="KFUPM y-coordinate (m)",
            zaxis_title="Elevation (m)",
            aspectmode="data",
            camera=dict(
                eye=dict(
                    x=1.45,
                    y=1.45,
                    z=1.15,
                )
            ),
            xaxis=dict(
                showgrid=True,
                gridcolor="#D9D9D9",
                zeroline=False,
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor="#D9D9D9",
                zeroline=False,
            ),
            zaxis=dict(
                showgrid=True,
                gridcolor="#D9D9D9",
                zeroline=False,
            ),
        ),
    )

    output_path = (
        IMPROVED_FIGURES_DIR
        / output_filename
    )

    figure.write_html(
        str(output_path),
        include_plotlyjs=True,
        full_html=True,
    )

    print("\nFigure saved to:")
    print(output_path)
    print("Buildings shown:", len(visible_buildings))
    print("Road objects shown:", len(visible_roads))
    print("Vegetation objects shown:", len(visible_vegetation))
    print("gNBs shown:", len(visible_gnbs))

    return figure


# ============================================================
# Generate overview and close-up figures
# ============================================================

overview_figure = make_digital_twin_figure(
    radius_m=None,
    title="KFUPM Digital Twin — Campus-Wide gNB Deployment",
    output_filename=(
        "kfupm_digital_twin_campus_overview.html"
    ),
)

focus_figure = make_digital_twin_figure(
    radius_m=1000.0,
    title=(
        "KFUPM Digital Twin — "
        "Activity-Area Close-Up"
    ),
    output_filename=(
        "kfupm_digital_twin_activity_area.html"
    ),
)

# Display the close-up interactively in VS Code
focus_figure.show()

In [ ]:
# ============================================================
# Locate gNB and UE icons from the previous notebook
# ============================================================

from pathlib import Path

asset_search_roots = [
    IMPROVED_ROOT / "assets",
    PROJECT_ROOT / "figures",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "notebooks",
]

image_files = []

for root in asset_search_roots:
    if root.exists():
        image_files.extend(
            root.rglob("*.png")
        )

# Remove duplicate paths
image_files = sorted(set(image_files))

gnb_icon_candidates = [
    path for path in image_files
    if any(
        token in path.name.lower()
        for token in ["gnodeb", "gnb", "base_station"]
    )
]

ue_icon_candidates = [
    path for path in image_files
    if any(
        token in path.name.lower()
        for token in ["car", "ue", "vehicle"]
    )
]

print("gNB icon candidates:")
for path in gnb_icon_candidates:
    print(path)

print("\nUE icon candidates:")
for path in ue_icon_candidates:
    print(path)

if not gnb_icon_candidates:
    raise FileNotFoundError(
        "No gNB PNG was found. Copy the previous gNB image into "
        f"{IMPROVED_ROOT / 'assets'}."
    )

if not ue_icon_candidates:
    raise FileNotFoundError(
        "No UE/car PNG was found. Copy the previous UE image into "
        f"{IMPROVED_ROOT / 'assets'}."
    )

GNB_ICON_PATH = gnb_icon_candidates[0]
UE_ICON_PATH = ue_icon_candidates[0]

print("\nSelected gNB icon:", GNB_ICON_PATH)
print("Selected UE icon:", UE_ICON_PATH)

In [ ]:
# ============================================================
# Improved road and vegetation detection
# Uses both object names and radio-material names
# ============================================================

ROAD_TOKENS = (
    "road",
    "street",
    "avenue",
    "asphalt",
    "pavement",
    "sidewalk",
    "parking",
    "path",
    "driveway",
    "curb",
)

VEGETATION_TOKENS = (
    "tree",
    "palm",
    "grass",
    "shrub",
    "bush",
    "plant",
    "vegetation",
    "lawn",
    "leaf",
    "leaves",
    "foliage",
    "trunk",
    "wood",
    "garden",
)

road_objects = []
vegetation_objects = []

matched_road_names = []
matched_vegetation_names = []

for object_name, scene_object in scene.objects.items():

    bbox_min, bbox_max = get_scene_bbox(scene_object)

    if bbox_min is None:
        continue

    # Inspect both the mesh name and its assigned radio material
    object_text = str(object_name).lower()

    try:
        material_text = str(
            scene_object.radio_material
        ).lower()
    except Exception:
        material_text = ""

    searchable_text = (
        object_text + " " + material_text
    )

    object_size = bbox_max - bbox_min
    footprint_area = (
        object_size[0] * object_size[1]
    )

    center_xy = 0.5 * (
        bbox_min[:2] + bbox_max[:2]
    )

    # Roads and paved areas
    if any(
        token in searchable_text
        for token in ROAD_TOKENS
    ):

        # Remove extremely large terrain-like meshes
        if (
            object_size[0] < 1000.0
            and object_size[1] < 1000.0
            and footprint_area > 10.0
        ):
            road_objects.append(
                {
                    "name": object_name,
                    "material": material_text,
                    "bbox_min": bbox_min,
                    "bbox_max": bbox_max,
                    "center_xy": center_xy,
                }
            )

            matched_road_names.append(
                (
                    object_name,
                    material_text,
                )
            )

    # Vegetation
    if any(
        token in searchable_text
        for token in VEGETATION_TOKENS
    ):

        vegetation_objects.append(
            {
                "name": object_name,
                "material": material_text,
                "bbox_min": bbox_min,
                "bbox_max": bbox_max,
                "center_xy": center_xy,
            }
        )

        matched_vegetation_names.append(
            (
                object_name,
                material_text,
            )
        )

print("Detected road/pavement objects:")
print(len(road_objects))

print("\nDetected vegetation objects:")
print(len(vegetation_objects))

print("\nExample road matches:")
for name, material in matched_road_names[:15]:
    print("-", name, "|", material)

print("\nExample vegetation matches:")
for name, material in matched_vegetation_names[:15]:
    print("-", name, "|", material)

In [ ]:
# ============================================================
# Dense UE trajectory bank for the improved experiment
# ============================================================

ROUTE_CONFIG = {
    "num_trajectories": 6,
    "points_per_trajectory": 240,
    "ue_height_m": float(
        UE_CONFIG["initial_position_m"][2]
    ),
    "site_offset_m": 35.0,
    "time_step_s": 0.5,
    "speeds_mps": [5.0, 8.0, 12.0],
    "random_seed": 2026,
}

np.random.seed(
    ROUTE_CONFIG["random_seed"]
)


def sample_polyline(waypoints_xy, num_points):
    """
    Uniformly sample a polyline by travelled distance.
    """

    waypoints_xy = np.asarray(
        waypoints_xy,
        dtype=np.float64,
    )

    segment_vectors = np.diff(
        waypoints_xy,
        axis=0,
    )

    segment_lengths = np.linalg.norm(
        segment_vectors,
        axis=1,
    )

    cumulative_distance = np.concatenate(
        [
            [0.0],
            np.cumsum(segment_lengths),
        ]
    )

    total_distance = cumulative_distance[-1]

    target_distances = np.linspace(
        0.0,
        total_distance,
        num_points,
    )

    sampled_points = []

    for distance in target_distances:

        segment_index = np.searchsorted(
            cumulative_distance,
            distance,
            side="right",
        ) - 1

        segment_index = int(
            np.clip(
                segment_index,
                0,
                len(segment_lengths) - 1,
            )
        )

        segment_start_distance = (
            cumulative_distance[segment_index]
        )

        segment_length = max(
            segment_lengths[segment_index],
            1e-9,
        )

        interpolation_factor = (
            distance - segment_start_distance
        ) / segment_length

        point = (
            waypoints_xy[segment_index]
            + interpolation_factor
            * segment_vectors[segment_index]
        )

        sampled_points.append(point)

    return np.asarray(
        sampled_points,
        dtype=np.float64,
    )


# ------------------------------------------------------------
# Build route waypoints near the five gNB sites
# ------------------------------------------------------------

site_waypoints = []

for item in GNB_CONFIG:

    site_xy = np.asarray(
        item["position_m"][:2],
        dtype=np.float64,
    )

    direction_to_activity_center = (
        np.asarray(ACTIVITY_CENTER_XY)
        - site_xy
    )

    direction_norm = np.linalg.norm(
        direction_to_activity_center
    )

    if direction_norm > 1e-6:
        direction_unit = (
            direction_to_activity_center
            / direction_norm
        )
    else:
        direction_unit = np.zeros(2)

    # Place the UE route near, but not on top of,
    # each gNB building.
    roadside_xy = (
        site_xy
        + ROUTE_CONFIG["site_offset_m"]
        * direction_unit
    )

    site_waypoints.append(
        {
            "gNB": item["short_label"],
            "building": item["building_name"],
            "xy": roadside_xy,
        }
    )

# ------------------------------------------------------------
# Order waypoints using a nearest-neighbor route
# ------------------------------------------------------------

ordered_waypoints = [
    site_waypoints[0]
]

remaining_waypoints = site_waypoints[1:].copy()

while remaining_waypoints:

    current_xy = ordered_waypoints[-1]["xy"]

    next_waypoint = min(
        remaining_waypoints,
        key=lambda item: np.linalg.norm(
            item["xy"] - current_xy
        )
    )

    ordered_waypoints.append(next_waypoint)
    remaining_waypoints.remove(next_waypoint)

base_waypoints_xy = np.array(
    [
        item["xy"]
        for item in ordered_waypoints
    ],
    dtype=np.float64,
)

print("UE route waypoint order:")
for item in ordered_waypoints:
    print(
        f"{item['gNB']} | "
        f"{item['building']} | "
        f"{np.round(item['xy'], 2).tolist()}"
    )

# ------------------------------------------------------------
# Generate six spatially different trajectories
# ------------------------------------------------------------

trajectory_offsets = np.array(
    [
        [-25.0, -20.0],
        [-15.0, 15.0],
        [0.0, 0.0],
        [15.0, -10.0],
        [25.0, 20.0],
        [40.0, -25.0],
    ],
    dtype=np.float64,
)

route_bank = []

for trajectory_id in range(
    ROUTE_CONFIG["num_trajectories"]
):

    offset = trajectory_offsets[
        trajectory_id
        % len(trajectory_offsets)
    ]

    shifted_waypoints = (
        base_waypoints_xy + offset
    )

    route_xy = sample_polyline(
        shifted_waypoints,
        ROUTE_CONFIG[
            "points_per_trajectory"
        ],
    )

    route_z = np.full(
        (
            len(route_xy),
            1,
        ),
        ROUTE_CONFIG["ue_height_m"],
        dtype=np.float64,
    )

    route_xyz = np.hstack(
        [
            route_xy,
            route_z,
        ]
    )

    route_bank.append(
        {
            "trajectory_id": trajectory_id,
            "speed_mps": ROUTE_CONFIG[
                "speeds_mps"
            ][
                trajectory_id
                % len(ROUTE_CONFIG["speeds_mps"])
            ],
            "positions": route_xyz,
            "offset_xy": offset,
        }
    )

# Use trajectory 0 for the current interactive map
ue_route = route_bank[0]["positions"]

total_observations = sum(
    len(route["positions"])
    for route in route_bank
)

print("\nTrajectory bank created.")
print("Number of trajectories:", len(route_bank))
print(
    "Points per trajectory:",
    ROUTE_CONFIG["points_per_trajectory"],
)
print("Total UE observations:", total_observations)
print(
    "First UE position:",
    np.round(route_bank[0]["positions"][0], 2)
)
print(
    "Last UE position:",
    np.round(route_bank[0]["positions"][-1], 2)
)

# ------------------------------------------------------------
# Save trajectory bank separately
# ------------------------------------------------------------

trajectory_positions = np.stack(
    [
        route["positions"]
        for route in route_bank
    ],
    axis=0,
)

trajectory_speeds = np.array(
    [
        route["speed_mps"]
        for route in route_bank
    ],
    dtype=np.float64,
)

trajectory_offsets_array = np.stack(
    [
        route["offset_xy"]
        for route in route_bank
    ],
    axis=0,
)

trajectory_file = (
    IMPROVED_DATA_DIR
    / "dense_ue_trajectory_bank.npz"
)

np.savez_compressed(
    trajectory_file,
    positions=trajectory_positions,
    speeds_mps=trajectory_speeds,
    offsets_xy=trajectory_offsets_array,
    time_step_s=ROUTE_CONFIG["time_step_s"],
)

print("\nTrajectory data saved to:")
print(trajectory_file)

In [ ]:
# ============================================================
# Add visualization-only roads and vegetation
# ============================================================

if "ue_route" not in globals():
    raise NameError(
        "ue_route is not defined. Run the dense trajectory cell first."
    )

if "make_icon_digital_twin_map" not in globals():
    raise NameError(
        "Run the interactive PNG map function cell first."
    )

# ------------------------------------------------------------
# Schematic road corridor derived from the UE route
# ------------------------------------------------------------

route_for_roads = np.asarray(
    ue_route,
    dtype=np.float64,
)

# Reduce the number of rectangles for browser performance
road_step = max(
    1,
    len(route_for_roads) // 40
)

road_centerline = route_for_roads[
    ::road_step
]

if not np.array_equal(
    road_centerline[-1],
    route_for_roads[-1],
):
    road_centerline = np.vstack(
        [
            road_centerline,
            route_for_roads[-1],
        ]
    )

schematic_road_objects = []

road_half_width_m = 11.0

for point_a, point_b in zip(
    road_centerline[:-1],
    road_centerline[1:],
):

    xmin = min(point_a[0], point_b[0])
    xmax = max(point_a[0], point_b[0])
    ymin = min(point_a[1], point_b[1])
    ymax = max(point_a[1], point_b[1])

    schematic_road_objects.append(
        {
            "name": (
                "Route-aligned roadway corridor "
                "(schematic)"
            ),
            "material": "visualization_only",
            "bbox_min": np.array(
                [
                    xmin - road_half_width_m,
                    ymin - road_half_width_m,
                    0.0,
                ],
                dtype=np.float64,
            ),
            "bbox_max": np.array(
                [
                    xmax + road_half_width_m,
                    ymax + road_half_width_m,
                    0.3,
                ],
                dtype=np.float64,
            ),
            "center_xy": np.array(
                [
                    0.5 * (xmin + xmax),
                    0.5 * (ymin + ymax),
                ],
                dtype=np.float64,
            ),
        }
    )

# Use detected roads if available; otherwise use the
# route-aligned schematic roadway
if len(road_objects) == 0:
    road_objects = schematic_road_objects
    roads_are_schematic = True
else:
    roads_are_schematic = False


# ------------------------------------------------------------
# Schematic vegetation clusters around building footprints
# ------------------------------------------------------------

if len(vegetation_objects) == 0:

    rng = np.random.default_rng(2026)
    schematic_vegetation_objects = []

    for building_index, building in enumerate(
        meaningful_buildings
    ):

        building_center = np.asarray(
            building["center_xy"],
            dtype=np.float64,
        )

        bbox_min = np.asarray(
            building["bbox_min"],
            dtype=np.float64,
        )

        bbox_max = np.asarray(
            building["bbox_max"],
            dtype=np.float64,
        )

        # Three deterministic landscaping points around
        # each named building
        for plant_index in range(3):

            angle = (
                2.0
                * np.pi
                * plant_index
                / 3.0
                + 0.35
            )

            radius = rng.uniform(
                18.0,
                38.0,
            )

            candidate_xy = (
                building_center
                + radius
                * np.array(
                    [
                        np.cos(angle),
                        np.sin(angle),
                    ]
                )
            )

            # Avoid placing a vegetation marker inside
            # the building footprint
            if (
                bbox_min[0] - 3.0
                <= candidate_xy[0]
                <= bbox_max[0] + 3.0
                and
                bbox_min[1] - 3.0
                <= candidate_xy[1]
                <= bbox_max[1] + 3.0
            ):
                candidate_xy = (
                    building_center
                    + 1.8
                    * radius
                    * np.array(
                        [
                            np.cos(angle),
                            np.sin(angle),
                        ]
                    )
                )

            schematic_vegetation_objects.append(
                {
                    "name": (
                        "Landscaping tree cluster "
                        "(schematic)"
                    ),
                    "material": "visualization_only",
                    "bbox_min": np.array(
                        [
                            candidate_xy[0] - 2.0,
                            candidate_xy[1] - 2.0,
                            0.0,
                        ],
                        dtype=np.float64,
                    ),
                    "bbox_max": np.array(
                        [
                            candidate_xy[0] + 2.0,
                            candidate_xy[1] + 2.0,
                            5.0,
                        ],
                        dtype=np.float64,
                    ),
                    "center_xy": candidate_xy,
                }
            )

    vegetation_objects = schematic_vegetation_objects
    vegetation_is_schematic = True

else:
    vegetation_is_schematic = False


print(
    "Road objects used:",
    len(road_objects),
)

print(
    "Vegetation objects used:",
    len(vegetation_objects),
)

print(
    "Road layer:",
    "schematic route-aligned"
    if roads_are_schematic
    else "scene-derived",
)

print(
    "Vegetation layer:",
    "schematic landscaping"
    if vegetation_is_schematic
    else "scene-derived",
)


# ============================================================
# Regenerate the maps
# ============================================================

icon_overview = make_icon_digital_twin_map(
    radius_m=None,
    title=(
        "KFUPM Digital Twin — "
        "Five-gNB Campus Overview"
    ),
    output_filename=(
        "kfupm_digital_twin_icon_overview_context.html"
    ),
)

icon_focus = make_icon_digital_twin_map(
    radius_m=1000.0,
    title=(
        "KFUPM Digital Twin — "
        "Activity-Area Close-Up"
    ),
    output_filename=(
        "kfupm_digital_twin_icon_activity_context.html"
    ),
)

# Make the visualization-layer labels explicit
for figure in [
    icon_overview,
    icon_focus,
]:

    for trace in figure.data:

        if trace.name == "Roads":
            trace.name = (
                "Road corridor (schematic)"
            )

        if trace.name == "Vegetation":
            trace.name = (
                "Landscaping (schematic)"
            )

    figure.write_html(
        str(
            IMPROVED_FIGURES_DIR
            / (
                "kfupm_digital_twin_context_"
                + (
                    "overview.html"
                    if figure is icon_overview
                    else "activity_area.html"
                )
            )
        ),
        include_plotlyjs=True,
        full_html=True,
    )

print("\nUpdated figures saved in:")
print(IMPROVED_FIGURES_DIR)

icon_focus.show()

In [ ]:
# ============================================================
# Five-gNB GPU ray-tracing pilot — corrected version
# ============================================================

import json
import time
import numpy as np

# Create a deterministic path solver
path_solver = rt.PathSolver(
    deterministic=True
)

# Get the UE receiver from the scene
ue_receiver = scene.get(
    UE_CONFIG["id"]
)

# Select representative points along trajectory 0
pilot_indices = np.linspace(
    0,
    len(ue_route) - 1,
    8,
    dtype=int,
)

pilot_records = []

pilot_start_time = time.perf_counter()

try:

    for pilot_number, route_index in enumerate(
        pilot_indices,
        start=1,
    ):

        # Ensure a native Python integer
        route_index = int(route_index)

        ue_position = np.asarray(
            ue_route[route_index],
            dtype=np.float64,
        )

        # Move the UE receiver
        ue_receiver.position = tuple(
            float(value)
            for value in ue_position
        )

        point_start_time = time.perf_counter()

        # Use a native Python integer for Dr.Jit
        pilot_seed = int(
            DATA_CONFIG["random_seed"]
        ) + route_index

        paths = path_solver(
            scene=scene,
            max_depth=int(
                DATA_CONFIG["max_depth"]
            ),
            los=bool(
                DATA_CONFIG["los"]
            ),
            specular_reflection=bool(
                DATA_CONFIG[
                    "specular_reflection"
                ]
            ),
            diffuse_reflection=bool(
                DATA_CONFIG[
                    "diffuse_reflection"
                ]
            ),
            refraction=bool(
                DATA_CONFIG["refraction"]
            ),
            synthetic_array=bool(
                DATA_CONFIG[
                    "synthetic_array"
                ]
            ),
            seed=pilot_seed,
        )

        point_runtime = (
            time.perf_counter()
            - point_start_time
        )

        # Convert Dr.Jit outputs to NumPy
        valid_array = np.asarray(
            paths.valid
        )

        delay_array = np.asarray(
            paths.tau
        )

        # Remove the source/receiver batch dimension
        if (
            valid_array.ndim == 3
            and valid_array.shape[0] == 1
        ):
            valid_array = valid_array[0]

        if (
            delay_array.ndim == 3
            and delay_array.shape[0] == 1
        ):
            delay_array = delay_array[0]

        valid_counts = []
        minimum_delays = []

        for gnb_index in range(
            len(GNB_CONFIG)
        ):

            valid_mask = (
                valid_array[gnb_index] > 0
            )

            valid_count = int(
                np.sum(valid_mask)
            )

            valid_counts.append(
                valid_count
            )

            if valid_count > 0:

                valid_delays = (
                    delay_array[gnb_index][
                        valid_mask
                    ]
                )

                valid_delays = valid_delays[
                    np.isfinite(valid_delays)
                ]

                valid_delays = valid_delays[
                    valid_delays >= 0.0
                ]

                if len(valid_delays) > 0:
                    minimum_delays.append(
                        float(
                            np.min(
                                valid_delays
                            )
                        )
                    )
                else:
                    minimum_delays.append(
                        None
                    )

            else:
                minimum_delays.append(
                    None
                )

        pilot_record = {
            "pilot_number": int(
                pilot_number
            ),
            "route_index": route_index,
            "position_xyz_m": (
                ue_position.tolist()
            ),
            "valid_path_counts": (
                valid_counts
            ),
            "minimum_delays_s": (
                minimum_delays
            ),
            "runtime_s": float(
                point_runtime
            ),
        }

        pilot_records.append(
            pilot_record
        )

        print(
            f"Pilot {pilot_number}/"
            f"{len(pilot_indices)} | "
            f"route index {route_index:3d} | "
            f"valid paths {valid_counts} | "
            f"time {point_runtime:.2f} s"
        )

finally:

    # Always restore the initial UE position
    ue_receiver.position = tuple(
        float(value)
        for value in UE_CONFIG[
            "initial_position_m"
        ]
    )


total_pilot_runtime = (
    time.perf_counter()
    - pilot_start_time
)

pilot_output = {
    "number_of_gnbs": len(GNB_CONFIG),
    "number_of_pilot_points": len(
        pilot_indices
    ),
    "total_runtime_s": float(
        total_pilot_runtime
    ),
    "average_runtime_s": float(
        total_pilot_runtime
        / len(pilot_indices)
    ),
    "records": pilot_records,
}

pilot_file = (
    IMPROVED_RESULTS_DIR
    / "five_gnb_ray_tracing_pilot.json"
)

with open(
    pilot_file,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        pilot_output,
        file,
        indent=2,
    )

print("\nFive-gNB pilot completed.")
print(
    "Average runtime per point:",
    round(
        pilot_output[
            "average_runtime_s"
        ],
        3,
    ),
    "seconds",
)
print("Pilot results saved to:")
print(pilot_file)

In [ ]:
# ============================================================
# Dense five-gNB RF dataset generation with checkpointing
# ============================================================

import csv
import json
import time
import numpy as np

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_GNBS = len(GNB_CONFIG)
CHECKPOINT_INTERVAL = 50

if "route_bank" not in globals():
    raise NameError(
        "route_bank is not defined. "
        "Run the dense UE trajectory cell first."
    )

if N_GNBS != 5:
    print(
        "Warning: expected five gNBs, found",
        N_GNBS,
    )

# ------------------------------------------------------------
# Flatten all trajectories into one observation list
# ------------------------------------------------------------

observation_rows = []

for trajectory in route_bank:

    trajectory_id = int(
        trajectory["trajectory_id"]
    )

    speed_mps = float(
        trajectory["speed_mps"]
    )

    positions = np.asarray(
        trajectory["positions"],
        dtype=np.float64,
    )

    for route_index, position in enumerate(
        positions
    ):

        observation_rows.append(
            {
                "trajectory_id": trajectory_id,
                "route_index": int(route_index),
                "speed_mps": speed_mps,
                "position_xyz_m": position,
            }
        )

N_OBSERVATIONS = len(
    observation_rows
)

positions = np.asarray(
    [
        row["position_xyz_m"]
        for row in observation_rows
    ],
    dtype=np.float64,
)

trajectory_ids = np.asarray(
    [
        row["trajectory_id"]
        for row in observation_rows
    ],
    dtype=np.int32,
)

route_indices = np.asarray(
    [
        row["route_index"]
        for row in observation_rows
    ],
    dtype=np.int32,
)

speeds_mps = np.asarray(
    [
        row["speed_mps"]
        for row in observation_rows
    ],
    dtype=np.float64,
)

# ------------------------------------------------------------
# Preallocate output arrays
# ------------------------------------------------------------

rf_power_db = np.full(
    (N_OBSERVATIONS, N_GNBS),
    -300.0,
    dtype=np.float32,
)

valid_path_counts = np.zeros(
    (N_OBSERVATIONS, N_GNBS),
    dtype=np.int32,
)

minimum_delay_s = np.full(
    (N_OBSERVATIONS, N_GNBS),
    -1.0,
    dtype=np.float64,
)

completed = np.zeros(
    N_OBSERVATIONS,
    dtype=bool,
)

# ------------------------------------------------------------
# Checkpoint support
# ------------------------------------------------------------

checkpoint_file = (
    IMPROVED_RESULTS_DIR
    / "rf_dense_dataset_checkpoint.npz"
)

if checkpoint_file.exists():

    print("Checkpoint found. Attempting to resume...")

    checkpoint = np.load(
        checkpoint_file,
        allow_pickle=False,
    )

    checkpoint_valid = (
        checkpoint["rf_power_db"].shape
        == rf_power_db.shape
        and checkpoint[
            "valid_path_counts"
        ].shape
        == valid_path_counts.shape
        and checkpoint[
            "minimum_delay_s"
        ].shape
        == minimum_delay_s.shape
    )

    if checkpoint_valid:

        rf_power_db[:] = checkpoint[
            "rf_power_db"
        ]

        valid_path_counts[:] = checkpoint[
            "valid_path_counts"
        ]

        minimum_delay_s[:] = checkpoint[
            "minimum_delay_s"
        ]

        completed[:] = checkpoint[
            "completed"
        ]

        print(
            "Completed observations loaded:",
            int(completed.sum()),
        )

    else:

        print(
            "Checkpoint dimensions do not match "
            "the current scenario. Starting again."
        )


def save_checkpoint():

    np.savez_compressed(
        checkpoint_file,
        rf_power_db=rf_power_db,
        valid_path_counts=valid_path_counts,
        minimum_delay_s=minimum_delay_s,
        completed=completed,
    )


# ------------------------------------------------------------
# Ray-tracing loop
# ------------------------------------------------------------

full_path_solver = rt.PathSolver(
    deterministic=True
)

full_ue_receiver = scene.get(
    UE_CONFIG["id"]
)

run_start_time = time.perf_counter()
processed_since_start = 0

try:

    for observation_index in range(
        N_OBSERVATIONS
    ):

        if completed[observation_index]:
            continue

        observation = observation_rows[
            observation_index
        ]

        route_position = np.asarray(
            observation["position_xyz_m"],
            dtype=np.float64,
        )

        full_ue_receiver.position = tuple(
            float(value)
            for value in route_position
        )

        paths = full_path_solver(
            scene=scene,
            max_depth=int(
                DATA_CONFIG["max_depth"]
            ),
            los=bool(
                DATA_CONFIG["los"]
            ),
            specular_reflection=bool(
                DATA_CONFIG[
                    "specular_reflection"
                ]
            ),
            diffuse_reflection=bool(
                DATA_CONFIG[
                    "diffuse_reflection"
                ]
            ),
            refraction=bool(
                DATA_CONFIG["refraction"]
            ),
            synthetic_array=bool(
                DATA_CONFIG[
                    "synthetic_array"
                ]
            ),
            seed=(
                int(
                    DATA_CONFIG[
                        "random_seed"
                    ]
                )
                + int(observation_index)
            ),
        )

        # ----------------------------------------------------
        # Valid path counts
        # ----------------------------------------------------

        valid_array = np.asarray(
            paths.valid
        )

        if (
            valid_array.ndim == 3
            and valid_array.shape[0] == 1
        ):
            valid_array = valid_array[0]

        valid_array = (
            valid_array > 0
        )

        valid_path_counts[
            observation_index,
            :,
        ] = valid_array.sum(
            axis=-1
        ).astype(np.int32)

        # ----------------------------------------------------
        # Path delays
        # ----------------------------------------------------

        delay_array = np.asarray(
            paths.tau
        )

        if (
            delay_array.ndim == 3
            and delay_array.shape[0] == 1
        ):
            delay_array = delay_array[0]

        for gnb_index in range(
            N_GNBS
        ):

            gnb_valid = valid_array[
                gnb_index
            ]

            if np.any(gnb_valid):

                gnb_delays = (
                    delay_array[
                        gnb_index
                    ][gnb_valid]
                )

                gnb_delays = gnb_delays[
                    np.isfinite(gnb_delays)
                ]

                gnb_delays = gnb_delays[
                    gnb_delays >= 0.0
                ]

                if len(gnb_delays) > 0:

                    minimum_delay_s[
                        observation_index,
                        gnb_index,
                    ] = np.min(
                        gnb_delays
                    )

        # ----------------------------------------------------
        # RF path-coefficient power
        # ----------------------------------------------------

        path_real = np.asarray(
            paths.a[0]
        )

        path_imag = np.asarray(
            paths.a[1]
        )

        # Expected shape before squeezing:
        # (1, RX antennas, gNBs, TX antennas, paths)
        if (
            path_real.ndim == 5
            and path_real.shape[0] == 1
        ):
            path_real = path_real[0]
            path_imag = path_imag[0]

        # Shape:
        # (RX antennas, gNBs, TX antennas, paths)
        path_power = (
            path_real ** 2
            + path_imag ** 2
        )

        # Broadcast valid-path mask:
        # (1, gNBs, 1, paths)
        valid_mask = valid_array[
            np.newaxis,
            :,
            np.newaxis,
            :,
        ]

        path_power = (
            path_power
            * valid_mask
        )

        # Sum over paths, then average over
        # RX and TX antenna dimensions
        power_linear = (
            path_power.sum(axis=-1)
            .mean(axis=(0, 2))
        )

        power_db = 10.0 * np.log10(
            np.maximum(
                power_linear,
                1e-30,
            )
        )

        power_db = np.maximum(
            power_db,
            -300.0,
        )

        rf_power_db[
            observation_index,
            :,
        ] = power_db.astype(
            np.float32
        )

        completed[
            observation_index
        ] = True

        processed_since_start += 1

        # ----------------------------------------------------
        # Periodic checkpoint and progress report
        # ----------------------------------------------------

        if (
            processed_since_start
            % CHECKPOINT_INTERVAL
            == 0
            or observation_index
            == N_OBSERVATIONS - 1
        ):

            save_checkpoint()

            elapsed = (
                time.perf_counter()
                - run_start_time
            )

            completed_count = int(
                completed.sum()
            )

            remaining_count = (
                N_OBSERVATIONS
                - completed_count
            )

            average_time = (
                elapsed
                / max(
                    processed_since_start,
                    1,
                )
            )

            estimated_remaining = (
                remaining_count
                * average_time
                / 60.0
            )

            print(
                f"Processed "
                f"{completed_count}/"
                f"{N_OBSERVATIONS} | "
                f"average "
                f"{average_time:.2f} s/point | "
                f"estimated remaining "
                f"{estimated_remaining:.1f} min"
            )

finally:

    # Always restore initial UE position
    full_ue_receiver.position = tuple(
        float(value)
        for value in UE_CONFIG[
            "initial_position_m"
        ]
    )

# ------------------------------------------------------------
# Save final RF dataset
# ------------------------------------------------------------

final_rf_file = (
    IMPROVED_DATA_DIR
    / "rf_dense_five_gnb_dataset.npz"
)

np.savez_compressed(
    final_rf_file,
    positions_xyz_m=positions,
    trajectory_ids=trajectory_ids,
    route_indices=route_indices,
    speeds_mps=speeds_mps,
    rf_power_db=rf_power_db,
    valid_path_counts=valid_path_counts,
    minimum_delay_s=minimum_delay_s,
    completed=completed,
    gnb_names=np.asarray(
        GNB_NAMES
    ),
)

# ------------------------------------------------------------
# Save a human-readable summary CSV
# ------------------------------------------------------------

summary_csv = (
    IMPROVED_RESULTS_DIR
    / "rf_dense_five_gnb_summary.csv"
)

with open(
    summary_csv,
    "w",
    newline="",
    encoding="utf-8",
) as file:

    writer = csv.writer(file)

    header = [
        "observation_index",
        "trajectory_id",
        "route_index",
        "x_m",
        "y_m",
        "z_m",
        "speed_mps",
    ]

    for gnb_name in GNB_NAMES:
        header.extend(
            [
                f"{gnb_name}_rf_power_db",
                f"{gnb_name}_valid_paths",
                f"{gnb_name}_minimum_delay_s",
            ]
        )

    writer.writerow(header)

    for observation_index in range(
        N_OBSERVATIONS
    ):

        row = [
            observation_index,
            int(
                trajectory_ids[
                    observation_index
                ]
            ),
            int(
                route_indices[
                    observation_index
                ]
            ),
            float(
                positions[
                    observation_index,
                    0,
                ]
            ),
            float(
                positions[
                    observation_index,
                    1,
                ]
            ),
            float(
                positions[
                    observation_index,
                    2,
                ]
            ),
            float(
                speeds_mps[
                    observation_index
                ]
            ),
        ]

        for gnb_index in range(
            N_GNBS
        ):

            row.extend(
                [
                    float(
                        rf_power_db[
                            observation_index,
                            gnb_index,
                        ]
                    ),
                    int(
                        valid_path_counts[
                            observation_index,
                            gnb_index,
                        ]
                    ),
                    float(
                        minimum_delay_s[
                            observation_index,
                            gnb_index,
                        ]
                    ),
                ]
            )

        writer.writerow(row)

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

metadata_file = (
    IMPROVED_RESULTS_DIR
    / "rf_dense_five_gnb_metadata.json"
)

metadata = {
    "number_of_observations": N_OBSERVATIONS,
    "number_of_gnbs": N_GNBS,
    "gnb_names": GNB_NAMES,
    "gnb_buildings": [
        item["building_name"]
        for item in GNB_CONFIG
    ],
    "carrier_frequency_hz": float(
        RADIO_CONFIG[
            "carrier_frequency_hz"
        ]
    ),
    "max_depth": int(
        DATA_CONFIG["max_depth"]
    ),
    "trajectory_count": len(
        route_bank
    ),
    "points_per_trajectory": int(
        ROUTE_CONFIG[
            "points_per_trajectory"
        ]
    ),
}

with open(
    metadata_file,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=2,
    )

print("\nDense RF dataset completed.")
print("Dataset:", final_rf_file)
print("Summary:", summary_csv)
print("Metadata:", metadata_file)

In [ ]:
# ============================================================
# Validate completed dense five-gNB RF dataset
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.home() / "wcnc_project"
EXPERIMENT_ROOT = PROJECT_ROOT / "improved_experiment"

DATA_FILE = EXPERIMENT_ROOT / "data" / "rf_dense_five_gnb_dataset.npz"
SUMMARY_FILE = EXPERIMENT_ROOT / "results" / "rf_dense_five_gnb_summary.csv"
METADATA_FILE = EXPERIMENT_ROOT / "results" / "rf_dense_five_gnb_metadata.json"
VALIDATION_FILE = EXPERIMENT_ROOT / "results" / "rf_dense_validation_report.json"

for file_path in [DATA_FILE, SUMMARY_FILE, METADATA_FILE]:
    print(file_path)
    print("Exists:", file_path.exists())
    if file_path.exists():
        print("Size:", round(file_path.stat().st_size / 1024, 2), "KB")
    print()

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found:\n{DATA_FILE}")

# Load NPZ dataset
with np.load(DATA_FILE, allow_pickle=True) as dataset_file:
    dataset_keys = list(dataset_file.files)
    dataset_arrays = {
        key: dataset_file[key]
        for key in dataset_keys
    }

print("Dataset keys:")
print(dataset_keys)

def get_array(candidates, required=False):
    for name in candidates:
        if name in dataset_arrays:
            return dataset_arrays[name], name

    if required:
        raise KeyError(
            f"Required array not found. Tried: {candidates}"
        )

    return None, None


rf_power_db, rf_key = get_array(
    ["rf_power_db"],
    required=True
)

valid_path_counts, valid_key = get_array(
    ["valid_path_counts"],
    required=True
)

minimum_delay_s, delay_key = get_array(
    ["minimum_delay_s", "minimum_delays_s"],
    required=True
)

positions_xyz_m, position_key = get_array(
    ["positions_xyz_m", "positions", "ue_positions"]
)

route_indices, route_key = get_array(
    ["route_indices", "route_index"]
)

trajectory_ids, trajectory_key = get_array(
    ["trajectory_ids", "trajectory_id"]
)

completed_mask, completed_key = get_array(
    ["completed", "completed_mask"]
)

print("\nDetected arrays:")
print("RF power:", rf_key, rf_power_db.shape)
print("Valid paths:", valid_key, valid_path_counts.shape)
print("Minimum delays:", delay_key, minimum_delay_s.shape)

if positions_xyz_m is not None:
    print("UE positions:", position_key, positions_xyz_m.shape)

if route_indices is not None:
    print("Route indices:", route_key, route_indices.shape)

if trajectory_ids is not None:
    print("Trajectory IDs:", trajectory_key, trajectory_ids.shape)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

EXPECTED_OBSERVATIONS = 6 * 240
EXPECTED_GNBS = 5

errors = []
warnings = []

if rf_power_db.shape != (EXPECTED_OBSERVATIONS, EXPECTED_GNBS):
    errors.append(
        f"Unexpected RF shape: {rf_power_db.shape}. "
        f"Expected {(EXPECTED_OBSERVATIONS, EXPECTED_GNBS)}."
    )

if valid_path_counts.shape != rf_power_db.shape:
    errors.append(
        "Valid-path tensor shape does not match RF power shape."
    )

if minimum_delay_s.shape != rf_power_db.shape:
    errors.append(
        "Delay tensor shape does not match RF power shape."
    )

if not np.all(np.isfinite(rf_power_db)):
    errors.append("RF power contains NaN or infinite values.")

if not np.all(np.isfinite(valid_path_counts)):
    errors.append("Valid-path counts contain NaN or infinite values.")

if not np.all(valid_path_counts >= 0):
    errors.append("Valid-path counts contain negative values.")

valid_delay_values = minimum_delay_s[minimum_delay_s >= 0]

if not np.all(np.isfinite(valid_delay_values)):
    errors.append(
        "Valid minimum delays contain NaN or infinite values."
    )

if np.any(valid_delay_values < 0):
    errors.append("Valid delays contain negative values.")

if positions_xyz_m is not None:
    if not np.all(np.isfinite(positions_xyz_m)):
        errors.append("UE positions contain NaN or infinite values.")

if completed_mask is not None:
    print("\nCompleted-mask status:")
    print("Completed observations:", int(np.sum(completed_mask)))
    print("Total observations:", len(completed_mask))

    if not np.all(completed_mask):
        warnings.append(
            "Some observations are marked incomplete."
        )

# ------------------------------------------------------------
# Per-gNB summary
# ------------------------------------------------------------

gnb_labels = globals().get(
    "GNB_NAMES",
    [f"gNB{i+1}" for i in range(EXPECTED_GNBS)]
)

per_gnb_summary = []

for gnb_index in range(EXPECTED_GNBS):

    power_column = rf_power_db[:, gnb_index]
    path_column = valid_path_counts[:, gnb_index]
    delay_column = minimum_delay_s[:, gnb_index]

    # -300 dB is the dataset sentinel for no usable RF path
    usable_power = power_column > -299.0
    usable_delay = delay_column >= 0.0

    record = {
        "gNB": str(gnb_labels[gnb_index]),
        "usable_rf_fraction": float(
            np.mean(usable_power)
        ),
        "valid_path_fraction": float(
            np.mean(path_column > 0)
        ),
        "mean_rf_power_db": (
            float(np.mean(power_column[usable_power]))
            if np.any(usable_power)
            else None
        ),
        "minimum_rf_power_db": (
            float(np.min(power_column[usable_power]))
            if np.any(usable_power)
            else None
        ),
        "maximum_valid_path_count": int(
            np.max(path_column)
        ),
        "mean_valid_path_count": float(
            np.mean(path_column)
        ),
        "mean_delay_us": (
            float(np.mean(delay_column[usable_delay]) * 1e6)
            if np.any(usable_delay)
            else None
        ),
    }

    per_gnb_summary.append(record)

summary_table = pd.DataFrame(per_gnb_summary)

print("\nPer-gNB dataset summary:")
display(summary_table)

# ------------------------------------------------------------
# Load auxiliary outputs
# ------------------------------------------------------------

if SUMMARY_FILE.exists():
    summary_csv = pd.read_csv(SUMMARY_FILE)
    print("\nSummary CSV shape:", summary_csv.shape)
    print("Summary CSV columns:")
    print(summary_csv.columns.tolist())

metadata = {}

if METADATA_FILE.exists():
    with open(METADATA_FILE, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    print("\nMetadata keys:")
    print(list(metadata.keys()))

# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

validation_report = {
    "dataset_file": str(DATA_FILE),
    "number_of_observations": int(rf_power_db.shape[0]),
    "number_of_gnbs": int(rf_power_db.shape[1]),
    "rf_power_shape": list(rf_power_db.shape),
    "valid_path_shape": list(valid_path_counts.shape),
    "minimum_delay_shape": list(minimum_delay_s.shape),
    "errors": errors,
    "warnings": warnings,
    "per_gnb_summary": per_gnb_summary,
}

with open(VALIDATION_FILE, "w", encoding="utf-8") as file:
    json.dump(validation_report, file, indent=2)

print("\nValidation report saved to:")
print(VALIDATION_FILE)

if errors:
    print("\nDATASET VALIDATION FAILED")
    for error in errors:
        print("ERROR:", error)
else:
    print("\nDATASET VALIDATION PASSED")

if warnings:
    print("\nWarnings:")
    for warning in warnings:
        print("WARNING:", warning)

In [ ]:
# ============================================================
# Dense five-gNB route and RF diagnostic figure
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.home() / "wcnc_project"
EXPERIMENT_ROOT = PROJECT_ROOT / "improved_experiment"
FIGURE_DIR = EXPERIMENT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    EXPERIMENT_ROOT
    / "data"
    / "rf_dense_five_gnb_dataset.npz"
)

with np.load(DATA_FILE, allow_pickle=True) as data:

    positions_xyz_m = data["positions_xyz_m"].astype(float)
    trajectory_ids = data["trajectory_ids"].astype(int)
    route_indices = data["route_indices"].astype(int)
    speeds_mps = data["speeds_mps"].astype(float)
    rf_power_db = data["rf_power_db"].astype(float)
    valid_path_counts = data["valid_path_counts"].astype(float)
    minimum_delay_s = data["minimum_delay_s"].astype(float)

    raw_gnb_names = data["gnb_names"].tolist()

def clean_name(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)

gnb_names = [
    clean_name(value)
    for value in raw_gnb_names
]

number_of_gnbs = rf_power_db.shape[1]
observation_indices = np.arange(
    rf_power_db.shape[0]
)

# -300 dB and negative delays are sentinel values
rf_plot = np.where(
    rf_power_db <= -299.0,
    np.nan,
    rf_power_db
)

delay_plot_us = np.where(
    minimum_delay_s < 0.0,
    np.nan,
    minimum_delay_s * 1e6
)

# IEEE-style plotting configuration
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.linewidth": 1.0,
    "lines.linewidth": 1.5,
})

gnb_colors = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
]

trajectory_colors = [
    "#1B9E77",
    "#D95F02",
    "#7570B3",
    "#E7298A",
    "#66A61E",
    "#E6AB02",
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
    constrained_layout=True,
)

ax_map = axes[0, 0]
ax_power = axes[0, 1]
ax_paths = axes[1, 0]
ax_delay = axes[1, 1]

# ------------------------------------------------------------
# Panel (a): UE route bank and gNB locations
# ------------------------------------------------------------

unique_trajectories = np.unique(trajectory_ids)

for trajectory_index, trajectory_id in enumerate(
    unique_trajectories
):

    trajectory_mask = (
        trajectory_ids == trajectory_id
    )

    order = np.argsort(
        route_indices[trajectory_mask]
    )

    trajectory_positions = positions_xyz_m[
        trajectory_mask
    ][order]

    ax_map.plot(
        trajectory_positions[:, 0],
        trajectory_positions[:, 1],
        color=trajectory_colors[
            trajectory_index
            % len(trajectory_colors)
        ],
        linewidth=1.4,
        alpha=0.85,
        label=f"UE route {trajectory_id}",
    )

# Plot gNBs if configuration exists in the notebook kernel
if "GNB_CONFIG" in globals():

    for gnb_index, gnb_config in enumerate(
        GNB_CONFIG
    ):

        gnb_position = np.asarray(
            gnb_config["position_m"],
            dtype=float,
        )

        gnb_label = gnb_config.get(
            "short_label",
            f"gNB{gnb_index + 1}",
        )

        ax_map.scatter(
            gnb_position[0],
            gnb_position[1],
            s=95,
            marker="^",
            color=gnb_colors[
                gnb_index % len(gnb_colors)
            ],
            edgecolor="black",
            linewidth=0.8,
            zorder=5,
        )

        ax_map.annotate(
            gnb_label,
            xy=(
                gnb_position[0],
                gnb_position[1],
            ),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
        )

ax_map.set_title(
    "(a) KFUPM UE route bank and gNB sites"
)
ax_map.set_xlabel("x coordinate (m)")
ax_map.set_ylabel("y coordinate (m)")
ax_map.grid(
    True,
    alpha=0.25,
    linewidth=0.7,
)
ax_map.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Panel (b): RF power
# ------------------------------------------------------------

for gnb_index in range(number_of_gnbs):

    ax_power.plot(
        observation_indices,
        rf_plot[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

ax_power.set_title(
    "(b) Ray-traced RF power"
)
ax_power.set_xlabel("Observation index")
ax_power.set_ylabel("Path-based RF power (dB)")
ax_power.grid(
    True,
    alpha=0.25,
    linewidth=0.7,
)
ax_power.legend(
    loc="best",
    frameon=True,
)

# ------------------------------------------------------------
# Panel (c): Valid path counts
# ------------------------------------------------------------

for gnb_index in range(number_of_gnbs):

    ax_paths.plot(
        observation_indices,
        valid_path_counts[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

ax_paths.set_title(
    "(c) Number of valid propagation paths"
)
ax_paths.set_xlabel("Observation index")
ax_paths.set_ylabel("Valid path count")
ax_paths.grid(
    True,
    alpha=0.25,
    linewidth=0.7,
)

# ------------------------------------------------------------
# Panel (d): Minimum path delay
# ------------------------------------------------------------

for gnb_index in range(number_of_gnbs):

    ax_delay.plot(
        observation_indices,
        delay_plot_us[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

ax_delay.set_title(
    "(d) Minimum valid-path delay"
)
ax_delay.set_xlabel("Observation index")
ax_delay.set_ylabel("Minimum delay ($\\mu$s)")
ax_delay.grid(
    True,
    alpha=0.25,
    linewidth=0.7,
)

# Mark trajectory boundaries
trajectory_boundaries = (
    np.where(
        np.diff(trajectory_ids) != 0
    )[0]
    + 0.5
)

for axis in [
    ax_power,
    ax_paths,
    ax_delay,
]:

    for boundary in trajectory_boundaries:

        axis.axvline(
            boundary,
            color="black",
            linestyle="--",
            linewidth=0.7,
            alpha=0.35,
        )

# Save publication formats
pdf_file = (
    FIGURE_DIR
    / "dense_five_gnb_route_diagnostics.pdf"
)

png_file = (
    FIGURE_DIR
    / "dense_five_gnb_route_diagnostics.png"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
)

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
)

plt.show()

print("Dense route diagnostic figure saved:")
print(pdf_file)
print(png_file)

In [ ]:
# ============================================================
# Save separate title-free IEEE figures as PDF
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.home() / "wcnc_project"
EXPERIMENT_ROOT = PROJECT_ROOT / "improved_experiment"
FIGURE_DIR = EXPERIMENT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    EXPERIMENT_ROOT
    / "data"
    / "rf_dense_five_gnb_dataset.npz"
)

with np.load(DATA_FILE, allow_pickle=True) as data:

    positions_xyz_m = data["positions_xyz_m"].astype(float)
    trajectory_ids = data["trajectory_ids"].astype(int)
    route_indices = data["route_indices"].astype(int)
    rf_power_db = data["rf_power_db"].astype(float)
    valid_path_counts = data["valid_path_counts"].astype(float)
    minimum_delay_s = data["minimum_delay_s"].astype(float)
    raw_gnb_names = data["gnb_names"].tolist()

def clean_name(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)

gnb_names = [
    clean_name(value)
    for value in raw_gnb_names
]

number_of_gnbs = rf_power_db.shape[1]
observation_indices = np.arange(
    rf_power_db.shape[0]
)

# Treat unavailable links as missing values
rf_plot = np.where(
    rf_power_db <= -299.0,
    np.nan,
    rf_power_db
)

delay_plot_us = np.where(
    minimum_delay_s < 0.0,
    np.nan,
    minimum_delay_s * 1e6
)

# Large-font IEEE configuration
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 11,
    "axes.linewidth": 1.3,
    "lines.linewidth": 2.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

gnb_colors = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
]

trajectory_colors = [
    "#1B9E77",
    "#D95F02",
    "#7570B3",
    "#E7298A",
    "#66A61E",
    "#E6AB02",
]

unique_trajectories = np.unique(
    trajectory_ids
)

trajectory_boundaries = (
    np.where(
        np.diff(trajectory_ids) != 0
    )[0]
    + 0.5
)

# ------------------------------------------------------------
# Figure 1: UE route bank and gNB locations
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)

for trajectory_index, trajectory_id in enumerate(
    unique_trajectories
):

    trajectory_mask = (
        trajectory_ids == trajectory_id
    )

    order = np.argsort(
        route_indices[trajectory_mask]
    )

    trajectory_positions = positions_xyz_m[
        trajectory_mask
    ][order]

    ax.plot(
        trajectory_positions[:, 0],
        trajectory_positions[:, 1],
        color=trajectory_colors[
            trajectory_index
            % len(trajectory_colors)
        ],
        linewidth=2.0,
        alpha=0.85,
        label=f"UE route {trajectory_id}",
    )

if "GNB_CONFIG" in globals():

    for gnb_index, gnb_config in enumerate(
        GNB_CONFIG
    ):

        gnb_position = np.asarray(
            gnb_config["position_m"],
            dtype=float,
        )

        short_label = gnb_config.get(
            "short_label",
            f"gNB{gnb_index + 1}",
        )

        site_label = gnb_config.get(
            "site_label",
            gnb_config.get("id", ""),
        )

        ax.scatter(
            gnb_position[0],
            gnb_position[1],
            s=150,
            marker="^",
            color=gnb_colors[
                gnb_index % len(gnb_colors)
            ],
            edgecolor="black",
            linewidth=1.2,
            zorder=5,
        )

        ax.annotate(
            f"{short_label}\n{site_label}",
            xy=(
                gnb_position[0],
                gnb_position[1],
            ),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=11,
            fontweight="bold",
            bbox={
                "boxstyle": "round,pad=0.25",
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.85,
            },
        )

ax.set_xlabel("East-west coordinate (m)")
ax.set_ylabel("North-south coordinate (m)")
ax.grid(
    True,
    alpha=0.3,
    linewidth=0.8,
)
ax.set_aspect(
    "equal",
    adjustable="box",
)

ax.legend(
    loc="best",
    frameon=True,
    ncol=2,
)

route_file = (
    FIGURE_DIR
    / "kfupm_ue_route_bank.pdf"
)

fig.savefig(
    route_file,
    format="pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# Figure 2: RF power
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

for gnb_index in range(number_of_gnbs):

    ax.plot(
        observation_indices,
        rf_plot[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

for boundary in trajectory_boundaries:

    ax.axvline(
        boundary,
        color="black",
        linestyle="--",
        linewidth=1.0,
        alpha=0.35,
    )

ax.set_xlabel("Observation index")
ax.set_ylabel("Path-based RF power (dB)")
ax.grid(
    True,
    alpha=0.3,
    linewidth=0.8,
)

ax.legend(
    loc="best",
    frameon=True,
)

rf_file = (
    FIGURE_DIR
    / "rf_power_per_gnb.pdf"
)

fig.savefig(
    rf_file,
    format="pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# Figure 3: Valid path counts
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

for gnb_index in range(number_of_gnbs):

    ax.plot(
        observation_indices,
        valid_path_counts[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

for boundary in trajectory_boundaries:

    ax.axvline(
        boundary,
        color="black",
        linestyle="--",
        linewidth=1.0,
        alpha=0.35,
    )

ax.set_xlabel("Observation index")
ax.set_ylabel("Number of valid propagation paths")
ax.grid(
    True,
    alpha=0.3,
    linewidth=0.8,
)

ax.legend(
    loc="best",
    frameon=True,
)

path_file = (
    FIGURE_DIR
    / "valid_path_count_per_gnb.pdf"
)

fig.savefig(
    path_file,
    format="pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# Figure 4: Minimum path delay
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

for gnb_index in range(number_of_gnbs):

    ax.plot(
        observation_indices,
        delay_plot_us[:, gnb_index],
        color=gnb_colors[
            gnb_index % len(gnb_colors)
        ],
        label=gnb_names[gnb_index],
    )

for boundary in trajectory_boundaries:

    ax.axvline(
        boundary,
        color="black",
        linestyle="--",
        linewidth=1.0,
        alpha=0.35,
    )

ax.set_xlabel("Observation index")
ax.set_ylabel("Minimum path delay ($\\mu$s)")
ax.grid(
    True,
    alpha=0.3,
    linewidth=0.8,
)

ax.legend(
    loc="best",
    frameon=True,
)

delay_file = (
    FIGURE_DIR
    / "minimum_path_delay_per_gnb.pdf"
)

fig.savefig(
    delay_file,
    format="pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("Separate IEEE-style PDF figures saved:")
print(route_file)
print(rf_file)
print(path_file)
print(delay_file)

In [ ]:
# ============================================================
# Synchronized multimodal KFUPM dataset
# RF + mobility + scene-derived virtual LiDAR/range features
# ============================================================

from pathlib import Path
import csv
import json
import time
import hashlib

import numpy as np
import mitsuba as mi
import drjit as dr


# ------------------------------------------------------------
# 1. Paths and scene checks
# ------------------------------------------------------------

PROJECT_ROOT = Path.home() / "wcnc_project"

DATA_DIR = (
    PROJECT_ROOT
    / "improved_experiment"
    / "data"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "improved_experiment"
    / "results"
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DENSE_RF_FILE = (
    DATA_DIR
    / "rf_dense_five_gnb_dataset.npz"
)

MULTIMODAL_FILE = (
    DATA_DIR
    / "synchronized_multimodal_five_gnb_dataset.npz"
)

CHECKPOINT_FILE = (
    RESULTS_DIR
    / "virtual_lidar_range_checkpoint.npz"
)

METADATA_FILE = (
    RESULTS_DIR
    / "synchronized_multimodal_metadata.json"
)

SUMMARY_FILE = (
    RESULTS_DIR
    / "synchronized_multimodal_summary.csv"
)


if not DENSE_RF_FILE.exists():
    raise FileNotFoundError(
        f"Dense RF dataset not found:\n{DENSE_RF_FILE}"
    )

if "scene" not in globals():
    raise RuntimeError(
        "The variable 'scene' is not defined. "
        "Run the KFUPM scene-loading cell first."
    )

mi_scene = getattr(
    scene,
    "mi_scene",
    None,
)

if mi_scene is None:
    raise RuntimeError(
        "The Sionna scene does not expose scene.mi_scene."
    )


# ------------------------------------------------------------
# 2. Load the dense five-gNB RF dataset
# ------------------------------------------------------------

with np.load(
    DENSE_RF_FILE,
    allow_pickle=True,
) as dense_data:

    required_keys = [
        "positions_xyz_m",
        "trajectory_ids",
        "route_indices",
        "speeds_mps",
        "rf_power_db",
        "valid_path_counts",
        "minimum_delay_s",
        "completed",
        "gnb_names",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in dense_data.files
    ]

    if missing_keys:
        raise KeyError(
            f"Missing keys: {missing_keys}"
        )

    positions_raw = np.asarray(
        dense_data["positions_xyz_m"],
        dtype=np.float64,
    )

    trajectory_ids_raw = np.asarray(
        dense_data["trajectory_ids"],
        dtype=np.int32,
    )

    route_indices_raw = np.asarray(
        dense_data["route_indices"],
        dtype=np.int32,
    )

    speeds_raw = np.asarray(
        dense_data["speeds_mps"],
        dtype=np.float32,
    )

    rf_power_raw = np.asarray(
        dense_data["rf_power_db"],
        dtype=np.float32,
    )

    valid_paths_raw = np.asarray(
        dense_data["valid_path_counts"],
        dtype=np.float32,
    )

    minimum_delay_raw = np.asarray(
        dense_data["minimum_delay_s"],
        dtype=np.float32,
    )

    completed_raw = np.asarray(
        dense_data["completed"],
        dtype=bool,
    )

    def decode_name(value):
        if isinstance(
            value,
            (
                bytes,
                np.bytes_,
            ),
        ):
            return value.decode(
                "utf-8",
                errors="replace",
            )
        return str(value)

    gnb_names = [
        decode_name(name)
        for name in dense_data["gnb_names"]
    ]


number_of_observations = int(
    positions_raw.shape[0]
)

number_of_gnbs = int(
    rf_power_raw.shape[1]
)

if not np.all(completed_raw):
    raise RuntimeError(
        "The dense RF dataset is incomplete."
    )


# Sort observations by trajectory and route index
observation_order = np.lexsort(
    (
        route_indices_raw,
        trajectory_ids_raw,
    )
)


positions_xyz_m = positions_raw[
    observation_order
]

trajectory_ids = trajectory_ids_raw[
    observation_order
]

route_indices = route_indices_raw[
    observation_order
]

speeds_mps = speeds_raw[
    observation_order
]

rf_power_db = rf_power_raw[
    observation_order
]

valid_path_counts = valid_paths_raw[
    observation_order
]

minimum_delay_s = minimum_delay_raw[
    observation_order
]


print(
    "Dense RF observations:",
    number_of_observations,
)

print(
    "Number of gNBs:",
    number_of_gnbs,
)

print(
    "gNB names:",
    gnb_names,
)


# ------------------------------------------------------------
# 3. Mobility features
# ------------------------------------------------------------

TIME_STEP_S = 0.5

mobility_feature_names = [
    "speed_mps",
    "velocity_x_mps",
    "velocity_y_mps",
    "acceleration_x_mps2",
    "acceleration_y_mps2",
    "heading_sin",
    "heading_cos",
    "yaw_rate_radps",
    "delta_x_m",
    "delta_y_m",
]

mobility_features = np.zeros(
    (
        number_of_observations,
        len(mobility_feature_names),
    ),
    dtype=np.float32,
)

heading_rad = np.zeros(
    number_of_observations,
    dtype=np.float32,
)

time_s = np.zeros(
    number_of_observations,
    dtype=np.float32,
)


for trajectory_id in np.unique(
    trajectory_ids
):

    trajectory_indices = np.flatnonzero(
        trajectory_ids == trajectory_id
    )

    trajectory_indices = trajectory_indices[
        np.argsort(
            route_indices[
                trajectory_indices
            ]
        )
    ]

    trajectory_positions = positions_xyz_m[
        trajectory_indices
    ]

    trajectory_speeds = speeds_mps[
        trajectory_indices
    ]

    trajectory_velocity = np.gradient(
        trajectory_positions,
        TIME_STEP_S,
        axis=0,
    )

    trajectory_acceleration = np.gradient(
        trajectory_velocity,
        TIME_STEP_S,
        axis=0,
    )

    trajectory_heading = np.arctan2(
        trajectory_velocity[:, 1],
        trajectory_velocity[:, 0],
    )

    trajectory_heading = np.unwrap(
        trajectory_heading
    )

    trajectory_yaw_rate = np.gradient(
        trajectory_heading,
        TIME_STEP_S,
    )

    trajectory_delta = np.zeros(
        (
            len(trajectory_indices),
            2,
        ),
        dtype=np.float32,
    )

    if len(trajectory_indices) > 1:

        trajectory_delta[1:] = np.diff(
            trajectory_positions[:, :2],
            axis=0,
        )

    trajectory_mobility = np.column_stack(
        [
            trajectory_speeds,
            trajectory_velocity[:, 0],
            trajectory_velocity[:, 1],
            trajectory_acceleration[:, 0],
            trajectory_acceleration[:, 1],
            np.sin(trajectory_heading),
            np.cos(trajectory_heading),
            trajectory_yaw_rate,
            trajectory_delta[:, 0],
            trajectory_delta[:, 1],
        ]
    ).astype(np.float32)

    mobility_features[
        trajectory_indices
    ] = trajectory_mobility

    heading_rad[
        trajectory_indices
    ] = trajectory_heading.astype(
        np.float32
    )

    time_s[
        trajectory_indices
    ] = (
        route_indices[
            trajectory_indices
        ].astype(np.float32)
        * TIME_STEP_S
    )


# ------------------------------------------------------------
# 4. Virtual LiDAR configuration
# ------------------------------------------------------------

# 24 azimuth rays x 3 elevation rays = 72 features
AZIMUTH_DEG = np.arange(
    0.0,
    360.0,
    15.0,
    dtype=np.float32,
)

ELEVATION_DEG = np.array(
    [
        -10.0,
        0.0,
        10.0,
    ],
    dtype=np.float32,
)

RANGE_SENSOR_HEIGHT_M = 1.5
MAX_RANGE_M = 300.0
MIN_RANGE_M = 0.05

NUMBER_OF_AZIMUTH_RAYS = int(
    len(AZIMUTH_DEG)
)

NUMBER_OF_ELEVATION_RAYS = int(
    len(ELEVATION_DEG)
)

NUMBER_OF_RAYS = (
    NUMBER_OF_AZIMUTH_RAYS
    * NUMBER_OF_ELEVATION_RAYS
)


azimuth_rad = np.deg2rad(
    AZIMUTH_DEG
)

elevation_rad = np.deg2rad(
    ELEVATION_DEG
)

ray_directions_body = []

for elevation in elevation_rad:

    for azimuth in azimuth_rad:

        ray_directions_body.append(
            [
                np.cos(elevation)
                * np.cos(azimuth),

                np.cos(elevation)
                * np.sin(azimuth),

                np.sin(elevation),
            ]
        )

ray_directions_body = np.asarray(
    ray_directions_body,
    dtype=np.float32,
)

ray_azimuth_deg = np.tile(
    AZIMUTH_DEG,
    NUMBER_OF_ELEVATION_RAYS,
)

ray_elevation_deg = np.repeat(
    ELEVATION_DEG,
    NUMBER_OF_AZIMUTH_RAYS,
)


# ------------------------------------------------------------
# 5. Mitsuba virtual LiDAR function
# ------------------------------------------------------------

def cast_virtual_lidar(
    positions_batch,
    headings_batch,
):
    """
    Cast batched rays against the loaded KFUPM mesh.

    Azimuth zero is aligned with the UE heading.
    Missed rays are assigned MAX_RANGE_M.
    """

    positions_batch = np.asarray(
        positions_batch,
        dtype=np.float32,
    )

    headings_batch = np.asarray(
        headings_batch,
        dtype=np.float32,
    )

    batch_size = int(
        positions_batch.shape[0]
    )

    origins = np.repeat(
        positions_batch,
        NUMBER_OF_RAYS,
        axis=0,
    )

    origins[:, 2] += (
        RANGE_SENSOR_HEIGHT_M
    )

    local_directions = np.tile(
        ray_directions_body,
        (
            batch_size,
            1,
        ),
    )

    repeated_headings = np.repeat(
        headings_batch,
        NUMBER_OF_RAYS,
    )

    cos_heading = np.cos(
        repeated_headings
    )

    sin_heading = np.sin(
        repeated_headings
    )

    directions = np.zeros_like(
        local_directions,
        dtype=np.float32,
    )

    directions[:, 0] = (
        local_directions[:, 0]
        * cos_heading
        - local_directions[:, 1]
        * sin_heading
    )

    directions[:, 1] = (
        local_directions[:, 0]
        * sin_heading
        + local_directions[:, 1]
        * cos_heading
    )

    directions[:, 2] = (
        local_directions[:, 2]
    )

    def to_mitsuba_float(values):
        return mi.Float(
            np.ascontiguousarray(
                values,
                dtype=np.float32,
            )
        )

    rays = mi.Ray3f(
        o=mi.Point3f(
            to_mitsuba_float(
                origins[:, 0]
            ),
            to_mitsuba_float(
                origins[:, 1]
            ),
            to_mitsuba_float(
                origins[:, 2]
            ),
        ),
        d=mi.Vector3f(
            to_mitsuba_float(
                directions[:, 0]
            ),
            to_mitsuba_float(
                directions[:, 1]
            ),
            to_mitsuba_float(
                directions[:, 2]
            ),
        ),
    )

    # Mitsuba 3 Ray3f has maxt but no mint.
    # Very short intersections are filtered below.
    rays.maxt = mi.Float(
        MAX_RANGE_M
    )

    surface_interaction = (
        mi_scene.ray_intersect(rays)
    )

    distances = surface_interaction.t

    dr.eval(distances)

    distances = np.asarray(
        distances,
        dtype=np.float32,
    ).reshape(
        batch_size,
        NUMBER_OF_RAYS,
    )

    invalid = (
        ~np.isfinite(distances)
        | (
            distances
            <= MIN_RANGE_M
        )
        | (
            distances
            > MAX_RANGE_M
        )
    )

    distances[invalid] = (
        MAX_RANGE_M
    )

    return distances


# ------------------------------------------------------------
# 6. Initialize or resume LiDAR checkpoint
# ------------------------------------------------------------

range_features_m = np.full(
    (
        number_of_observations,
        NUMBER_OF_RAYS,
    ),
    MAX_RANGE_M,
    dtype=np.float32,
)

range_completed = np.zeros(
    number_of_observations,
    dtype=bool,
)


if CHECKPOINT_FILE.exists():

    try:

        with np.load(
            CHECKPOINT_FILE,
            allow_pickle=True,
        ) as checkpoint:

            checkpoint_positions = np.asarray(
                checkpoint[
                    "positions_xyz_m"
                ]
            )

            checkpoint_trajectories = np.asarray(
                checkpoint[
                    "trajectory_ids"
                ]
            )

            checkpoint_routes = np.asarray(
                checkpoint[
                    "route_indices"
                ]
            )

            checkpoint_ranges = np.asarray(
                checkpoint[
                    "range_features_m"
                ]
            )

            checkpoint_completed = np.asarray(
                checkpoint[
                    "range_completed"
                ],
                dtype=bool,
            )

        checkpoint_is_compatible = (
            checkpoint_positions.shape
            == positions_xyz_m.shape
            and checkpoint_ranges.shape
            == range_features_m.shape
            and np.allclose(
                checkpoint_positions,
                positions_xyz_m,
            )
            and np.array_equal(
                checkpoint_trajectories,
                trajectory_ids,
            )
            and np.array_equal(
                checkpoint_routes,
                route_indices,
            )
        )

        if checkpoint_is_compatible:

            range_features_m[:] = (
                checkpoint_ranges
            )

            range_completed[:] = (
                checkpoint_completed
            )

            print(
                "Compatible LiDAR checkpoint loaded."
            )

        else:

            print(
                "Existing checkpoint is incompatible."
            )

    except Exception as checkpoint_error:

        print(
            "Checkpoint could not be loaded:"
        )

        print(checkpoint_error)

        print(
            "Starting a new LiDAR extraction."
        )


def save_lidar_checkpoint():

    temporary_file = (
        RESULTS_DIR
        / "virtual_lidar_range_checkpoint.tmp.npz"
    )

    np.savez_compressed(
        temporary_file,

        positions_xyz_m=positions_xyz_m,

        trajectory_ids=trajectory_ids,

        route_indices=route_indices,

        range_features_m=range_features_m,

        range_completed=range_completed,

        ray_azimuth_deg=ray_azimuth_deg,

        ray_elevation_deg=ray_elevation_deg,

        max_range_m=np.array(
            MAX_RANGE_M,
            dtype=np.float32,
        ),
    )

    temporary_file.replace(
        CHECKPOINT_FILE
    )


# ------------------------------------------------------------
# 7. GPU LiDAR extraction
# ------------------------------------------------------------

RANGE_BATCH_OBSERVATIONS = 64

remaining_before = int(
    np.sum(~range_completed)
)

print(
    f"Observations requiring LiDAR: "
    f"{remaining_before}/"
    f"{number_of_observations}"
)

range_start_time = time.perf_counter()

for start in range(
    0,
    number_of_observations,
    RANGE_BATCH_OBSERVATIONS,
):

    stop = min(
        start
        + RANGE_BATCH_OBSERVATIONS,
        number_of_observations,
    )

    batch_indices = np.arange(
        start,
        stop,
    )

    batch_indices = batch_indices[
        ~range_completed[
            batch_indices
        ]
    ]

    if len(batch_indices) == 0:
        continue

    batch_ranges = cast_virtual_lidar(
        positions_xyz_m[
            batch_indices
        ],
        heading_rad[
            batch_indices
        ],
    )

    range_features_m[
        batch_indices
    ] = batch_ranges

    range_completed[
        batch_indices
    ] = True

    save_lidar_checkpoint()

    completed_count = int(
        np.sum(range_completed)
    )

    elapsed = (
        time.perf_counter()
        - range_start_time
    )

    print(
        f"LiDAR progress: "
        f"{completed_count}/"
        f"{number_of_observations} | "
        f"elapsed: {elapsed:.1f} s"
    )


if not np.all(range_completed):
    raise RuntimeError(
        "LiDAR extraction did not complete."
    )


# ------------------------------------------------------------
# 8. LiDAR summary features
# ------------------------------------------------------------

lidar_hit_mask = (
    range_features_m
    < (
        MAX_RANGE_M
        - 1e-3
    )
)

hit_count = np.sum(
    lidar_hit_mask,
    axis=1,
).astype(np.float32)

hit_fraction = (
    hit_count
    / float(NUMBER_OF_RAYS)
)

safe_ranges = np.where(
    lidar_hit_mask,
    range_features_m,
    MAX_RANGE_M,
)

nearest_range_m = np.min(
    safe_ranges,
    axis=1,
)

farthest_hit_range_m = np.max(
    safe_ranges,
    axis=1,
)

mean_range_m = (
    np.sum(
        np.where(
            lidar_hit_mask,
            range_features_m,
            0.0,
        ),
        axis=1,
    )
    / np.maximum(
        hit_count,
        1.0,
    )
)

p10_range_m = np.full(
    number_of_observations,
    MAX_RANGE_M,
    dtype=np.float32,
)

p50_range_m = np.full(
    number_of_observations,
    MAX_RANGE_M,
    dtype=np.float32,
)

p90_range_m = np.full(
    number_of_observations,
    MAX_RANGE_M,
    dtype=np.float32,
)


for observation_index in range(
    number_of_observations
):

    valid_ranges = range_features_m[
        observation_index
    ][
        lidar_hit_mask[
            observation_index
        ]
    ]

    if len(valid_ranges) > 0:

        p10_range_m[
            observation_index
        ] = np.percentile(
            valid_ranges,
            10,
        )

        p50_range_m[
            observation_index
        ] = np.percentile(
            valid_ranges,
            50,
        )

        p90_range_m[
            observation_index
        ] = np.percentile(
            valid_ranges,
            90,
        )


lidar_summary = np.column_stack(
    [
        nearest_range_m,
        mean_range_m,
        farthest_hit_range_m,
        p10_range_m,
        p50_range_m,
        p90_range_m,
        hit_fraction,
    ]
).astype(np.float32)

lidar_summary_names = [
    "nearest_range_m",
    "mean_range_m",
    "farthest_hit_range_m",
    "p10_range_m",
    "p50_range_m",
    "p90_range_m",
    "hit_fraction",
]


# ------------------------------------------------------------
# 9. RF feature block
# ------------------------------------------------------------

clean_rf_power_db = np.where(
    np.isfinite(rf_power_db),
    rf_power_db,
    -300.0,
).astype(np.float32)

clean_valid_paths = np.where(
    np.isfinite(valid_path_counts),
    valid_path_counts,
    0.0,
).astype(np.float32)

clean_delay_us = np.where(
    np.isfinite(minimum_delay_s)
    & (
        minimum_delay_s
        >= 0.0
    ),
    minimum_delay_s * 1e6,
    0.0,
).astype(np.float32)


rf_feature_names = (
    [
        f"{name}_rf_power_db"
        for name in gnb_names
    ]
    + [
        f"{name}_valid_paths"
        for name in gnb_names
    ]
    + [
        f"{name}_minimum_delay_us"
        for name in gnb_names
    ]
)

rf_features = np.column_stack(
    [
        clean_rf_power_db,
        clean_valid_paths,
        clean_delay_us,
    ]
).astype(np.float32)


# Best currently usable gNB
usable_links = (
    clean_valid_paths > 0
)

masked_power = np.where(
    usable_links,
    clean_rf_power_db,
    -np.inf,
)

best_gnb_index = np.argmax(
    masked_power,
    axis=1,
).astype(np.int16)

no_usable_link = (
    np.sum(
        usable_links,
        axis=1,
    )
    == 0
)

best_gnb_index[
    no_usable_link
] = -1


# ------------------------------------------------------------
# 10. Combined multimodal feature block
# ------------------------------------------------------------

lidar_feature_names = [
    f"lidar_range_{index:02d}_m"
    for index in range(NUMBER_OF_RAYS)
]

combined_features = np.column_stack(
    [
        rf_features,
        mobility_features,
        range_features_m,
    ]
).astype(np.float32)

combined_feature_names = (
    rf_feature_names
    + mobility_feature_names
    + lidar_feature_names
)


# ------------------------------------------------------------
# 11. Save synchronized multimodal dataset
# ------------------------------------------------------------

np.savez_compressed(
    MULTIMODAL_FILE,

    positions_xyz_m=positions_xyz_m.astype(
        np.float32
    ),

    trajectory_ids=trajectory_ids,

    route_indices=route_indices,

    time_s=time_s,

    speeds_mps=speeds_mps,

    heading_rad=heading_rad,

    mobility_features=mobility_features,

    rf_power_db=clean_rf_power_db,

    valid_path_counts=clean_valid_paths,

    minimum_delay_s=minimum_delay_s,

    minimum_delay_us=clean_delay_us,

    rf_features=rf_features,

    lidar_ranges_m=range_features_m,

    lidar_hit_mask=lidar_hit_mask,

    lidar_summary=lidar_summary,

    combined_features=combined_features,

    best_gnb_index=best_gnb_index,

    completed=np.ones(
        number_of_observations,
        dtype=bool,
    ),

    gnb_names=np.asarray(
        gnb_names,
        dtype="U",
    ),

    ray_azimuth_deg=ray_azimuth_deg,

    ray_elevation_deg=ray_elevation_deg,

    max_lidar_range_m=np.array(
        MAX_RANGE_M,
        dtype=np.float32,
    ),

    lidar_sensor_height_m=np.array(
        RANGE_SENSOR_HEIGHT_M,
        dtype=np.float32,
    ),
)


# ------------------------------------------------------------
# 12. Save metadata
# ------------------------------------------------------------

dense_sha256 = hashlib.sha256(
    DENSE_RF_FILE.read_bytes()
).hexdigest()

metadata = {
    "dataset_name": (
        "synchronized_multimodal_five_gnb"
    ),

    "source_dense_rf_dataset": str(
        DENSE_RF_FILE
    ),

    "source_dense_rf_sha256": (
        dense_sha256
    ),

    "number_of_observations": int(
        number_of_observations
    ),

    "number_of_gnbs": int(
        number_of_gnbs
    ),

    "gnb_names": gnb_names,

    "trajectory_count": int(
        len(
            np.unique(
                trajectory_ids
            )
        )
    ),

    "points_per_trajectory": int(
        len(route_indices)
        // len(
            np.unique(
                trajectory_ids
            )
        )
    ),

    "time_step_s": TIME_STEP_S,

    "rf_feature_dimension": int(
        rf_features.shape[1]
    ),

    "mobility_feature_dimension": int(
        mobility_features.shape[1]
    ),

    "lidar_feature_dimension": int(
        range_features_m.shape[1]
    ),

    "combined_feature_dimension": int(
        combined_features.shape[1]
    ),

    "rf_feature_names": rf_feature_names,

    "mobility_feature_names": (
        mobility_feature_names
    ),

    "lidar_summary_names": (
        lidar_summary_names
    ),

    "combined_feature_names": (
        combined_feature_names
    ),

    "lidar_method": (
        "Virtual LiDAR generated by "
        "batched Mitsuba ray intersections "
        "against the loaded KFUPM scene mesh"
    ),

    "lidar_is_scene_derived": True,

    "lidar_coordinate_frame": (
        "UE body frame rotated into the "
        "global scene frame using route heading"
    ),

    "ray_count": int(
        NUMBER_OF_RAYS
    ),

    "azimuth_count": int(
        NUMBER_OF_AZIMUTH_RAYS
    ),

    "elevation_count": int(
        NUMBER_OF_ELEVATION_RAYS
    ),

    "azimuth_spacing_deg": 15.0,

    "elevation_angles_deg": (
        ELEVATION_DEG.tolist()
    ),

    "sensor_height_m": (
        RANGE_SENSOR_HEIGHT_M
    ),

    "maximum_range_m": (
        MAX_RANGE_M
    ),

    "checkpoint_file": str(
        CHECKPOINT_FILE
    ),

    "multimodal_dataset_file": str(
        MULTIMODAL_FILE
    ),
}


with open(
    METADATA_FILE,
    "w",
    encoding="utf-8",
) as metadata_file:

    json.dump(
        metadata,
        metadata_file,
        indent=2,
    )


# ------------------------------------------------------------
# 13. Save row-level summary CSV
# ------------------------------------------------------------

summary_columns = [
    "observation_index",
    "trajectory_id",
    "route_index",
    "time_s",
    "x_m",
    "y_m",
    "z_m",
    "speed_mps",
    "best_gnb_index",
    "best_gnb_name",
]

for name in gnb_names:

    summary_columns.extend(
        [
            f"{name}_rf_power_db",
            f"{name}_valid_paths",
            f"{name}_minimum_delay_us",
        ]
    )

summary_columns.extend(
    lidar_summary_names
)


with open(
    SUMMARY_FILE,
    "w",
    newline="",
    encoding="utf-8",
) as summary_file:

    writer = csv.writer(
        summary_file
    )

    writer.writerow(
        summary_columns
    )

    for index in range(
        number_of_observations
    ):

        best_index = int(
            best_gnb_index[index]
        )

        best_name = (
            gnb_names[best_index]
            if best_index >= 0
            else "none"
        )

        row = [
            index,
            int(
                trajectory_ids[index]
            ),
            int(
                route_indices[index]
            ),
            float(
                time_s[index]
            ),
            float(
                positions_xyz_m[
                    index,
                    0,
                ]
            ),
            float(
                positions_xyz_m[
                    index,
                    1,
                ]
            ),
            float(
                positions_xyz_m[
                    index,
                    2,
                ]
            ),
            float(
                speeds_mps[index]
            ),
            best_index,
            best_name,
        ]

        for gnb_index in range(
            number_of_gnbs
        ):

            row.extend(
                [
                    float(
                        clean_rf_power_db[
                            index,
                            gnb_index,
                        ]
                    ),
                    float(
                        clean_valid_paths[
                            index,
                            gnb_index,
                        ]
                    ),
                    float(
                        clean_delay_us[
                            index,
                            gnb_index,
                        ]
                    ),
                ]
            )

        row.extend(
            [
                float(
                    lidar_summary[
                        index,
                        summary_index,
                    ]
                )
                for summary_index in range(
                    lidar_summary.shape[1]
                )
            ]
        )

        writer.writerow(row)


# ------------------------------------------------------------
# 14. Validation
# ------------------------------------------------------------

assert MULTIMODAL_FILE.exists()
assert METADATA_FILE.exists()
assert SUMMARY_FILE.exists()

assert range_features_m.shape == (
    number_of_observations,
    72,
)

assert mobility_features.shape == (
    number_of_observations,
    10,
)

assert rf_features.shape == (
    number_of_observations,
    number_of_gnbs * 3,
)

assert combined_features.shape == (
    number_of_observations,
    (
        number_of_gnbs * 3
        + 10
        + 72
    ),
)

assert np.all(
    np.isfinite(
        range_features_m
    )
)

assert np.all(
    np.isfinite(
        mobility_features
    )
)

assert np.all(
    np.isfinite(
        combined_features
    )
)


print()
print(
    "Synchronized multimodal dataset created."
)

print(
    "RF feature shape:",
    rf_features.shape,
)

print(
    "Mobility feature shape:",
    mobility_features.shape,
)

print(
    "LiDAR feature shape:",
    range_features_m.shape,
)

print(
    "Combined feature shape:",
    combined_features.shape,
)

print(
    "Mean LiDAR hit fraction:",
    round(
        float(
            np.mean(
                hit_fraction
            )
        ),
        4,
    ),
)

print(
    "Best-gNB distribution:"
)

for gnb_index, gnb_name in enumerate(
    gnb_names
):

    print(
        f"  {gnb_name}:",
        int(
            np.sum(
                best_gnb_index
                == gnb_index
            )
        ),
    )

print()
print(
    "Saved dataset:"
)
print(MULTIMODAL_FILE)

print()
print(
    "Saved metadata:"
)
print(METADATA_FILE)

print()
print(
    "Saved summary:"
)
print(SUMMARY_FILE)

print()
print(
    "Checkpoint retained:"
)
print(CHECKPOINT_FILE)

In [ ]:
# ============================================================
# Verify synchronized multimodal dataset
# ============================================================

from pathlib import Path
import numpy as np
import json


PROJECT_ROOT = Path.home() / "wcnc_project"

MULTIMODAL_FILE = (
    PROJECT_ROOT
    / "improved_experiment"
    / "data"
    / "synchronized_multimodal_five_gnb_dataset.npz"
)

METADATA_FILE = (
    PROJECT_ROOT
    / "improved_experiment"
    / "results"
    / "synchronized_multimodal_metadata.json"
)


if not MULTIMODAL_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{MULTIMODAL_FILE}"
    )


with np.load(
    MULTIMODAL_FILE,
    allow_pickle=True,
) as loaded_data:

    multimodal_data = {
        key: loaded_data[key]
        for key in loaded_data.files
    }


with open(
    METADATA_FILE,
    "r",
    encoding="utf-8",
) as metadata_file:

    multimodal_metadata = json.load(
        metadata_file
    )


positions_xyz_m = (
    multimodal_data[
        "positions_xyz_m"
    ]
)

trajectory_ids = (
    multimodal_data[
        "trajectory_ids"
    ]
)

route_indices = (
    multimodal_data[
        "route_indices"
    ]
)

rf_features = (
    multimodal_data[
        "rf_features"
    ]
)

mobility_features = (
    multimodal_data[
        "mobility_features"
    ]
)

lidar_ranges_m = (
    multimodal_data[
        "lidar_ranges_m"
    ]
)

lidar_hit_mask = (
    multimodal_data[
        "lidar_hit_mask"
    ]
)

combined_features = (
    multimodal_data[
        "combined_features"
    ]
)

best_gnb_index = (
    multimodal_data[
        "best_gnb_index"
    ]
)

gnb_names = [
    str(name)
    for name in multimodal_data[
        "gnb_names"
    ]
]


# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

number_of_observations = (
    positions_xyz_m.shape[0]
)

number_of_gnbs = len(
    gnb_names
)

assert rf_features.shape == (
    number_of_observations,
    number_of_gnbs * 3,
)

assert mobility_features.shape == (
    number_of_observations,
    10,
)

assert lidar_ranges_m.shape == (
    number_of_observations,
    72,
)

assert combined_features.shape == (
    number_of_observations,
    number_of_gnbs * 3 + 10 + 72,
)

assert np.all(
    np.isfinite(
        combined_features
    )
)

assert np.all(
    multimodal_data[
        "completed"
    ]
)

for trajectory_id in np.unique(
    trajectory_ids
):

    trajectory_mask = (
        trajectory_ids
        == trajectory_id
    )

    trajectory_routes = (
        route_indices[
            trajectory_mask
        ]
    )

    assert np.all(
        np.diff(
            trajectory_routes
        )
        >= 0
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("MULTIMODAL DATASET VALIDATION PASSED")
print("-------------------------------------")

print(
    "Dataset:",
    MULTIMODAL_FILE,
)

print(
    "Observations:",
    number_of_observations,
)

print(
    "Trajectories:",
    len(
        np.unique(
            trajectory_ids
        )
    ),
)

print(
    "gNBs:",
    gnb_names,
)

print(
    "RF feature shape:",
    rf_features.shape,
)

print(
    "Mobility feature shape:",
    mobility_features.shape,
)

print(
    "LiDAR feature shape:",
    lidar_ranges_m.shape,
)

print(
    "Combined feature shape:",
    combined_features.shape,
)

print(
    "Mean LiDAR hit fraction:",
    round(
        float(
            np.mean(
                lidar_hit_mask
            )
        ),
        4,
    ),
)

print(
    "Minimum LiDAR range:",
    round(
        float(
            np.min(
                lidar_ranges_m
            )
        ),
        3,
    ),
    "m",
)

print(
    "Median LiDAR range:",
    round(
        float(
            np.median(
                lidar_ranges_m
            )
        ),
        3,
    ),
    "m",
)

print(
    "Maximum LiDAR range:",
    round(
        float(
            np.max(
                lidar_ranges_m
            )
        ),
        3,
    ),
    "m",
)

print("\nBest-gNB distribution:")

for gnb_index, gnb_name in enumerate(
    gnb_names
):

    count = int(
        np.sum(
            best_gnb_index
            == gnb_index
        )
    )

    print(
        f"{gnb_name}: {count}"
    )

print("\nMetadata feature dimensions:")

print(
    "RF:",
    multimodal_metadata[
        "rf_feature_dimension"
    ],
)

print(
    "Mobility:",
    multimodal_metadata[
        "mobility_feature_dimension"
    ],
)

print(
    "LiDAR:",
    multimodal_metadata[
        "lidar_feature_dimension"
    ],
)

print(
    "Combined:",
    multimodal_metadata[
        "combined_feature_dimension"
    ],
)

In [ ]:
# ============================================================
# Diagnose virtual LiDAR ranges
# ============================================================

from pathlib import Path
import numpy as np


PROJECT_ROOT = Path.home() / "wcnc_project"

MULTIMODAL_FILE = (
    PROJECT_ROOT
    / "improved_experiment"
    / "data"
    / "synchronized_multimodal_five_gnb_dataset.npz"
)


with np.load(
    MULTIMODAL_FILE,
    allow_pickle=True,
) as data:

    lidar_ranges_m = data[
        "lidar_ranges_m"
    ]

    trajectory_ids = data[
        "trajectory_ids"
    ]

    route_indices = data[
        "route_indices"
    ]

    positions_xyz_m = data[
        "positions_xyz_m"
    ]

    ray_azimuth_deg = data[
        "ray_azimuth_deg"
    ]

    ray_elevation_deg = data[
        "ray_elevation_deg"
    ]

    gnb_names = [
        str(name)
        for name in data[
            "gnb_names"
        ]
    ]


number_of_observations = (
    lidar_ranges_m.shape[0]
)

number_of_rays = (
    lidar_ranges_m.shape[1]
)

print("LiDAR range diagnostics")
print("-----------------------")

print(
    "Total range values:",
    lidar_ranges_m.size,
)

print(
    "Number of observations:",
    number_of_observations,
)

print(
    "Number of rays per observation:",
    number_of_rays,
)

print()


# Range percentiles
percentiles = [
    0,
    1,
    5,
    25,
    50,
    75,
    95,
    99,
    100,
]

print("Range percentiles:")

for percentile in percentiles:

    value = np.percentile(
        lidar_ranges_m,
        percentile,
    )

    print(
        f"{percentile:>3d}th percentile:"
        f" {value:8.3f} m"
    )


print()


# Near-zero range counts
print("Near-zero range values:")

for threshold in [
    0.1,
    0.2,
    0.5,
    1.0,
    2.0,
]:

    count = int(
        np.sum(
            lidar_ranges_m
            < threshold
        )
    )

    percentage = (
        100.0
        * count
        / lidar_ranges_m.size
    )

    print(
        f"Below {threshold:>4.1f} m:"
        f" {count:6d}"
        f" ({percentage:.4f}%)"
    )


print()


# Maximum-range values
maximum_range_mask = (
    lidar_ranges_m
    >= 299.999
)

maximum_range_count = int(
    np.sum(
        maximum_range_mask
    )
)

maximum_range_percentage = (
    100.0
    * maximum_range_count
    / lidar_ranges_m.size
)

print(
    "Values at maximum range:",
    maximum_range_count,
)

print(
    "Maximum-range percentage:",
    round(
        maximum_range_percentage,
        4,
    ),
    "%",
)


# Identify the smallest individual ranges
flat_order = np.argsort(
    lidar_ranges_m,
    axis=None,
)

smallest_flat_indices = (
    flat_order[:10]
)

smallest_indices = np.unravel_index(
    smallest_flat_indices,
    lidar_ranges_m.shape,
)

print()
print("Ten smallest range values:")
print("--------------------------")

for observation_index, ray_index in zip(
    smallest_indices[0],
    smallest_indices[1],
):

    print(
        f"Observation {observation_index:4d} | "
        f"trajectory "
        f"{int(trajectory_ids[observation_index])} | "
        f"route "
        f"{int(route_indices[observation_index]):3d} | "
        f"ray {int(ray_index):2d} | "
        f"azimuth "
        f"{ray_azimuth_deg[ray_index]:6.1f} deg | "
        f"elevation "
        f"{ray_elevation_deg[ray_index]:6.1f} deg | "
        f"range "
        f"{lidar_ranges_m[observation_index, ray_index]:8.4f} m | "
        f"position "
        f"{np.round(positions_xyz_m[observation_index], 3)}"
    )

In [ ]:
# ============================================================
# Prepare leakage-free forecasting sequences
# RF + mobility + LiDAR
# ============================================================

from pathlib import Path
import json
import numpy as np


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.home() / "wcnc_project"

DATA_DIR = (
    PROJECT_ROOT
    / "improved_experiment"
    / "data"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "improved_experiment"
    / "results"
)

MULTIMODAL_FILE = (
    DATA_DIR
    / "synchronized_multimodal_five_gnb_dataset.npz"
)

SEQUENCE_FILE = (
    DATA_DIR
    / "forecast_sequences_five_gnb.npz"
)

SEQUENCE_METADATA_FILE = (
    RESULTS_DIR
    / "forecast_sequences_metadata.json"
)


if not MULTIMODAL_FILE.exists():
    raise FileNotFoundError(
        f"Multimodal dataset not found:\n{MULTIMODAL_FILE}"
    )


# ------------------------------------------------------------
# 2. Load multimodal dataset
# ------------------------------------------------------------

with np.load(
    MULTIMODAL_FILE,
    allow_pickle=True,
) as data:

    positions_xyz_m = np.asarray(
        data["positions_xyz_m"],
        dtype=np.float32,
    )

    trajectory_ids = np.asarray(
        data["trajectory_ids"],
        dtype=np.int32,
    )

    route_indices = np.asarray(
        data["route_indices"],
        dtype=np.int32,
    )

    rf_power_db = np.asarray(
        data["rf_power_db"],
        dtype=np.float32,
    )

    valid_path_counts = np.asarray(
        data["valid_path_counts"],
        dtype=np.float32,
    )

    minimum_delay_s = np.asarray(
        data["minimum_delay_s"],
        dtype=np.float32,
    )

    mobility_features = np.asarray(
        data["mobility_features"],
        dtype=np.float32,
    )

    lidar_ranges_m = np.asarray(
        data["lidar_ranges_m"],
        dtype=np.float32,
    )

    lidar_hit_mask = np.asarray(
        data["lidar_hit_mask"],
        dtype=bool,
    )

    best_gnb_index = np.asarray(
        data["best_gnb_index"],
        dtype=np.int16,
    )

    def decode_name(value):
        if isinstance(
            value,
            (
                bytes,
                np.bytes_,
            ),
        ):
            return value.decode(
                "utf-8",
                errors="replace",
            )

        return str(value)

    gnb_names = [
        decode_name(name)
        for name in data["gnb_names"]
    ]


number_of_observations = int(
    positions_xyz_m.shape[0]
)

number_of_gnbs = int(
    rf_power_db.shape[1]
)

print(
    "Loaded observations:",
    number_of_observations,
)

print(
    "Loaded gNBs:",
    gnb_names,
)


# ------------------------------------------------------------
# 3. Forecasting configuration
# ------------------------------------------------------------

TIME_STEP_S = 0.5

HISTORY_STEPS = 12
FORECAST_HORIZON = 6

HISTORY_DURATION_S = (
    HISTORY_STEPS
    * TIME_STEP_S
)

FORECAST_DURATION_S = (
    FORECAST_HORIZON
    * TIME_STEP_S
)

POWER_FLOOR_DB = -220.0
POWER_CEILING_DB = 0.0

LIDAR_MIN_MODEL_RANGE_M = 0.5
LIDAR_MAX_MODEL_RANGE_M = 300.0


# ------------------------------------------------------------
# 4. Leakage-free trajectory split
# ------------------------------------------------------------

unique_trajectory_ids = np.sort(
    np.unique(
        trajectory_ids
    )
)

if len(unique_trajectory_ids) < 3:
    raise RuntimeError(
        "At least three trajectories are required."
    )


# Four trajectories for training,
# one trajectory for validation,
# one trajectory for testing.
train_trajectory_ids = (
    unique_trajectory_ids[:-2]
)

validation_trajectory_ids = (
    unique_trajectory_ids[-2:-1]
)

test_trajectory_ids = (
    unique_trajectory_ids[-1:]
)


print()
print("Trajectory split")
print("----------------")

print(
    "Training trajectories:",
    train_trajectory_ids.tolist(),
)

print(
    "Validation trajectories:",
    validation_trajectory_ids.tolist(),
)

print(
    "Testing trajectories:",
    test_trajectory_ids.tolist(),
)

print()


for trajectory_id in unique_trajectory_ids:

    trajectory_mask = (
        trajectory_ids == trajectory_id
    )

    number_of_points = int(
        np.sum(trajectory_mask)
    )

    median_speed_mps = float(
        np.median(
            mobility_features[
                trajectory_mask,
                0,
            ]
        )
    )

    print(
        f"Trajectory {int(trajectory_id)} | "
        f"points: {number_of_points} | "
        f"median speed: "
        f"{median_speed_mps:.2f} m/s"
    )


# ------------------------------------------------------------
# 5. Construct model feature block
# ------------------------------------------------------------

# RF received-power features
model_rf_power_db = np.clip(
    rf_power_db,
    POWER_FLOOR_DB,
    POWER_CEILING_DB,
).astype(np.float32)


# Log-transformed valid-path features
model_valid_paths = np.log1p(
    np.maximum(
        valid_path_counts,
        0.0,
    )
).astype(np.float32)


# Convert delay to microseconds and apply log transform
model_delay_us = np.where(
    np.isfinite(minimum_delay_s)
    & (
        minimum_delay_s >= 0.0
    ),
    minimum_delay_s * 1e6,
    0.0,
)

model_delay_us = np.log1p(
    np.maximum(
        model_delay_us,
        0.0,
    )
).astype(np.float32)


# Mobility features remain physically interpretable
model_mobility = (
    mobility_features.copy()
)


# Clip only for model preprocessing.
# The original raw LiDAR dataset remains unchanged.
model_lidar_ranges_m = np.clip(
    lidar_ranges_m,
    LIDAR_MIN_MODEL_RANGE_M,
    LIDAR_MAX_MODEL_RANGE_M,
).astype(np.float32)


model_features = np.concatenate(
    [
        model_rf_power_db,
        model_valid_paths,
        model_delay_us,
        model_mobility,
        model_lidar_ranges_m,
    ],
    axis=1,
).astype(np.float32)


feature_names = (
    [
        f"{name}_power_db"
        for name in gnb_names
    ]
    + [
        f"{name}_log_valid_paths"
        for name in gnb_names
    ]
    + [
        f"{name}_log_delay_us"
        for name in gnb_names
    ]
    + [
        "speed_mps",
        "velocity_x_mps",
        "velocity_y_mps",
        "acceleration_x_mps2",
        "acceleration_y_mps2",
        "heading_sin",
        "heading_cos",
        "yaw_rate_radps",
        "delta_x_m",
        "delta_y_m",
    ]
    + [
        f"lidar_range_{index:02d}_m"
        for index in range(
            lidar_ranges_m.shape[1]
        )
    ]
)


print()
print(
    "Model feature shape:",
    model_features.shape,
)

print(
    "Model feature dimension:",
    model_features.shape[1],
)


# ------------------------------------------------------------
# 6. Construct forecasting target
# ------------------------------------------------------------

target_power_db = np.clip(
    rf_power_db,
    POWER_FLOOR_DB,
    POWER_CEILING_DB,
).astype(np.float32)

target_valid_paths = (
    valid_path_counts > 0
).astype(np.float32)


# ------------------------------------------------------------
# 7. Normalize using training trajectories only
# ------------------------------------------------------------

training_rows = np.isin(
    trajectory_ids,
    train_trajectory_ids,
)

feature_mean = np.mean(
    model_features[
        training_rows
    ],
    axis=0,
).astype(np.float32)

feature_std = np.std(
    model_features[
        training_rows
    ],
    axis=0,
).astype(np.float32)

feature_std[
    feature_std < 1e-6
] = 1.0

target_mean = np.mean(
    target_power_db[
        training_rows
    ],
    axis=0,
).astype(np.float32)

target_std = np.std(
    target_power_db[
        training_rows
    ],
    axis=0,
).astype(np.float32)

target_std[
    target_std < 1e-6
] = 1.0

normalized_features = (
    (
        model_features
        - feature_mean
    )
    / feature_std
).astype(np.float32)

normalized_target_power = (
    (
        target_power_db
        - target_mean
    )
    / target_std
).astype(np.float32)


# ------------------------------------------------------------
# 8. Build temporal forecasting windows
# ------------------------------------------------------------

def build_forecasting_windows(
    selected_trajectory_ids,
):

    input_windows = []
    target_windows = []
    target_valid_windows = []
    best_gnb_windows = []
    start_route_indices = []
    window_trajectory_ids = []
    future_positions = []

    for trajectory_id in (
        selected_trajectory_ids
    ):

        trajectory_indices = np.flatnonzero(
            trajectory_ids == trajectory_id
        )

        trajectory_indices = (
            trajectory_indices[
                np.argsort(
                    route_indices[
                        trajectory_indices
                    ]
                )
            ]
        )

        number_of_points = len(
            trajectory_indices
        )

        maximum_start = (
            number_of_points
            - HISTORY_STEPS
            - FORECAST_HORIZON
            + 1
        )

        for start_index in range(
            maximum_start
        ):

            input_indices = (
                trajectory_indices[
                    start_index:
                    start_index
                    + HISTORY_STEPS
                ]
            )

            target_indices = (
                trajectory_indices[
                    start_index
                    + HISTORY_STEPS:
                    start_index
                    + HISTORY_STEPS
                    + FORECAST_HORIZON
                ]
            )

            input_windows.append(
                normalized_features[
                    input_indices
                ]
            )

            target_windows.append(
                normalized_target_power[
                    target_indices
                ]
            )

            target_valid_windows.append(
                target_valid_paths[
                    target_indices
                ]
            )

            best_gnb_windows.append(
                best_gnb_index[
                    target_indices
                ]
            )

            start_route_indices.append(
                route_indices[
                    input_indices[0]
                ]
            )

            window_trajectory_ids.append(
                trajectory_id
            )

            future_positions.append(
                positions_xyz_m[
                    target_indices
                ]
            )

    if len(input_windows) == 0:
        raise RuntimeError(
            "No temporal windows were created."
        )

    return {
        "X": np.stack(
            input_windows
        ).astype(np.float32),

        "Y": np.stack(
            target_windows
        ).astype(np.float32),

        "Y_valid": np.stack(
            target_valid_windows
        ).astype(np.float32),

        "future_best_gnb": np.stack(
            best_gnb_windows
        ).astype(np.int16),

        "start_route_index": np.asarray(
            start_route_indices,
            dtype=np.int32,
        ),

        "trajectory_id": np.asarray(
            window_trajectory_ids,
            dtype=np.int32,
        ),

        "future_positions_xyz_m": np.stack(
            future_positions
        ).astype(np.float32),
    }


train_windows = (
    build_forecasting_windows(
        train_trajectory_ids
    )
)

validation_windows = (
    build_forecasting_windows(
        validation_trajectory_ids
    )
)

test_windows = (
    build_forecasting_windows(
        test_trajectory_ids
    )
)


# ------------------------------------------------------------
# 9. Save sequence dataset
# ------------------------------------------------------------

np.savez_compressed(
    SEQUENCE_FILE,

    train_X=train_windows["X"],
    train_Y=train_windows["Y"],
    train_Y_valid=train_windows[
        "Y_valid"
    ],
    train_future_best_gnb=train_windows[
        "future_best_gnb"
    ],
    train_start_route_index=train_windows[
        "start_route_index"
    ],
    train_trajectory_id=train_windows[
        "trajectory_id"
    ],
    train_future_positions_xyz_m=train_windows[
        "future_positions_xyz_m"
    ],

    validation_X=validation_windows[
        "X"
    ],
    validation_Y=validation_windows[
        "Y"
    ],
    validation_Y_valid=validation_windows[
        "Y_valid"
    ],
    validation_future_best_gnb=validation_windows[
        "future_best_gnb"
    ],
    validation_start_route_index=validation_windows[
        "start_route_index"
    ],
    validation_trajectory_id=validation_windows[
        "trajectory_id"
    ],
    validation_future_positions_xyz_m=validation_windows[
        "future_positions_xyz_m"
    ],

    test_X=test_windows[
        "X"
    ],
    test_Y=test_windows[
        "Y"
    ],
    test_Y_valid=test_windows[
        "Y_valid"
    ],
    test_future_best_gnb=test_windows[
        "future_best_gnb"
    ],
    test_start_route_index=test_windows[
        "start_route_index"
    ],
    test_trajectory_id=test_windows[
        "trajectory_id"
    ],
    test_future_positions_xyz_m=test_windows[
        "future_positions_xyz_m"
    ],

    feature_mean=feature_mean,

    feature_std=feature_std,

    target_mean=target_mean,

    target_std=target_std,

    gnb_names=np.asarray(
        gnb_names,
        dtype="U",
    ),

    feature_names=np.asarray(
        feature_names,
        dtype="U",
    ),

    history_steps=np.array(
        HISTORY_STEPS,
        dtype=np.int32,
    ),

    forecast_horizon=np.array(
        FORECAST_HORIZON,
        dtype=np.int32,
    ),

    time_step_s=np.array(
        TIME_STEP_S,
        dtype=np.float32,
    ),

    lidar_min_model_range_m=np.array(
        LIDAR_MIN_MODEL_RANGE_M,
        dtype=np.float32,
    ),
)


# ------------------------------------------------------------
# 10. Save metadata
# ------------------------------------------------------------

sequence_metadata = {
    "sequence_dataset": str(
        SEQUENCE_FILE
    ),

    "number_of_gnbs": int(
        number_of_gnbs
    ),

    "gnb_names": gnb_names,

    "number_of_input_features": int(
        model_features.shape[1]
    ),

    "history_steps": int(
        HISTORY_STEPS
    ),

    "forecast_horizon": int(
        FORECAST_HORIZON
    ),

    "time_step_s": float(
        TIME_STEP_S
    ),

    "history_duration_s": float(
        HISTORY_DURATION_S
    ),

    "forecast_duration_s": float(
        FORECAST_DURATION_S
    ),

    "power_floor_db": float(
        POWER_FLOOR_DB
    ),

    "lidar_min_model_range_m": float(
        LIDAR_MIN_MODEL_RANGE_M
    ),

    "lidar_max_model_range_m": float(
        LIDAR_MAX_MODEL_RANGE_M
    ),

    "train_trajectory_ids": (
        train_trajectory_ids.tolist()
    ),

    "validation_trajectory_ids": (
        validation_trajectory_ids.tolist()
    ),

    "test_trajectory_ids": (
        test_trajectory_ids.tolist()
    ),

    "train_window_count": int(
        train_windows["X"].shape[0]
    ),

    "validation_window_count": int(
        validation_windows["X"].shape[0]
    ),

    "test_window_count": int(
        test_windows["X"].shape[0]
    ),

    "feature_names": feature_names,
}


with open(
    SEQUENCE_METADATA_FILE,
    "w",
    encoding="utf-8",
) as metadata_file:

    json.dump(
        sequence_metadata,
        metadata_file,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Final validation report
# ------------------------------------------------------------

print()
print(
    "Forecasting sequence dataset created."
)

print(
    "Input feature dimension:",
    model_features.shape[1],
)

print(
    "History:",
    HISTORY_STEPS,
    "steps =",
    HISTORY_DURATION_S,
    "seconds",
)

print(
    "Forecast horizon:",
    FORECAST_HORIZON,
    "steps =",
    FORECAST_DURATION_S,
    "seconds",
)

print()
print(
    "Training X shape:",
    train_windows["X"].shape,
)

print(
    "Training Y shape:",
    train_windows["Y"].shape,
)

print(
    "Validation X shape:",
    validation_windows["X"].shape,
)

print(
    "Validation Y shape:",
    validation_windows["Y"].shape,
)

print(
    "Testing X shape:",
    test_windows["X"].shape,
)

print(
    "Testing Y shape:",
    test_windows["Y"].shape,
)

print()
print(
    "Saved sequence dataset:"
)

print(SEQUENCE_FILE)

print()
print(
    "Saved sequence metadata:"
)

print(SEQUENCE_METADATA_FILE)

## 2. LOTO folds and hurdle-aware targets

This section reorganizes the saved synchronized dataset for forecasting. For each fold, one trajectory is held out for testing and a neighboring trajectory is used for validation; the remaining trajectories are used for training. Normalization statistics are calculated from training trajectories only.

The binary target is:

```text
valid_path = 1 if the number of valid ray-traced paths is greater than zero
             0 otherwise
```

This target is the availability component used by the hurdle-aware model and by execution validation. It represents ray-path availability, not successful data transmission.


## 2.1 Build the LOTO data files


In [ ]:
from pathlib import Path
import json
import numpy as np

# ------------------------------------------------------------
# 1. Paths — reuse the ALREADY GENERATED dataset, write to a NEW folder
# ------------------------------------------------------------

PROJECT_ROOT = Path.home() / "wcnc_project"

OLD_DATA_DIR = PROJECT_ROOT / "improved_experiment" / "data"
MULTIMODAL_FILE = OLD_DATA_DIR / "synchronized_multimodal_five_gnb_dataset.npz"

V2_ROOT = PROJECT_ROOT / "improved_experiment_v2"
V2_DATA_DIR = V2_ROOT / "data"
V2_RESULTS_DIR = V2_ROOT / "results"

for folder in [V2_DATA_DIR, V2_RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not MULTIMODAL_FILE.exists():
    raise FileNotFoundError(
        f"Expected the already-generated multimodal dataset at:\n{MULTIMODAL_FILE}\n"
        "This notebook does not regenerate it — run the original "
        "'Synchronized multimodal KFUPM dataset' cell first if it is missing."
    )

print("Reading existing dataset from:", MULTIMODAL_FILE)
print("New (non-mixing) output root: ", V2_ROOT)


In [ ]:
# ------------------------------------------------------------
# 2. Load the already-generated synchronized multimodal dataset
# ------------------------------------------------------------

with np.load(MULTIMODAL_FILE, allow_pickle=True) as data:
    positions_xyz_m = np.asarray(data["positions_xyz_m"], dtype=np.float32)
    trajectory_ids = np.asarray(data["trajectory_ids"], dtype=np.int32)
    route_indices = np.asarray(data["route_indices"], dtype=np.int32)
    rf_power_db = np.asarray(data["rf_power_db"], dtype=np.float32)
    valid_path_counts = np.asarray(data["valid_path_counts"], dtype=np.float32)
    minimum_delay_s = np.asarray(data["minimum_delay_s"], dtype=np.float32)
    mobility_features = np.asarray(data["mobility_features"], dtype=np.float32)
    lidar_ranges_m = np.asarray(data["lidar_ranges_m"], dtype=np.float32)
    lidar_summary = np.asarray(data["lidar_summary"], dtype=np.float32)
    best_gnb_index = np.asarray(data["best_gnb_index"], dtype=np.int16)

    def decode_name(value):
        if isinstance(value, (bytes, np.bytes_)):
            return value.decode("utf-8", errors="replace")
        return str(value)

    gnb_names = [decode_name(name) for name in data["gnb_names"]]

number_of_observations = positions_xyz_m.shape[0]
number_of_gnbs = len(gnb_names)
unique_trajectory_ids = np.sort(np.unique(trajectory_ids))
number_of_trajectories = len(unique_trajectory_ids)

print("Observations:", number_of_observations)
print("gNBs:", gnb_names)
print("Trajectories:", unique_trajectory_ids.tolist())

if number_of_trajectories < 3:
    raise RuntimeError("At least 3 trajectories are required for a train/val/test split per fold.")


In [ ]:
# ------------------------------------------------------------
# 3. Forecasting configuration (unchanged from the original notebook,
#    so results stay directly comparable)
# ------------------------------------------------------------

TIME_STEP_S = 0.5
HISTORY_STEPS = 12
FORECAST_HORIZON = 6

POWER_FLOOR_DB = -220.0
POWER_CEILING_DB = 0.0

LIDAR_MIN_MODEL_RANGE_M = 0.5
LIDAR_MAX_MODEL_RANGE_M = 300.0

# Continuous power target (unchanged transform)
target_power_db = np.clip(rf_power_db, POWER_FLOOR_DB, POWER_CEILING_DB).astype(np.float32)

# NEW: explicit hurdle / link-availability target carried alongside the
# continuous target at every stage from here on, instead of being computed
# only after the fact for reporting.
target_valid = (valid_path_counts > 0).astype(np.float32)

model_rf_power_db = target_power_db.copy()
model_valid_paths = np.log1p(np.maximum(valid_path_counts, 0.0)).astype(np.float32)
model_delay_us = np.log1p(np.maximum(
    np.where(np.isfinite(minimum_delay_s) & (minimum_delay_s >= 0.0), minimum_delay_s * 1e6, 0.0),
    0.0,
)).astype(np.float32)
model_mobility = mobility_features.copy()
model_lidar_full = np.clip(lidar_ranges_m, LIDAR_MIN_MODEL_RANGE_M, LIDAR_MAX_MODEL_RANGE_M).astype(np.float32)
model_lidar_summary = lidar_summary.copy()

MOBILITY_NAMES = ["speed_mps", "velocity_x_mps", "velocity_y_mps", "acceleration_x_mps2",
                  "acceleration_y_mps2", "heading_sin", "heading_cos", "yaw_rate_radps",
                  "delta_x_m", "delta_y_m"]

def build_feature_block(lidar_mode):
    """lidar_mode: 'summary' (7-D, recommended) or 'full' (72-D, ablation)."""
    if lidar_mode == "summary":
        lidar_block = model_lidar_summary
        lidar_names = ["lidar_nearest_m", "lidar_mean_m", "lidar_farthest_m",
                       "lidar_p10_m", "lidar_p50_m", "lidar_p90_m", "lidar_hit_fraction"]
    elif lidar_mode == "full":
        lidar_block = model_lidar_full
        lidar_names = [f"lidar_range_{i:02d}_m" for i in range(lidar_block.shape[1])]
    else:
        raise ValueError(f"Unknown lidar_mode: {lidar_mode}")

    feats = np.concatenate(
        [model_rf_power_db, model_valid_paths, model_delay_us, model_mobility, lidar_block], axis=1
    ).astype(np.float32)

    names = (
        [f"{n}_power_db" for n in gnb_names]
        + [f"{n}_log_valid_paths" for n in gnb_names]
        + [f"{n}_log_delay_us" for n in gnb_names]
        + MOBILITY_NAMES
        + lidar_names
    )
    return feats, names

print("Feature block sizes:")
for mode in ["summary", "full"]:
    feats, names = build_feature_block(mode)
    print(f"  lidar_mode={mode:7s} -> input dimension {feats.shape[1]}")


In [ ]:
# ------------------------------------------------------------
# 4. Rotating leave-one-trajectory-out (LOTO) fold definition
# ------------------------------------------------------------

def rotating_fold_trajectories(fold_index, all_traj_ids):
    """
    Fold k: test = trajectory k, validation = trajectory k+1 (mod N),
    train = the remaining N-2 trajectories. This exactly reproduces the
    original notebook's 4-train / 1-val / 1-test structure, repeated N times
    so every trajectory is used once as the held-out test trajectory.
    """
    order = list(all_traj_ids)
    n = len(order)
    test_traj = [order[fold_index % n]]
    val_traj = [order[(fold_index + 1) % n]]
    train_traj = [t for t in order if t not in test_traj + val_traj]
    return train_traj, val_traj, test_traj


def build_windows(selected_traj_ids, normalized_features, normalized_target, valid_target, best_gnb):
    Xs, Ys, Yvalids, bests, starts, tids, futpos = [], [], [], [], [], [], []
    for tid in selected_traj_ids:
        idx = np.flatnonzero(trajectory_ids == tid)
        idx = idx[np.argsort(route_indices[idx])]
        n_pts = len(idx)
        max_start = n_pts - HISTORY_STEPS - FORECAST_HORIZON + 1
        for s in range(max_start):
            in_idx = idx[s: s + HISTORY_STEPS]
            tgt_idx = idx[s + HISTORY_STEPS: s + HISTORY_STEPS + FORECAST_HORIZON]
            Xs.append(normalized_features[in_idx])
            Ys.append(normalized_target[tgt_idx])
            Yvalids.append(valid_target[tgt_idx])
            bests.append(best_gnb[tgt_idx])
            starts.append(route_indices[in_idx[0]])
            tids.append(tid)
            futpos.append(positions_xyz_m[tgt_idx])
    if not Xs:
        raise RuntimeError(f"No temporal windows were created for trajectories {selected_traj_ids}.")
    return dict(
        X=np.stack(Xs).astype(np.float32),
        Y=np.stack(Ys).astype(np.float32),
        Y_valid=np.stack(Yvalids).astype(np.float32),
        future_best_gnb=np.stack(bests).astype(np.int16),
        start_route_index=np.asarray(starts, dtype=np.int32),
        trajectory_id=np.asarray(tids, dtype=np.int32),
        future_positions_xyz_m=np.stack(futpos).astype(np.float32),
    )


In [ ]:
# ------------------------------------------------------------
# 5. Build and save every fold x lidar_mode combination
#    (normalization statistics are always computed from that fold's
#    training trajectories only — no leakage across folds or splits)
# ------------------------------------------------------------

manifest = {
    "source_multimodal_dataset": str(MULTIMODAL_FILE),
    "num_trajectories": int(number_of_trajectories),
    "gnb_names": gnb_names,
    "history_steps": HISTORY_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "folds": [],
}

for lidar_mode in ["summary", "full"]:
    model_features, feature_names = build_feature_block(lidar_mode)

    for fold_index in range(number_of_trajectories):
        train_traj, val_traj, test_traj = rotating_fold_trajectories(fold_index, unique_trajectory_ids)
        train_rows = np.isin(trajectory_ids, train_traj)

        feature_mean = model_features[train_rows].mean(axis=0).astype(np.float32)
        feature_std = model_features[train_rows].std(axis=0).astype(np.float32)
        feature_std[feature_std < 1e-6] = 1.0

        target_mean = target_power_db[train_rows].mean(axis=0).astype(np.float32)
        target_std = target_power_db[train_rows].std(axis=0).astype(np.float32)
        target_std[target_std < 1e-6] = 1.0

        normalized_features = (model_features - feature_mean) / feature_std
        normalized_target = (target_power_db - target_mean) / target_std

        train_w = build_windows(train_traj, normalized_features, normalized_target, target_valid, best_gnb_index)
        val_w = build_windows(val_traj, normalized_features, normalized_target, target_valid, best_gnb_index)
        test_w = build_windows(test_traj, normalized_features, normalized_target, target_valid, best_gnb_index)

        fold_file = V2_DATA_DIR / f"fold_{fold_index}_{lidar_mode}_sequences.npz"
        np.savez_compressed(
            fold_file,
            train_X=train_w["X"], train_Y=train_w["Y"], train_Y_valid=train_w["Y_valid"],
            train_future_best_gnb=train_w["future_best_gnb"], train_trajectory_id=train_w["trajectory_id"],
            val_X=val_w["X"], val_Y=val_w["Y"], val_Y_valid=val_w["Y_valid"],
            val_future_best_gnb=val_w["future_best_gnb"], val_trajectory_id=val_w["trajectory_id"],
            test_X=test_w["X"], test_Y=test_w["Y"], test_Y_valid=test_w["Y_valid"],
            test_future_best_gnb=test_w["future_best_gnb"], test_trajectory_id=test_w["trajectory_id"],
            test_start_route_index=test_w["start_route_index"],
            test_future_positions_xyz_m=test_w["future_positions_xyz_m"],
            feature_mean=feature_mean, feature_std=feature_std,
            target_mean=target_mean, target_std=target_std,
            gnb_names=np.asarray(gnb_names, dtype="U"),
            feature_names=np.asarray(feature_names, dtype="U"),
            history_steps=np.array(HISTORY_STEPS), forecast_horizon=np.array(FORECAST_HORIZON),
        )

        manifest["folds"].append({
            "fold": fold_index, "lidar_mode": lidar_mode,
            "train_trajectories": [int(t) for t in train_traj],
            "validation_trajectory": int(val_traj[0]),
            "test_trajectory": int(test_traj[0]),
            "train_windows": int(train_w["X"].shape[0]),
            "validation_windows": int(val_w["X"].shape[0]),
            "test_windows": int(test_w["X"].shape[0]),
            "input_dimension": int(model_features.shape[1]),
            "file": str(fold_file),
        })

        print(f"[{lidar_mode:7s}] fold {fold_index} | test_traj={test_traj[0]:>2} val_traj={val_traj[0]:>2} | "
              f"windows train/val/test = {train_w['X'].shape[0]}/{val_w['X'].shape[0]}/{test_w['X'].shape[0]}")

with open(V2_RESULTS_DIR / "loto_fold_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print()
print("Saved fold manifest to:", V2_RESULTS_DIR / "loto_fold_manifest.json")
print("Total fold files written:", len(manifest["folds"]))


## 3. Train and evaluate the forecasting models

The forecasting models receive a history of synchronized RF, mobility, validity, and LiDAR/geometric features and predict six future steps for all five gNBs.

The models are:

- Persistence;
- deterministic GRU;
- independent probabilistic GRU;
- joint probabilistic GRU; and
- hurdle-aware joint probabilistic GRU.

The deterministic model is trained with Smooth L1 loss. The probabilistic models use negative log-likelihood. The hurdle-aware model adds a binary availability head trained with binary cross-entropy. This section corresponds to the forecasting output used by Eq. (7) and the future-quality terms in Eq. (8).


## 3.1 Forecasting model definitions, training, and LOTO evaluation


In [ ]:
from pathlib import Path
import copy
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Normal, Independent, LowRankMultivariateNormal

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.home() / "wcnc_project"
V2_DATA_DIR = PROJECT_ROOT / "improved_experiment_v2" / "data"
V2_RESULTS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "results"
V2_MODELS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "models"
V2_MODELS_DIR.mkdir(parents=True, exist_ok=True)
V2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = V2_RESULTS_DIR / "loto_fold_manifest.json"
if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_FILE}. Run 01_build_loto_folds_hurdle_targets.ipynb first."
    )
with open(MANIFEST_FILE) as f:
    fold_manifest = json.load(f)

NUM_FOLDS = fold_manifest["num_trajectories"]
GNB_NAMES = fold_manifest["gnb_names"]
print(f"Found {NUM_FOLDS} LOTO folds for gNBs: {GNB_NAMES}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ------------------------------------------------------------
# Model definitions
# ------------------------------------------------------------

class GRUEncoder(nn.Module):
    """Shared temporal encoder — unchanged from the original notebook."""

    def __init__(self, input_dim, hidden_dim=128, number_of_layers=2, dropout=0.10):
        super().__init__()
        effective_dropout = dropout if number_of_layers > 1 else 0.0
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim,
                           num_layers=number_of_layers, batch_first=True,
                           dropout=effective_dropout)
        self.projection = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, hidden_dim), nn.GELU()
        )

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.projection(hidden[-1])


class DeterministicGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def forward(self, x):
        out = self.output_head(self.encoder(x))
        return out.view(-1, self.horizon, self.number_of_gnbs)


class IndependentProbabilisticGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.mean_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.variance_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        variance = F.softplus(self.variance_head(context)) + 1e-3
        return Independent(Normal(mean, torch.sqrt(variance)), 1)

    def nll(self, x, y):
        dist = self.distribution(x)
        return -dist.log_prob(y.reshape(y.shape[0], -1))


class JointProbabilisticGRU(nn.Module):
    """The paper's current proposed model — low-rank joint Gaussian, unchanged."""

    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs, covariance_rank=4):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_dimension = horizon * number_of_gnbs
        self.covariance_rank = covariance_rank
        self.mean_head = nn.Linear(hidden_dim, self.output_dimension)
        self.diagonal_head = nn.Linear(hidden_dim, self.output_dimension)
        self.factor_head = nn.Linear(hidden_dim, self.output_dimension * covariance_rank)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        diagonal = F.softplus(self.diagonal_head(context)) + 1e-3
        factor = self.factor_head(context).view(-1, self.output_dimension, self.covariance_rank)
        factor = factor / np.sqrt(self.output_dimension)
        return LowRankMultivariateNormal(loc=mean, cov_factor=factor, cov_diag=diagonal)

    def nll(self, x, y):
        dist = self.distribution(x)
        return -dist.log_prob(y.reshape(y.shape[0], -1))


class HurdleJointProbabilisticGRU(nn.Module):
    """
    Proposed improved agentic forecaster.

    Adds a per-(step, gNB) link-availability classifier trained jointly with
    the low-rank joint Gaussian power forecaster. During training, targets
    for (step, gNB) entries with no usable link are replaced by the model's
    own detached prediction before computing the Gaussian NLL, so the loss
    is not dominated by an arbitrary sentinel floor value. The classifier
    head also gives the agentic handover policy a direct P(link usable)
    signal to gate candidate gNBs (see notebook 03).
    """

    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs,
                 covariance_rank=4, aux_weight=0.3):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_dimension = horizon * number_of_gnbs
        self.covariance_rank = covariance_rank
        self.aux_weight = aux_weight
        self.mean_head = nn.Linear(hidden_dim, self.output_dimension)
        self.diagonal_head = nn.Linear(hidden_dim, self.output_dimension)
        self.factor_head = nn.Linear(hidden_dim, self.output_dimension * covariance_rank)
        self.link_head = nn.Linear(hidden_dim, self.output_dimension)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        diagonal = F.softplus(self.diagonal_head(context)) + 1e-3
        factor = self.factor_head(context).view(-1, self.output_dimension, self.covariance_rank)
        factor = factor / np.sqrt(self.output_dimension)
        return LowRankMultivariateNormal(loc=mean, cov_factor=factor, cov_diag=diagonal)

    def link_logits(self, x):
        return self.link_head(self.encoder(x))

    def link_probabilities(self, x):
        return torch.sigmoid(self.link_logits(x))

    def loss(self, x, y, y_valid):
        dist = self.distribution(x)
        target = y.reshape(y.shape[0], -1)
        mask_flat = y_valid.reshape(y_valid.shape[0], -1)
        masked_target = mask_flat * target + (1.0 - mask_flat) * dist.mean.detach()
        power_nll = -dist.log_prob(masked_target)
        link_logits = self.link_head(self.encoder(x))
        link_bce = F.binary_cross_entropy_with_logits(link_logits, mask_flat, reduction="none").mean(dim=1)
        return power_nll + self.aux_weight * link_bce


In [ ]:
# ------------------------------------------------------------
# Training hyperparameters — identical to the original notebook
# ------------------------------------------------------------

HIDDEN_DIMENSION = 128
NUMBER_OF_LAYERS = 2
COVARIANCE_RANK = 4
AUX_WEIGHT = 0.3
BATCH_SIZE = 64
EPOCHS = 120
PATIENCE = 18
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


def train_one_model(model, model_name, kind, train_X, train_Y, train_Yv, val_X, val_Y, val_Yv, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(train_X), torch.from_numpy(train_Y), torch.from_numpy(train_Yv)),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(val_X), torch.from_numpy(val_Y), torch.from_numpy(val_Yv)),
        batch_size=BATCH_SIZE, shuffle=False,
    )

    best_val, best_state, bad_epochs = np.inf, None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for batch_X, batch_Y, batch_Yv in train_loader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)
            batch_Yv = batch_Yv.to(device)
            optimizer.zero_grad(set_to_none=True)
            if kind == "deterministic":
                loss = F.smooth_l1_loss(model(batch_X), batch_Y)
            elif kind == "probabilistic":
                loss = model.nll(batch_X, batch_Y).mean()
            elif kind == "hurdle":
                loss = model.loss(batch_X, batch_Y, batch_Yv).mean()
            else:
                raise ValueError(kind)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_X, batch_Y, batch_Yv in val_loader:
                batch_X = batch_X.to(device)
                batch_Y = batch_Y.to(device)
                batch_Yv = batch_Yv.to(device)
                if kind == "deterministic":
                    vloss = F.smooth_l1_loss(model(batch_X), batch_Y)
                elif kind == "probabilistic":
                    vloss = model.nll(batch_X, batch_Y).mean()
                elif kind == "hurdle":
                    vloss = model.loss(batch_X, batch_Y, batch_Yv).mean()
                val_losses.append(float(vloss.detach().cpu()))

        mean_train, mean_val = float(np.mean(train_losses)), float(np.mean(val_losses))
        history["train_loss"].append(mean_train)
        history["val_loss"].append(mean_val)

        if mean_val < best_val:
            best_val, best_state, bad_epochs = mean_val, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1

        if verbose and (epoch == 1 or epoch % 20 == 0 or epoch == EPOCHS):
            print(f"    {model_name:32s} | epoch {epoch:3d} | train {mean_train:.4f} | val {mean_val:.4f}")
        if bad_epochs >= PATIENCE:
            if verbose:
                print(f"    {model_name}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_val


In [ ]:
# ------------------------------------------------------------
# Fair evaluation helpers
# ------------------------------------------------------------

def inverse_norm(values_normalized, mean, std):
    return values_normalized * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)


def masked_point_metrics(pred_db, target_db, valid_mask, true_best_gnb):
    """masked_* only scores (step, gNB) entries where the ground-truth link is usable
    -- the fair number. raw_* reproduces the original notebook's sentinel-inclusive metric."""
    valid_mask_bool = valid_mask.astype(bool)
    error_db = pred_db - target_db

    if valid_mask_bool.any():
        masked_mae = float(np.mean(np.abs(error_db[valid_mask_bool])))
        masked_rmse = float(np.sqrt(np.mean(error_db[valid_mask_bool] ** 2)))
    else:
        masked_mae, masked_rmse = float("nan"), float("nan")

    raw_mae = float(np.mean(np.abs(error_db)))
    raw_rmse = float(np.sqrt(np.mean(error_db ** 2)))

    predicted_best = np.argmax(pred_db, axis=2)
    valid_best = true_best_gnb >= 0
    serving_accuracy = (
        float(np.mean(predicted_best[valid_best] == true_best_gnb[valid_best]))
        if valid_best.any() else float("nan")
    )

    return {
        "masked_mae_db": masked_mae, "masked_rmse_db": masked_rmse,
        "raw_mae_db": raw_mae, "raw_rmse_db": raw_rmse,
        "serving_gnb_accuracy": serving_accuracy,
        "fraction_valid_targets": float(valid_mask_bool.mean()),
    }


def link_classifier_metrics(link_prob, y_valid):
    predicted = (link_prob >= 0.5).astype(np.float32)
    truth = y_valid.astype(np.float32)
    tp = float(np.sum((predicted == 1) & (truth == 1)))
    fp = float(np.sum((predicted == 1) & (truth == 0)))
    fn = float(np.sum((predicted == 0) & (truth == 1)))
    tn = float(np.sum((predicted == 0) & (truth == 0)))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    return {"link_accuracy": accuracy, "link_precision": precision, "link_recall": recall, "link_f1": f1}


In [ ]:
# ------------------------------------------------------------
# Revised per-fold training + evaluation driver
#
# The model definitions and training procedure above are unchanged.  This
# driver additionally saves validation predictions and the last observed RF
# vector so that policy parameters are selected without using the test route.
# ------------------------------------------------------------

def observed_power_db(X, feature_mean, feature_std, number_of_gnbs):
    """Recover the final observed RF vector from the history window."""
    return (
        X[:, -1, :number_of_gnbs] * feature_std[:number_of_gnbs]
        + feature_mean[:number_of_gnbs]
    ).astype(np.float32)


def _distribution_outputs(model, X, target_mean, target_std, horizon,
                          number_of_gnbs, sample_count=300):
    """Return denormalized mean and predictive standard deviation."""
    model.eval()
    with torch.no_grad():
        tx = torch.from_numpy(X).to(device)
        dist = model.distribution(tx)
        mean_norm = dist.mean.reshape(-1, horizon, number_of_gnbs).cpu().numpy()
        samples = dist.rsample((sample_count,))
        samples = samples.permute(1, 0, 2).reshape(
            -1, sample_count, horizon, number_of_gnbs
        ).cpu().numpy()

    mean_db = inverse_norm(mean_norm, target_mean, target_std)
    std_db = samples.std(axis=1) * target_std.reshape(1, 1, -1)
    return mean_db.astype(np.float32), std_db.astype(np.float32)


def _deterministic_outputs(model, X, target_mean, target_std):
    model.eval()
    with torch.no_grad():
        pred_norm = model(torch.from_numpy(X).to(device)).cpu().numpy()
    return {"mean_db": inverse_norm(pred_norm, target_mean, target_std).astype(np.float32)}


def _probabilistic_outputs(model, X, target_mean, target_std, horizon,
                           number_of_gnbs):
    mean_db, std_db = _distribution_outputs(
        model, X, target_mean, target_std, horizon, number_of_gnbs
    )
    return {"mean_db": mean_db, "std_db": std_db}


def _hurdle_outputs(model, X, target_mean, target_std, horizon, number_of_gnbs):
    outputs = _probabilistic_outputs(
        model, X, target_mean, target_std, horizon, number_of_gnbs
    )
    model.eval()
    with torch.no_grad():
        tx = torch.from_numpy(X).to(device)
        outputs["link_prob"] = model.link_probabilities(tx).reshape(
            -1, horizon, number_of_gnbs
        ).cpu().numpy().astype(np.float32)
    return outputs


def run_fold(fold_index, lidar_mode, models_to_run):
    fold_file = V2_DATA_DIR / f"fold_{fold_index}_{lidar_mode}_sequences.npz"
    with np.load(fold_file, allow_pickle=True) as d:
        train_X, train_Y, train_Yv = d["train_X"], d["train_Y"], d["train_Y_valid"]
        val_X, val_Y, val_Yv = d["val_X"], d["val_Y"], d["val_Y_valid"]
        test_X, test_Y, test_Yv = d["test_X"], d["test_Y"], d["test_Y_valid"]
        val_best = d["val_future_best_gnb"]
        test_best = d["test_future_best_gnb"]
        feature_mean, feature_std = d["feature_mean"], d["feature_std"]
        target_mean, target_std = d["target_mean"], d["target_std"]
        gnb_names = [str(x) for x in d["gnb_names"]]
        horizon = int(d["forecast_horizon"])

    number_of_gnbs = len(gnb_names)
    input_dim = train_X.shape[-1]
    val_target_db = inverse_norm(val_Y, target_mean, target_std)
    test_target_db = inverse_norm(test_Y, target_mean, target_std)
    val_observed_power_db = observed_power_db(
        val_X, feature_mean, feature_std, number_of_gnbs
    )
    test_observed_power_db = observed_power_db(
        test_X, feature_mean, feature_std, number_of_gnbs
    )

    metrics, saved_scores = {}, {}
    common = dict(
        input_dim=input_dim, hidden_dim=HIDDEN_DIMENSION,
        number_of_layers=NUMBER_OF_LAYERS, horizon=horizon,
        number_of_gnbs=number_of_gnbs,
    )

    def save_and_score(name, val_outputs, test_outputs):
        saved_scores[name] = {"val": val_outputs, "test": test_outputs}
        fold_metrics = masked_point_metrics(
            test_outputs["mean_db"], test_target_db, test_Yv, test_best
        )
        if name == "hurdle_joint":
            fold_metrics.update(link_classifier_metrics(
                test_outputs["link_prob"], test_Yv
            ))
        metrics[name] = fold_metrics

    if "persistence" in models_to_run:
        val_last = val_observed_power_db
        test_last = test_observed_power_db
        val_outputs = {"mean_db": np.repeat(val_last[:, None, :], horizon, axis=1)}
        test_outputs = {"mean_db": np.repeat(test_last[:, None, :], horizon, axis=1)}
        save_and_score("persistence", val_outputs, test_outputs)

    if "deterministic" in models_to_run:
        model = DeterministicGRU(**common)
        model, _, _ = train_one_model(
            model, f"deterministic[{lidar_mode}] fold{fold_index}",
            "deterministic", train_X, train_Y, train_Yv,
            val_X, val_Y, val_Yv
        )
        save_and_score(
            "deterministic",
            _deterministic_outputs(model, val_X, target_mean, target_std),
            _deterministic_outputs(model, test_X, target_mean, target_std),
        )
        torch.save(
            model.state_dict(),
            V2_MODELS_DIR / f"deterministic_{lidar_mode}_fold{fold_index}.pt",
        )

    if "independent" in models_to_run:
        model = IndependentProbabilisticGRU(**common)
        model, _, _ = train_one_model(
            model, f"independent[{lidar_mode}] fold{fold_index}",
            "probabilistic", train_X, train_Y, train_Yv,
            val_X, val_Y, val_Yv
        )
        save_and_score(
            "independent",
            _probabilistic_outputs(
                model, val_X, target_mean, target_std, horizon, number_of_gnbs
            ),
            _probabilistic_outputs(
                model, test_X, target_mean, target_std, horizon, number_of_gnbs
            ),
        )
        torch.save(
            model.state_dict(),
            V2_MODELS_DIR / f"independent_{lidar_mode}_fold{fold_index}.pt",
        )

    if "joint" in models_to_run:
        model = JointProbabilisticGRU(covariance_rank=COVARIANCE_RANK, **common)
        model, _, _ = train_one_model(
            model, f"joint[{lidar_mode}] fold{fold_index}",
            "probabilistic", train_X, train_Y, train_Yv,
            val_X, val_Y, val_Yv
        )
        save_and_score(
            "joint",
            _probabilistic_outputs(
                model, val_X, target_mean, target_std, horizon, number_of_gnbs
            ),
            _probabilistic_outputs(
                model, test_X, target_mean, target_std, horizon, number_of_gnbs
            ),
        )
        torch.save(
            model.state_dict(),
            V2_MODELS_DIR / f"joint_{lidar_mode}_fold{fold_index}.pt",
        )

    if "hurdle_joint" in models_to_run:
        model = HurdleJointProbabilisticGRU(
            covariance_rank=COVARIANCE_RANK, aux_weight=AUX_WEIGHT, **common
        )
        model, _, _ = train_one_model(
            model, f"hurdle_joint[{lidar_mode}] fold{fold_index}",
            "hurdle", train_X, train_Y, train_Yv,
            val_X, val_Y, val_Yv
        )
        save_and_score(
            "hurdle_joint",
            _hurdle_outputs(
                model, val_X, target_mean, target_std, horizon, number_of_gnbs
            ),
            _hurdle_outputs(
                model, test_X, target_mean, target_std, horizon, number_of_gnbs
            ),
        )
        torch.save(
            model.state_dict(),
            V2_MODELS_DIR / f"hurdle_joint_{lidar_mode}_fold{fold_index}.pt",
        )

    return metrics, saved_scores, dict(
        val_target_db=val_target_db, val_Yv=val_Yv, val_best=val_best,
        val_observed_power_db=val_observed_power_db,
        test_target_db=test_target_db, test_Yv=test_Yv, test_best=test_best,
        test_observed_power_db=test_observed_power_db,
        gnb_names=gnb_names, horizon=horizon, number_of_gnbs=number_of_gnbs,
    )


In [ ]:
# ------------------------------------------------------------
# Main comparison: all five models, compact (summary) LiDAR features, six LOTO folds
# ------------------------------------------------------------

MAIN_MODELS = ["persistence", "deterministic", "independent", "joint", "hurdle_joint"]

per_fold_results = {}
per_fold_scores = {}
per_fold_context = {}

for fold_index in range(NUM_FOLDS):
    print(f"\n=== Fold {fold_index} (lidar_mode=summary) ===")
    metrics, scores, context = run_fold(fold_index, "summary", MAIN_MODELS)
    per_fold_results[fold_index] = metrics
    per_fold_scores[fold_index] = scores
    per_fold_context[fold_index] = context

    payload = {
        "val_target_db": context["val_target_db"],
        "val_Yv": context["val_Yv"],
        "val_best": context["val_best"],
        "val_observed_power_db": context["val_observed_power_db"],
        "test_target_db": context["test_target_db"],
        "test_Yv": context["test_Yv"],
        "test_best": context["test_best"],
        "test_observed_power_db": context["test_observed_power_db"],
        "gnb_names": np.asarray(context["gnb_names"], dtype="U"),
    }
    for name, split_outputs in scores.items():
        for split, outputs in split_outputs.items():
            for suffix, values in outputs.items():
                payload[f"{split}_{name}_{suffix}"] = values

    np.savez_compressed(
        V2_RESULTS_DIR / f"fold_{fold_index}_summary_predictions.npz", **payload
    )


In [ ]:
# ------------------------------------------------------------
# Aggregate mean +/- std across the 6 folds, plus paired significance tests
# ------------------------------------------------------------

from scipy import stats

def aggregate_metric(model_name, metric_name):
    values = np.array([per_fold_results[k][model_name][metric_name] for k in range(NUM_FOLDS)
                        if metric_name in per_fold_results[k][model_name]], dtype=np.float64)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return None
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0, values


summary_rows = []
key_metrics = ["masked_mae_db", "masked_rmse_db", "raw_mae_db", "raw_rmse_db", "serving_gnb_accuracy"]

print(f"{'model':16s} " + " | ".join(f"{m:22s}" for m in key_metrics))
for model_name in MAIN_MODELS:
    row = {"model": model_name}
    line = f"{model_name:16s} "
    for metric_name in key_metrics:
        agg = aggregate_metric(model_name, metric_name)
        if agg is None:
            line += " | " + " " * 22
            continue
        mean_v, std_v, _ = agg
        row[metric_name + "_mean"] = mean_v
        row[metric_name + "_std"] = std_v
        line += f" | {mean_v:8.3f} +/- {std_v:8.3f} "
    summary_rows.append(row)
    print(line)

# Link-detection quality (hurdle_joint only)
link_metrics = ["link_accuracy", "link_precision", "link_recall", "link_f1"]
print("\nHurdle-joint auxiliary link-availability classifier (mean +/- std over folds):")
for metric_name in link_metrics:
    agg = aggregate_metric("hurdle_joint", metric_name)
    if agg is not None:
        print(f"  {metric_name:16s} {agg[0]:.4f} +/- {agg[1]:.4f}")

# Paired significance test: hurdle_joint vs the paper's current proposed model (joint),
# paired by fold, on the fair masked_mae_db metric.
hurdle_vals = np.array([per_fold_results[k]["hurdle_joint"]["masked_mae_db"] for k in range(NUM_FOLDS)])
joint_vals = np.array([per_fold_results[k]["joint"]["masked_mae_db"] for k in range(NUM_FOLDS)])
t_stat, p_value = stats.ttest_rel(joint_vals, hurdle_vals)
print(f"\nPaired t-test (masked_mae_db), joint vs hurdle_joint across {NUM_FOLDS} folds:")
print(f"  mean masked MAE  joint={joint_vals.mean():.3f} dB   hurdle_joint={hurdle_vals.mean():.3f} dB")
print(f"  t={t_stat:.3f}  p={p_value:.4f}  (paired, one row per held-out trajectory)")

with open(V2_RESULTS_DIR / "forecaster_loto_summary.json", "w", encoding="utf-8") as f:
    json.dump({
        "num_folds": NUM_FOLDS,
        "models": MAIN_MODELS,
        "per_fold_results": {str(k): v for k, v in per_fold_results.items()},
        "aggregate_rows": summary_rows,
        "significance_masked_mae_joint_vs_hurdle_joint": {
            "t_stat": float(t_stat), "p_value": float(p_value),
            "joint_mean": float(joint_vals.mean()), "hurdle_joint_mean": float(hurdle_vals.mean()),
        },
    }, f, indent=2)

print("\nSaved:", V2_RESULTS_DIR / "forecaster_loto_summary.json")


## Ablation: compact vs full LiDAR feature block

The main comparison above uses the compact 7-D `lidar_summary` features by
default. This cell repeats training for just the two joint models
(`joint`, `hurdle_joint`) with the original 72-D raw LiDAR rays
(`lidar_mode="full"`) across all 6 folds, to check whether the dimensionality
reduction helps, hurts, or is neutral on this dataset size. Skip this cell if
you only need the headline comparison above — it roughly doubles the
notebook''s runtime.


In [ ]:
ABLATION_MODELS = ["joint", "hurdle_joint"]
ablation_results = {"summary": {}, "full": {}}

for lidar_mode in ["summary", "full"]:
    for fold_index in range(NUM_FOLDS):
        if lidar_mode == "summary":
            # Already computed above -- reuse instead of retraining.
            ablation_results["summary"][fold_index] = {
                m: per_fold_results[fold_index][m] for m in ABLATION_MODELS
            }
            continue
        print(f"\n=== Ablation fold {fold_index} (lidar_mode=full) ===")
        metrics, _, _ = run_fold(fold_index, "full", ABLATION_MODELS)
        ablation_results["full"][fold_index] = metrics

print("\nLiDAR feature ablation (masked_mae_db, mean +/- std over folds):")
for model_name in ABLATION_MODELS:
    for lidar_mode in ["summary", "full"]:
        vals = np.array([ablation_results[lidar_mode][k][model_name]["masked_mae_db"] for k in range(NUM_FOLDS)])
        vals = vals[~np.isnan(vals)]
        print(f"  {model_name:14s} lidar_mode={lidar_mode:7s} {vals.mean():.3f} +/- {vals.std(ddof=1):.3f} dB")

with open(V2_RESULTS_DIR / "lidar_feature_ablation.json", "w", encoding="utf-8") as f:
    json.dump({
        lidar_mode: {str(k): v for k, v in results.items()}
        for lidar_mode, results in ablation_results.items()
    }, f, indent=2)

print("\nSaved:", V2_RESULTS_DIR / "lidar_feature_ablation.json")


## 4. Three-agent runtime evaluation

The following cells implement the paper's three logical agents.

### Forecasting Agent — Eq. (7)
It receives the current observation and validation priors, selects a forecasting model, and supplies future power, uncertainty, and optional availability probabilities.

### Planning Agent — Eq. (8)
It uses risk-aware future scores and backward-induction dynamic programming to compare feasible action sequences. It includes uncertainty, link availability, reliability, handover cost, and TTT persistence. Only the first action of the selected sequence is applied.

### Adaptation Agent — Eqs. (9)--(10)
It validates the executed action using the realized validity mask, updates the selected-gNB reliability and recent failure state, and updates forecasting preferences for later decisions.


## 4.1 Robust three-agent system, baselines, and curves


In [ ]:
# Cell 1 — Configuration and environment setup

from pathlib import Path
import json
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional PyTorch setup
try:
    import torch

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch device: {DEVICE}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    torch = None
    DEVICE = "cpu"
    print("PyTorch is not available; using CPU.")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# Project directories
PROJECT_ROOT = Path("/home/afanali/wcnc_project")

RESULTS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "results"
FIGURES_DIR = PROJECT_ROOT / "improved_experiment_v2" / "figures_three_agents_robust"
REPORT_DIR = PROJECT_ROOT / "improved_experiment_v2" / "robust_three_agent_results"

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment configuration
NUM_FOLDS = 6
NUM_GNBS = 5
NUM_FORECAST_MODELS = 5

FORECAST_MODELS = [
    "persistence",
    "deterministic",
    "independent",
    "joint",
    "hurdle_joint",
]

# Policy configuration
DEFAULT_TTT_K = 2
DEFAULT_HANDOVER_COST_DB = 2.0
DEFAULT_FAILURE_PENALTY_DB = 2.0
DEFAULT_UNCERTAINTY_WEIGHT = 1.0

# Basic checks
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert RESULTS_DIR.exists(), f"Results directory not found: {RESULTS_DIR}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Figures will be saved to: {FIGURES_DIR}")
print(f"Reports will be saved to: {REPORT_DIR}")
print(f"Forecasting models: {FORECAST_MODELS}")
print(f"Number of LOTO folds: {NUM_FOLDS}")

In [ ]:
# Cell 2 — Locate saved forecast and LOTO result files

from pathlib import Path

# Search recursively for previously generated data files
candidate_files = []

for pattern in ["*.pkl", "*.pickle", "*.joblib", "*.npz", "*.json"]:
    candidate_files.extend(RESULTS_DIR.rglob(pattern))

candidate_files = sorted(candidate_files)

print(f"Found {len(candidate_files)} result files:\n")

for file_path in candidate_files:
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{file_path.relative_to(PROJECT_ROOT)}  |  {size_mb:.2f} MB")

# Likely files containing forecasts or LOTO fold outputs
keywords = [
    "forecast",
    "fold",
    "comparison",
    "runtime",
    "three_agent",
    "policy",
    "main",
]

likely_files = [
    path for path in candidate_files
    if any(key in path.name.lower() for key in keywords)
]

print("\nLikely experiment-input files:")
for file_path in likely_files:
    print(f"  {file_path.relative_to(PROJECT_ROOT)}")

if not candidate_files:
    print(
        "\nNo serialized result files were found. "
        "Check that the previous training/evaluation notebook has been executed."
    )

In [ ]:
# Cell 3 — Load six LOTO folds and existing forecast outputs

import json
import numpy as np

# Load fold-level forecast files
fold_data = []

for fold_index in range(NUM_FOLDS):
    fold_path = RESULTS_DIR / f"fold_{fold_index}_summary_predictions.npz"

    if not fold_path.exists():
        raise FileNotFoundError(f"Missing fold file: {fold_path}")

    with np.load(fold_path, allow_pickle=True) as data:
        fold = {key: data[key] for key in data.files}

    fold_data.append(fold)

# Load supporting summaries
def load_json(path):
    with open(path, "r") as file:
        return json.load(file)

forecaster_summary = load_json(
    RESULTS_DIR / "forecaster_loto_summary.json"
)

policy_summary = load_json(
    RESULTS_DIR / "policy_perfold_metrics.json"
)

agentic_summary = load_json(
    RESULTS_DIR / "agentic_policy_loto_comparison.json"
)

fold_manifest = load_json(
    RESULTS_DIR / "loto_fold_manifest.json"
)

# Inspect the loaded data
print(f"Loaded {len(fold_data)} LOTO folds.")

print("\nFirst-fold keys:")
print(sorted(fold_data[0].keys()))

print("\nImportant array shapes:")
for key in [
    "test_target_db",
    "test_Yv",
    "test_observed_power_db",
]:
    if key in fold_data[0]:
        print(f"{key}: {fold_data[0][key].shape}")

print("\nAvailable summary files:")
print(f"Forecaster summary: {type(forecaster_summary).__name__}")
print(f"Policy summary:     {type(policy_summary).__name__}")
print(f"Agentic summary:    {type(agentic_summary).__name__}")
print(f"Fold manifest:      {type(fold_manifest).__name__}")

In [ ]:
# Cell 4 — Common forecast interface and evaluation helpers

from dataclasses import dataclass
from typing import Optional

MODEL_KEY_MAP = {
    "persistence": "persistence",
    "deterministic": "deterministic",
    "independent": "independent",
    "joint": "joint",
    "hurdle_joint": "hurdle_joint",
}

@dataclass
class ForecastBatch:
    mean_db: np.ndarray
    std_db: Optional[np.ndarray]
    validity_prob: Optional[np.ndarray]
    target_db: np.ndarray
    validity_mask: np.ndarray


def get_forecast_batch(fold, split, model_name):
    """
    Extract one model's forecast and target arrays.

    Returned shapes:
        mean_db:        [decision_epoch, horizon, gNB]
        std_db:         [decision_epoch, horizon, gNB] or None
        validity_prob:  [decision_epoch, horizon, gNB] or None
        target_db:      [decision_epoch, horizon, gNB]
        validity_mask:  [decision_epoch, horizon, gNB]
    """
    prefix = f"{split}_"

    mean_key = f"{prefix}{model_name}_mean_db"
    std_key = f"{prefix}{model_name}_std_db"
    prob_key = f"{prefix}{model_name}_link_prob"

    if mean_key not in fold:
        raise KeyError(f"Missing forecast key: {mean_key}")

    mean_db = np.asarray(fold[mean_key], dtype=np.float32)
    target_db = np.asarray(fold[f"{prefix}target_db"], dtype=np.float32)
    validity_mask = np.asarray(fold[f"{prefix}Yv"], dtype=np.float32)

    std_db = None
    if std_key in fold:
        std_db = np.asarray(fold[std_key], dtype=np.float32)

    validity_prob = None
    if prob_key in fold:
        validity_prob = np.asarray(fold[prob_key], dtype=np.float32)

    return ForecastBatch(
        mean_db=mean_db,
        std_db=std_db,
        validity_prob=validity_prob,
        target_db=target_db,
        validity_mask=validity_mask,
    )


def masked_mae_db(pred_db, target_db, validity_mask, eps=1e-8):
    """MAE computed only on valid ray-traced links."""
    error = np.abs(pred_db - target_db)
    mask = validity_mask.astype(bool)

    if not np.any(mask):
        return np.nan

    return float(error[mask].sum() / (mask.sum() + eps))


def first_step_mae_db(pred_db, target_db, validity_mask, eps=1e-8):
    """MAE for the first predicted future step."""
    return masked_mae_db(
        pred_db[:, 0, :],
        target_db[:, 0, :],
        validity_mask[:, 0, :],
        eps=eps,
    )


def validity_brier_score(prob, validity_mask, eps=1e-8):
    """Brier score for predicted link availability."""
    if prob is None:
        return np.nan

    prob = np.clip(prob, 0.0, 1.0)
    error = (prob - validity_mask) ** 2
    return float(np.mean(error))


def describe_fold(fold, fold_index=0):
    print(f"Fold {fold_index}")
    print(f"Number of decision epochs: {fold['test_target_db'].shape[0]}")
    print(f"Prediction horizon:       {fold['test_target_db'].shape[1]}")
    print(f"Number of candidate gNBs:  {fold['test_target_db'].shape[2]}")
    print(f"gNB names:                {list(fold['gnb_names'])}")


# Verify the interface on the first fold
describe_fold(fold_data[0], fold_index=0)

print("\nValidation forecast quality:")
for model_name in FORECAST_MODELS:
    batch = get_forecast_batch(fold_data[0], "val", model_name)

    mae = masked_mae_db(
        batch.mean_db,
        batch.target_db,
        batch.validity_mask,
    )

    brier = validity_brier_score(
        batch.validity_prob,
        batch.validity_mask,
    )

    print(
        f"{model_name:15s} | "
        f"masked MAE = {mae:7.3f} dB | "
        f"availability Brier = {brier:.4f}"
        if not np.isnan(brier)
        else
        f"{model_name:15s} | "
        f"masked MAE = {mae:7.3f} dB | "
        f"availability Brier = N/A"
    )

In [ ]:
# Cell 5 — Compute policy-aware validation statistics

def first_step_policy_statistics(batch, availability_threshold=0.5):
    """
    Evaluate how useful a forecast is for selecting the next target gNB.
    Statistics are computed only from validation data.
    """
    pred_db = batch.mean_db[:, 0, :].copy()
    true_db = batch.target_db[:, 0, :]
    true_valid = batch.validity_mask[:, 0, :].astype(bool)

    # True best usable gNB
    masked_true_db = np.where(true_valid, true_db, -np.inf)
    has_valid_target = np.any(true_valid, axis=1)

    true_best = np.argmax(masked_true_db, axis=1)

    # Use predicted availability only for the hurdle-aware model
    if batch.validity_prob is not None:
        pred_prob = np.clip(batch.validity_prob[:, 0, :], 0.0, 1.0)
        predicted_valid = pred_prob >= availability_threshold

        # If all candidates are rejected, fall back to the highest predicted power
        all_rejected = ~np.any(predicted_valid, axis=1)
        pred_score_db = pred_db.copy()
        pred_score_db[~predicted_valid] = -np.inf
        pred_score_db[all_rejected] = pred_db[all_rejected]

        predicted_best = np.argmax(pred_score_db, axis=1)
        predicted_validity = pred_prob[
            np.arange(len(predicted_best)),
            predicted_best,
        ]
    else:
        predicted_best = np.argmax(pred_db, axis=1)
        predicted_validity = np.ones(len(predicted_best))

    selected_real_validity = true_valid[
        np.arange(len(predicted_best)),
        predicted_best,
    ]

    selected_power = true_db[
        np.arange(len(predicted_best)),
        predicted_best,
    ]

    oracle_power = np.max(masked_true_db, axis=1)
    valid_gap = oracle_power - selected_power

    usable = has_valid_target & np.isfinite(valid_gap)

    return {
        "selection_accuracy": float(
            np.mean(predicted_best[usable] == true_best[usable])
        ) if np.any(usable) else np.nan,

        "selected_validity": float(
            np.mean(selected_real_validity[usable])
        ) if np.any(usable) else np.nan,

        "mean_oracle_gap_db": float(
            np.mean(valid_gap[usable])
        ) if np.any(usable) else np.nan,

        "predicted_availability_mean": float(
            np.mean(predicted_validity[usable])
        ) if np.any(usable) else np.nan,

        "usable_samples": int(np.sum(usable)),
        "total_samples": int(len(predicted_best)),
    }


# Compute validation statistics across all folds
validation_policy_stats = {}

for model_name in FORECAST_MODELS:
    model_stats = []

    for fold_index, fold in enumerate(fold_data):
        batch = get_forecast_batch(fold, "val", model_name)

        stats = first_step_policy_statistics(
            batch,
            availability_threshold=0.5,
        )
        stats["fold"] = fold_index
        model_stats.append(stats)

    validation_policy_stats[model_name] = model_stats


# Print aggregated validation statistics
print("Validation policy usefulness:")
print("-" * 78)

for model_name in FORECAST_MODELS:
    rows = validation_policy_stats[model_name]

    selection_accuracy = np.array(
        [row["selection_accuracy"] for row in rows],
        dtype=float,
    )
    selected_validity = np.array(
        [row["selected_validity"] for row in rows],
        dtype=float,
    )
    oracle_gap = np.array(
        [row["mean_oracle_gap_db"] for row in rows],
        dtype=float,
    )

    print(
        f"{model_name:15s} | "
        f"target accuracy = {np.nanmean(selection_accuracy):.3f} | "
        f"validity = {np.nanmean(selected_validity):.3f} | "
        f"oracle gap = {np.nanmean(oracle_gap):.3f} dB"
    )

In [ ]:
# Cell 6 — Build policy-aware forecasting priors

# Aggregate validation statistics
prior_rows = []

for model_name in FORECAST_MODELS:
    rows = validation_policy_stats[model_name]

    mae_values = []
    selection_values = []
    validity_values = []
    gap_values = []

    for fold_index, fold in enumerate(fold_data):
        batch = get_forecast_batch(fold, "val", model_name)

        mae_values.append(
            masked_mae_db(
                batch.mean_db,
                batch.target_db,
                batch.validity_mask,
            )
        )

    for row in rows:
        selection_values.append(row["selection_accuracy"])
        validity_values.append(row["selected_validity"])
        gap_values.append(row["mean_oracle_gap_db"])

    prior_rows.append({
        "model": model_name,
        "mae_db": np.nanmean(mae_values),
        "selection_accuracy": np.nanmean(selection_values),
        "selected_validity": np.nanmean(validity_values),
        "oracle_gap_db": np.nanmean(gap_values),
    })

policy_prior_df = pd.DataFrame(prior_rows)


def minmax_good(values, lower_is_better=False):
    """
    Normalize values to [0, 1], where larger means better.
    """
    values = np.asarray(values, dtype=float)
    v_min = np.nanmin(values)
    v_max = np.nanmax(values)

    if np.isclose(v_max, v_min):
        score = np.ones_like(values)
    else:
        score = (values - v_min) / (v_max - v_min)

    if lower_is_better:
        score = 1.0 - score

    return score


# Normalize each validation criterion
policy_prior_df["mae_score"] = minmax_good(
    policy_prior_df["mae_db"],
    lower_is_better=True,
)

policy_prior_df["selection_score"] = minmax_good(
    policy_prior_df["selection_accuracy"],
    lower_is_better=False,
)

policy_prior_df["validity_score"] = minmax_good(
    policy_prior_df["selected_validity"],
    lower_is_better=False,
)

policy_prior_df["gap_score"] = minmax_good(
    policy_prior_df["oracle_gap_db"],
    lower_is_better=True,
)

# Balanced policy-aware score.
# The score intentionally prevents masked MAE from dominating target selection.
POLICY_SCORE_WEIGHTS = {
    "mae_score": 0.20,
    "selection_score": 0.30,
    "validity_score": 0.25,
    "gap_score": 0.25,
}

policy_prior_df["policy_score"] = sum(
    weight * policy_prior_df[criterion]
    for criterion, weight in POLICY_SCORE_WEIGHTS.items()
)

policy_prior_df = policy_prior_df.sort_values(
    "policy_score",
    ascending=False,
).reset_index(drop=True)

print("Policy-aware validation priors:")
display(
    policy_prior_df[
        [
            "model",
            "mae_db",
            "selection_accuracy",
            "selected_validity",
            "oracle_gap_db",
            "policy_score",
        ]
    ].round(4)
)

print("\nPreferred model according to policy-aware validation:")
print(policy_prior_df.iloc[0]["model"])

In [ ]:
# Cell 7 — Policy-aware Forecasting Agent

class PolicyAwareForecastingAgent:
    """
    Selects the forecaster using:
      1. Validation-based policy usefulness
      2. Recent causal forecasting reliability
      3. Predictive uncertainty
      4. Predicted link-availability confidence
    """

    def __init__(
        self,
        policy_prior_df,
        models,
        uncertainty_weight=0.15,
        availability_weight=0.15,
        reliability_weight=0.25,
    ):
        self.models = list(models)

        self.prior_scores = {
            row["model"]: float(row["policy_score"])
            for _, row in policy_prior_df.iterrows()
        }

        self.uncertainty_weight = uncertainty_weight
        self.availability_weight = availability_weight
        self.reliability_weight = reliability_weight

        self.recent_errors_db = {
            model: [] for model in self.models
        }

        self.selection_history = []

    def reset(self):
        self.recent_errors_db = {
            model: [] for model in self.models
        }
        self.selection_history = []

    def update_feedback(
        self,
        model_name,
        predicted_db,
        observed_db,
        observed_validity=None,
        max_history=20,
    ):
        """
        Update causal model reliability after the next observation becomes
        available. The current observation is never used before prediction.
        """
        predicted_db = np.asarray(predicted_db).reshape(-1)
        observed_db = np.asarray(observed_db).reshape(-1)

        valid = np.isfinite(predicted_db) & np.isfinite(observed_db)

        if observed_validity is not None:
            observed_validity = np.asarray(observed_validity).reshape(-1)
            valid = valid & observed_validity.astype(bool)

        if np.any(valid):
            error_db = float(
                np.mean(np.abs(predicted_db[valid] - observed_db[valid]))
            )

            self.recent_errors_db[model_name].append(error_db)
            self.recent_errors_db[model_name] = (
                self.recent_errors_db[model_name][-max_history:]
            )

    def _reliability_score(self, model_name):
        errors = self.recent_errors_db[model_name]

        if not errors:
            return 1.0

        mean_error = float(np.mean(errors))

        # Smooth conversion from recent MAE to a [0, 1] reliability score
        return float(np.exp(-mean_error / 10.0))

    @staticmethod
    def _normalize(values, larger_is_better=True):
        values = np.asarray(values, dtype=float)

        if np.allclose(values.max(), values.min()):
            return np.ones_like(values)

        normalized = (values - values.min()) / (
            values.max() - values.min()
        )

        if not larger_is_better:
            normalized = 1.0 - normalized

        return normalized

    def select(self, forecast_batches, decision_epoch=0):
        """
        forecast_batches:
            Dictionary mapping model names to ForecastBatch objects.
            Each batch should contain one or more decision epochs.

        Returns:
            selected model, model scores, and diagnostic information.
        """
        model_uncertainty = {}
        model_availability = {}
        model_predicted_target = {}

        for model_name in self.models:
            batch = forecast_batches[model_name]

            mean_db = batch.mean_db[0, 0, :]
            model_predicted_target[model_name] = int(np.argmax(mean_db))

            if batch.std_db is not None:
                uncertainty = float(
                    np.mean(batch.std_db[0, : min(2, batch.std_db.shape[1]), :])
                )
            else:
                uncertainty = 0.0

            model_uncertainty[model_name] = uncertainty

            if batch.validity_prob is not None:
                target_index = model_predicted_target[model_name]
                availability = float(
                    np.mean(
                        batch.validity_prob[
                            0,
                            : min(2, batch.validity_prob.shape[1]),
                            target_index,
                        ]
                    )
                )
            else:
                # No explicit availability head
                availability = 0.90

            model_availability[model_name] = availability

        uncertainty_values = np.array(
            [model_uncertainty[m] for m in self.models],
            dtype=float,
        )

        availability_values = np.array(
            [model_availability[m] for m in self.models],
            dtype=float,
        )

        uncertainty_scores = self._normalize(
            uncertainty_values,
            larger_is_better=False,
        )

        availability_scores = self._normalize(
            availability_values,
            larger_is_better=True,
        )

        total_scores = {}

        for index, model_name in enumerate(self.models):
            prior_score = self.prior_scores.get(model_name, 0.0)
            reliability_score = self._reliability_score(model_name)

            total_score = (
                0.60 * prior_score
                + self.reliability_weight * reliability_score
                + self.uncertainty_weight * uncertainty_scores[index]
                + self.availability_weight * availability_scores[index]
            )

            total_scores[model_name] = float(total_score)

        selected_model = max(
            total_scores,
            key=total_scores.get,
        )

        self.selection_history.append({
            "decision_epoch": int(decision_epoch),
            "selected_model": selected_model,
            "scores": total_scores.copy(),
        })

        return {
            "selected_model": selected_model,
            "scores": total_scores,
            "uncertainty": model_uncertainty,
            "availability": model_availability,
            "predicted_target": model_predicted_target,
        }


# Initialize the agent
forecasting_agent = PolicyAwareForecastingAgent(
    policy_prior_df=policy_prior_df,
    models=FORECAST_MODELS,
)

# Test the agent on the first test epoch of Fold 0
test_forecasts_fold0 = {
    model_name: get_forecast_batch(
        fold_data[0],
        "test",
        model_name,
    )
    for model_name in FORECAST_MODELS
}

first_epoch_forecasts = {
    model_name: ForecastBatch(
        mean_db=batch.mean_db[:1],
        std_db=None if batch.std_db is None else batch.std_db[:1],
        validity_prob=(
            None
            if batch.validity_prob is None
            else batch.validity_prob[:1]
        ),
        target_db=batch.target_db[:1],
        validity_mask=batch.validity_mask[:1],
    )
    for model_name, batch in test_forecasts_fold0.items()
}

forecasting_decision = forecasting_agent.select(
    first_epoch_forecasts,
    decision_epoch=0,
)

print("Forecasting Agent decision:")
print(f"Selected model: {forecasting_decision['selected_model']}")

print("\nModel scores:")
for model_name, score in sorted(
    forecasting_decision["scores"].items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{model_name:15s}: {score:.4f}")

In [ ]:
# Cell 8 — Risk-aware Planning Agent with finite-horizon DP and TTT

class RiskAwarePlanningAgent:
    """
    Selects:
      - handover target
      - planning horizon
      - uncertainty weight
      - failure penalty
      - TTT persistence

    The planner evaluates candidate future target sequences using a
    finite-horizon dynamic-programming objective.
    """

    def __init__(
        self,
        horizon_options=(1, 2, 3, 4),
        beta_options=(0.25, 0.5, 1.0, 1.5),
        failure_penalty_options=(1.0, 2.0, 4.0),
        ttt_options=(1, 2, 3),
        handover_cost_db=2.0,
        reliability_penalty_db=2.0,
    ):
        self.horizon_options = tuple(horizon_options)
        self.beta_options = tuple(beta_options)
        self.failure_penalty_options = tuple(
            failure_penalty_options
        )
        self.ttt_options = tuple(ttt_options)

        self.handover_cost_db = float(handover_cost_db)
        self.reliability_penalty_db = float(
            reliability_penalty_db
        )

        self.plan_history = []

    @staticmethod
    def _get_uncertainty(batch, horizon):
        if batch.std_db is None:
            return np.zeros_like(batch.mean_db[0, :horizon, :])

        return np.nan_to_num(
            batch.std_db[0, :horizon, :],
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    @staticmethod
    def _get_availability(batch, horizon):
        if batch.validity_prob is None:
            return np.ones_like(batch.mean_db[0, :horizon, :])

        return np.clip(
            np.nan_to_num(
                batch.validity_prob[0, :horizon, :],
                nan=0.0,
                posinf=1.0,
                neginf=0.0,
            ),
            0.0,
            1.0,
        )

    def _dynamic_programming_plan(
        self,
        batch,
        serving_index,
        reliability,
        horizon,
        beta,
        failure_penalty,
        ttt_k,
    ):
        mean_db = np.asarray(
            batch.mean_db[0, :horizon, :],
            dtype=float,
        )

        uncertainty_db = self._get_uncertainty(
            batch,
            horizon,
        )

        availability = self._get_availability(
            batch,
            horizon,
        )

        reliability = np.asarray(reliability, dtype=float)
        reliability = np.clip(reliability, 0.0, 1.0)

        # Risk-aware predicted utility
        utility = (
            mean_db
            - beta * uncertainty_db
            - failure_penalty * (1.0 - availability)
            - self.reliability_penalty_db
            * (1.0 - reliability[None, :])
        )

        num_steps, num_gnbs = utility.shape

        # Dynamic-programming tables
        dp = np.full(
            (num_steps, num_gnbs),
            -np.inf,
            dtype=float,
        )

        first_action = np.full(
            (num_steps, num_gnbs),
            serving_index,
            dtype=int,
        )

        # First future step: transition from current serving gNB
        for target_index in range(num_gnbs):
            switch_cost = (
                self.handover_cost_db
                if target_index != serving_index
                else 0.0
            )

            dp[0, target_index] = (
                utility[0, target_index]
                - switch_cost
            )

            first_action[0, target_index] = target_index

        # Future steps
        for step in range(1, num_steps):
            for target_index in range(num_gnbs):
                transition_values = np.empty(num_gnbs)

                for previous_index in range(num_gnbs):
                    switch_cost = (
                        self.handover_cost_db
                        if target_index != previous_index
                        else 0.0
                    )

                    transition_values[previous_index] = (
                        dp[step - 1, previous_index]
                        - switch_cost
                    )

                best_previous = int(
                    np.argmax(transition_values)
                )

                dp[step, target_index] = (
                    utility[step, target_index]
                    + transition_values[best_previous]
                )

                first_action[step, target_index] = (
                    first_action[
                        step - 1,
                        best_previous,
                    ]
                )

        # Select best terminal state
        terminal_index = int(np.argmax(dp[-1, :]))
        recommended_target = int(
            first_action[-1, terminal_index]
        )

        # Enforce TTT persistence on a proposed handover
        if recommended_target != serving_index:
            persistence_steps = min(ttt_k, num_steps)

            target_utility = utility[
                :persistence_steps,
                recommended_target,
            ]

            serving_utility = utility[
                :persistence_steps,
                serving_index,
            ]

            persistent = np.all(
                target_utility >= serving_utility
            )

            if not persistent:
                recommended_target = serving_index

        objective = float(
            dp[-1, terminal_index]
        )

        uncertainty_level = float(
            np.mean(uncertainty_db)
        )

        return {
            "recommended_target": recommended_target,
            "horizon": int(horizon),
            "beta": float(beta),
            "failure_penalty_db": float(failure_penalty),
            "ttt_k": int(ttt_k),
            "uncertainty_level": uncertainty_level,
            "objective": objective,
            "utility": utility,
        }

    def plan(
        self,
        batch,
        serving_index,
        reliability=None,
        decision_epoch=0,
    ):
        num_gnbs = batch.mean_db.shape[-1]

        if reliability is None:
            reliability = np.ones(num_gnbs)

        candidate_plans = []

        max_available_horizon = batch.mean_db.shape[1]

        for horizon in self.horizon_options:
            if horizon > max_available_horizon:
                continue

            for beta in self.beta_options:
                for failure_penalty in (
                    self.failure_penalty_options
                ):
                    for ttt_k in self.ttt_options:
                        plan = self._dynamic_programming_plan(
                            batch=batch,
                            serving_index=serving_index,
                            reliability=reliability,
                            horizon=horizon,
                            beta=beta,
                            failure_penalty=failure_penalty,
                            ttt_k=ttt_k,
                        )

                        candidate_plans.append(plan)

        if not candidate_plans:
            raise RuntimeError(
                "No valid planning configuration was found."
            )

        best_plan = max(
            candidate_plans,
            key=lambda plan: plan["objective"],
        )

        if best_plan["recommended_target"] == serving_index:
            action = "stay"
        else:
            action = "handover"

        best_plan["serving_index"] = int(serving_index)
        best_plan["action"] = action
        best_plan["decision_epoch"] = int(decision_epoch)

        self.plan_history.append(best_plan)

        return best_plan


# Initialize the Planning Agent
planning_agent = RiskAwarePlanningAgent(
    horizon_options=(1, 2, 3, 4),
    beta_options=(0.25, 0.5, 1.0, 1.5),
    failure_penalty_options=(1.0, 2.0, 4.0),
    ttt_options=(1, 2, 3),
    handover_cost_db=2.0,
    reliability_penalty_db=2.0,
)

# Test on Fold 0, first test epoch, using the selected forecaster
selected_model = forecasting_decision["selected_model"]
selected_batch = first_epoch_forecasts[selected_model]

initial_serving_index = 4
initial_reliability = np.ones(NUM_GNBS)

planning_decision = planning_agent.plan(
    batch=selected_batch,
    serving_index=initial_serving_index,
    reliability=initial_reliability,
    decision_epoch=0,
)

print("Planning Agent decision:")
for key in [
    "action",
    "recommended_target",
    "serving_index",
    "horizon",
    "beta",
    "failure_penalty_db",
    "ttt_k",
    "uncertainty_level",
    "objective",
]:
    print(f"{key:22s}: {planning_decision[key]}")

In [ ]:
# Cell 9 — Adaptation Agent for reliability and risk updates

class AdaptationAgent:
    """
    Updates online control information using execution feedback:

      - per-gNB reliability
      - per-forecaster reliability
      - forecast weights
      - recent failure rate
      - switching pressure
      - planning risk multiplier
    """

    def __init__(
        self,
        models,
        num_gnbs,
        prior_scores=None,
        reliability_alpha=0.20,
        error_scale_db=10.0,
        softmax_temperature=0.20,
    ):
        self.models = list(models)
        self.num_gnbs = int(num_gnbs)

        self.reliability_alpha = float(
            reliability_alpha
        )
        self.error_scale_db = float(error_scale_db)
        self.softmax_temperature = float(
            softmax_temperature
        )

        self.gnb_reliability = np.ones(self.num_gnbs)
        self.model_reliability = {
            model: 1.0 for model in self.models
        }

        self.model_error_ema_db = {
            model: 0.0 for model in self.models
        }

        if prior_scores is None:
            prior_scores = {
                model: 1.0 for model in self.models
            }

        self.prior_scores = {
            model: float(prior_scores.get(model, 1.0))
            for model in self.models
        }

        self.forecast_weights = self._compute_forecast_weights()

        self.recent_failure_rate = 0.0
        self.recent_switching_pressure = 0.0
        self.update_history = []

    def reset(self):
        self.gnb_reliability = np.ones(self.num_gnbs)

        self.model_reliability = {
            model: 1.0 for model in self.models
        }

        self.model_error_ema_db = {
            model: 0.0 for model in self.models
        }

        self.forecast_weights = self._compute_forecast_weights()

        self.recent_failure_rate = 0.0
        self.recent_switching_pressure = 0.0
        self.update_history = []

    def _compute_forecast_weights(self):
        """
        Combine validation priors and online model reliability.
        """
        raw_scores = []

        for model in self.models:
            prior = max(self.prior_scores[model], 1e-6)
            reliability = max(
                self.model_reliability[model],
                1e-6,
            )

            raw_scores.append(
                np.log(prior) + np.log(reliability)
            )

        raw_scores = np.asarray(raw_scores)

        raw_scores = raw_scores / self.softmax_temperature
        raw_scores = raw_scores - np.max(raw_scores)

        weights = np.exp(raw_scores)
        weights = weights / np.sum(weights)

        return {
            model: float(weight)
            for model, weight in zip(
                self.models,
                weights,
            )
        }

    def update(
        self,
        predictions_by_model,
        observed_power_db,
        observed_validity,
        serving_index,
        selected_target,
        action,
    ):
        """
        Update the agent after the next realized observation is available.

        predictions_by_model:
            Dictionary of first-step predictions with shape [gNB].

        observed_power_db:
            Realized first-step path power with shape [gNB].

        observed_validity:
            Realized validity mask with shape [gNB].
        """
        observed_power_db = np.asarray(
            observed_power_db,
            dtype=float,
        ).reshape(-1)

        observed_validity = np.asarray(
            observed_validity,
            dtype=bool,
        ).reshape(-1)

        # Update model reliability using only realized valid links
        for model_name in self.models:
            prediction = np.asarray(
                predictions_by_model[model_name],
                dtype=float,
            ).reshape(-1)

            valid = (
                observed_validity
                & np.isfinite(prediction)
                & np.isfinite(observed_power_db)
            )

            if np.any(valid):
                error_db = float(
                    np.mean(
                        np.abs(
                            prediction[valid]
                            - observed_power_db[valid]
                        )
                    )
                )

                old_error = self.model_error_ema_db[
                    model_name
                ]

                new_error = (
                    (1.0 - self.reliability_alpha)
                    * old_error
                    + self.reliability_alpha
                    * error_db
                )

                self.model_error_ema_db[
                    model_name
                ] = new_error

                self.model_reliability[
                    model_name
                ] = float(
                    np.exp(
                        -new_error
                        / self.error_scale_db
                    )
                )

        # Update target-gNB reliability
        target_valid = bool(
            observed_validity[selected_target]
        )

        old_gnb_reliability = self.gnb_reliability[
            selected_target
        ]

        target_feedback = 1.0 if target_valid else 0.0

        self.gnb_reliability[
            selected_target
        ] = (
            (1.0 - self.reliability_alpha)
            * old_gnb_reliability
            + self.reliability_alpha
            * target_feedback
        )

        # Update failure and switching statistics
        is_handover = action == "handover"
        is_failure = is_handover and not target_valid

        self.recent_failure_rate = (
            (1.0 - self.reliability_alpha)
            * self.recent_failure_rate
            + self.reliability_alpha
            * float(is_failure)
        )

        self.recent_switching_pressure = (
            (1.0 - self.reliability_alpha)
            * self.recent_switching_pressure
            + self.reliability_alpha
            * float(is_handover)
        )

        # Recompute model weights after feedback
        self.forecast_weights = (
            self._compute_forecast_weights()
        )

        planning_risk_multiplier = (
            1.0
            + self.recent_failure_rate
            + 0.5
            * self.recent_switching_pressure
        )

        update_record = {
            "selected_target": int(selected_target),
            "target_valid": target_valid,
            "action": action,
            "failure": bool(is_failure),
            "failure_rate": self.recent_failure_rate,
            "switching_pressure": (
                self.recent_switching_pressure
            ),
            "planning_risk_multiplier": (
                planning_risk_multiplier
            ),
            "gnb_reliability": (
                self.gnb_reliability.copy()
            ),
            "model_reliability": (
                self.model_reliability.copy()
            ),
            "forecast_weights": (
                self.forecast_weights.copy()
            ),
        }

        self.update_history.append(update_record)

        return update_record


# Initialize the Adaptation Agent
prior_score_lookup = {
    row["model"]: float(row["policy_score"])
    for _, row in policy_prior_df.iterrows()
}

adaptation_agent = AdaptationAgent(
    models=FORECAST_MODELS,
    num_gnbs=NUM_GNBS,
    prior_scores=prior_score_lookup,
)

# Demonstration update using the first realized future step of Fold 0
predictions_by_model = {
    model_name: test_forecasts_fold0[
        model_name
    ].mean_db[0, 0, :]
    for model_name in FORECAST_MODELS
}

observed_power_db = fold_data[0][
    "test_target_db"
][0, 0, :]

observed_validity = fold_data[0][
    "test_Yv"
][0, 0, :]

adaptation_update = adaptation_agent.update(
    predictions_by_model=predictions_by_model,
    observed_power_db=observed_power_db,
    observed_validity=observed_validity,
    serving_index=initial_serving_index,
    selected_target=planning_decision[
        "recommended_target"
    ],
    action=planning_decision["action"],
)

print("Adaptation Agent update:")
print(
    "gNB reliability:",
    np.round(
        adaptation_update["gnb_reliability"],
        3,
    ),
)

print("\nForecast weights:")
for model_name, weight in sorted(
    adaptation_update["forecast_weights"].items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{model_name:15s}: {weight:.3f}")

print(
    "\nRecent failure rate:",
    round(
        adaptation_update["failure_rate"],
        3,
    ),
)

print(
    "Planning risk multiplier:",
    round(
        adaptation_update["planning_risk_multiplier"],
        3,
    ),
)

In [ ]:
# Cell 10 — Run the complete three-agent loop on one fold

def slice_forecast_batch(batch, epoch_index):
    """
    Extract one decision epoch while preserving the expected
    [epoch, horizon, gNB] dimensions.
    """
    return ForecastBatch(
        mean_db=batch.mean_db[
            epoch_index:epoch_index + 1
        ],
        std_db=(
            None
            if batch.std_db is None
            else batch.std_db[
                epoch_index:epoch_index + 1
            ]
        ),
        validity_prob=(
            None
            if batch.validity_prob is None
            else batch.validity_prob[
                epoch_index:epoch_index + 1
            ]
        ),
        target_db=batch.target_db[
            epoch_index:epoch_index + 1
        ],
        validity_mask=batch.validity_mask[
            epoch_index:epoch_index + 1
        ],
    )


def run_three_agent_fold(
    fold,
    fold_index,
    initial_serving_index=4,
):
    """
    Run Forecasting, Planning, and Adaptation Agents
    over one complete LOTO test fold.
    """
    num_epochs = fold["test_target_db"].shape[0]

    # Load all model forecasts once
    all_forecasts = {
        model_name: get_forecast_batch(
            fold,
            "test",
            model_name,
        )
        for model_name in FORECAST_MODELS
    }

    # Create fresh agents for this fold
    forecasting_agent = PolicyAwareForecastingAgent(
        policy_prior_df=policy_prior_df,
        models=FORECAST_MODELS,
    )

    planning_agent = RiskAwarePlanningAgent(
        horizon_options=(1, 2, 3, 4),
        beta_options=(0.25, 0.5, 1.0, 1.5),
        failure_penalty_options=(1.0, 2.0, 4.0),
        ttt_options=(1, 2, 3),
        handover_cost_db=2.0,
        reliability_penalty_db=2.0,
    )

    adaptation_agent = AdaptationAgent(
        models=FORECAST_MODELS,
        num_gnbs=NUM_GNBS,
        prior_scores=prior_score_lookup,
    )

    serving_index = int(initial_serving_index)
    pending_target = None
    pending_count = 0

    trajectory_records = []
    handover_count = 0

    for epoch_index in range(num_epochs):
        # Extract all forecasts for the current decision epoch
        epoch_forecasts = {
            model_name: slice_forecast_batch(
                batch,
                epoch_index,
            )
            for model_name, batch in all_forecasts.items()
        }

        # ---------------------------------------------------------
        # 1. Forecasting Agent
        # ---------------------------------------------------------
        forecast_decision = forecasting_agent.select(
            epoch_forecasts,
            decision_epoch=epoch_index,
        )

        selected_model = forecast_decision[
            "selected_model"
        ]

        selected_batch = epoch_forecasts[
            selected_model
        ]

        # ---------------------------------------------------------
        # 2. Planning Agent
        # ---------------------------------------------------------
        risk_multiplier = (
            1.0
            + adaptation_agent.recent_failure_rate
            + 0.5
            * adaptation_agent.recent_switching_pressure
        )

        # Adapt the planning risk penalty using feedback
        planning_agent.reliability_penalty_db = (
            2.0 * risk_multiplier
        )

        planning_decision = planning_agent.plan(
            batch=selected_batch,
            serving_index=serving_index,
            reliability=(
                adaptation_agent.gnb_reliability
            ),
            decision_epoch=epoch_index,
        )

        recommended_target = int(
            planning_decision[
                "recommended_target"
            ]
        )

        selected_ttt = int(
            planning_decision["ttt_k"]
        )

        # ---------------------------------------------------------
        # TTT execution state
        # ---------------------------------------------------------
        if recommended_target == serving_index:
            pending_target = None
            pending_count = 0
            executed_action = "stay"
            new_serving_index = serving_index

        else:
            if pending_target == recommended_target:
                pending_count += 1
            else:
                pending_target = recommended_target
                pending_count = 1

            if pending_count >= selected_ttt:
                new_serving_index = recommended_target
                executed_action = "handover"
                handover_count += 1

                pending_target = None
                pending_count = 0
            else:
                new_serving_index = serving_index
                executed_action = "stay"

        # ---------------------------------------------------------
        # 3. Realized next-step feedback
        # ---------------------------------------------------------
        realized_power_db = fold[
            "test_target_db"
        ][epoch_index, 0, :]

        realized_validity = fold[
            "test_Yv"
        ][epoch_index, 0, :].astype(bool)

        # Update forecasting reliability for every model
        for model_name in FORECAST_MODELS:
            forecasting_agent.update_feedback(
                model_name=model_name,
                predicted_db=epoch_forecasts[
                    model_name
                ].mean_db[0, 0, :],
                observed_db=realized_power_db,
                observed_validity=realized_validity,
            )

        # Update adaptation state using the recommended target
        adaptation_agent.update(
            predictions_by_model={
                model_name: epoch_forecasts[
                    model_name
                ].mean_db[0, 0, :]
                for model_name in FORECAST_MODELS
            },
            observed_power_db=realized_power_db,
            observed_validity=realized_validity,
            serving_index=serving_index,
            selected_target=recommended_target,
            action=executed_action,
        )

        # ---------------------------------------------------------
        # Evaluation metrics for this epoch
        # ---------------------------------------------------------
        selected_actual_index = int(
            new_serving_index
        )

        selected_power_db = float(
            realized_power_db[
                selected_actual_index
            ]
        )

        selected_valid = bool(
            realized_validity[
                selected_actual_index
            ]
        )

        usable_powers = realized_power_db[
            realized_validity
        ]

        if len(usable_powers) > 0:
            oracle_power_db = float(
                np.max(usable_powers)
            )
            oracle_gap_db = (
                oracle_power_db
                - selected_power_db
            )
        else:
            oracle_power_db = np.nan
            oracle_gap_db = np.nan

        trajectory_records.append({
            "fold": fold_index,
            "epoch": epoch_index,
            "serving_before": serving_index,
            "serving_after": new_serving_index,
            "selected_model": selected_model,
            "recommended_target": recommended_target,
            "executed_action": executed_action,
            "handover": int(
                executed_action == "handover"
            ),
            "ttt_k": selected_ttt,
            "horizon": planning_decision[
                "horizon"
            ],
            "beta": planning_decision[
                "beta"
            ],
            "uncertainty_level": planning_decision[
                "uncertainty_level"
            ],
            "selected_power_db": selected_power_db,
            "selected_valid": int(
                selected_valid
            ),
            "oracle_gap_db": oracle_gap_db,
            "failure_rate": (
                adaptation_agent.recent_failure_rate
            ),
            "switching_pressure": (
                adaptation_agent.recent_switching_pressure
            ),
            "risk_multiplier": risk_multiplier,
        })

        serving_index = new_serving_index

    trajectory_df = pd.DataFrame(
        trajectory_records
    )

    summary = {
        "fold": fold_index,
        "epochs": num_epochs,
        "handovers": int(handover_count),
        "first_step_validity": float(
            trajectory_df["selected_valid"].mean()
        ),
        "invalid_action_rate": float(
            1.0
            - trajectory_df["selected_valid"].mean()
        ),
        "mean_oracle_gap_db": float(
            trajectory_df["oracle_gap_db"].mean()
        ),
        "mean_selected_power_db": float(
            trajectory_df["selected_power_db"].mean()
        ),
        "final_serving_index": int(
            serving_index
        ),
    }

    return {
        "summary": summary,
        "trajectory": trajectory_df,
        "forecasting_agent": forecasting_agent,
        "planning_agent": planning_agent,
        "adaptation_agent": adaptation_agent,
    }


# Run one complete test fold
fold0_run = run_three_agent_fold(
    fold=fold_data[0],
    fold_index=0,
    initial_serving_index=4,
)

print("Fold 0 three-agent summary:")
for key, value in fold0_run["summary"].items():
    print(f"{key:25s}: {value}")

print("\nFirst five trajectory records:")
display(
    fold0_run["trajectory"].head()
)

In [ ]:
# Cell 11 — Correct metrics for invalid-link entries

def recompute_safe_summary(
    fold,
    trajectory_df,
):
    """
    Recompute metrics without allowing invalid-link values
    such as -220 dB to inflate the oracle gap.
    """
    num_epochs = len(trajectory_df)

    selected_validity = []
    selected_power_valid = []
    usable_gaps = []
    no_usable_gnb_count = 0

    for epoch_index in range(num_epochs):
        serving_after = int(
            trajectory_df.iloc[epoch_index][
                "serving_after"
            ]
        )

        target_db = fold[
            "test_target_db"
        ][epoch_index, 0, :]

        validity = fold[
            "test_Yv"
        ][epoch_index, 0, :].astype(bool)

        selected_valid = bool(
            validity[serving_after]
        )

        selected_validity.append(
            float(selected_valid)
        )

        if selected_valid:
            selected_power_valid.append(
                float(target_db[serving_after])
            )

        if not np.any(validity):
            no_usable_gnb_count += 1
            continue

        if selected_valid:
            oracle_power = float(
                np.max(target_db[validity])
            )

            selected_power = float(
                target_db[serving_after]
            )

            usable_gaps.append(
                oracle_power - selected_power
            )

    safe_summary = {
        "epochs": int(num_epochs),
        "handovers": int(
            trajectory_df["handover"].sum()
        ),
        "first_step_validity": float(
            np.mean(selected_validity)
        ),
        "invalid_action_rate": float(
            1.0 - np.mean(selected_validity)
        ),
        "usable_oracle_gap_db": float(
            np.mean(usable_gaps)
        ) if usable_gaps else np.nan,
        "mean_selected_power_valid_db": float(
            np.mean(selected_power_valid)
        ) if selected_power_valid else np.nan,
        "valid_selected_samples": int(
            len(selected_power_valid)
        ),
        "no_usable_gnb_samples": int(
            no_usable_gnb_count
        ),
        "final_serving_index": int(
            trajectory_df.iloc[-1][
                "serving_after"
            ]
        ),
    }

    return safe_summary


# Recompute the Fold 0 metrics safely
safe_fold0_summary = recompute_safe_summary(
    fold=fold_data[0],
    trajectory_df=fold0_run["trajectory"],
)

print("Corrected Fold 0 three-agent summary:")
for key, value in safe_fold0_summary.items():
    print(f"{key:30s}: {value}")

In [ ]:
# Cell 12 — Run the robust three-agent system on all folds

all_three_agent_runs = {}
robust_fold_summaries = []
all_trajectory_records = []

for fold_index, fold in enumerate(fold_data):
    print(f"\nRunning robust three-agent system on Fold {fold_index}...")

    fold_run = run_three_agent_fold(
        fold=fold,
        fold_index=fold_index,
        initial_serving_index=4,
    )

    safe_summary = recompute_safe_summary(
        fold=fold,
        trajectory_df=fold_run["trajectory"],
    )

    safe_summary["fold"] = fold_index

    all_three_agent_runs[fold_index] = fold_run
    robust_fold_summaries.append(safe_summary)

    fold_trajectory = fold_run["trajectory"].copy()
    all_trajectory_records.append(fold_trajectory)

    print(
        f"  Handovers: {safe_summary['handovers']} | "
        f"Validity: {safe_summary['first_step_validity']:.3f} | "
        f"Usable gap: "
        f"{safe_summary['usable_oracle_gap_db']:.3f} dB"
    )

# Aggregate results
robust_summary_df = pd.DataFrame(
    robust_fold_summaries
).sort_values("fold").reset_index(drop=True)

robust_trajectory_df = pd.concat(
    all_trajectory_records,
    ignore_index=True,
)

print("\nSix-fold robust three-agent summary:")
display(
    robust_summary_df[
        [
            "fold",
            "handovers",
            "first_step_validity",
            "invalid_action_rate",
            "usable_oracle_gap_db",
            "mean_selected_power_valid_db",
            "no_usable_gnb_samples",
        ]
    ].round(4)
)

print("\nMean ± standard deviation:")
summary_metrics = [
    "handovers",
    "first_step_validity",
    "invalid_action_rate",
    "usable_oracle_gap_db",
    "mean_selected_power_valid_db",
]

for metric in summary_metrics:
    values = robust_summary_df[metric].astype(float)

    print(
        f"{metric:30s}: "
        f"{values.mean():.4f} ± {values.std(ddof=1):.4f}"
    )

# Save fold-level results
robust_summary_path = (
    REPORT_DIR / "robust_three_agent_per_fold.csv"
)

robust_trajectory_path = (
    REPORT_DIR / "robust_three_agent_trajectories.csv"
)

robust_summary_df.to_csv(
    robust_summary_path,
    index=False,
)

robust_trajectory_df.to_csv(
    robust_trajectory_path,
    index=False,
)

print("\nSaved:")
print(robust_summary_path)
print(robust_trajectory_path)

In [ ]:
# Cell 13 — Evaluate Event A3, persistence, and fixed DP–TTT baselines

def apply_ttt_decision(
    serving_index,
    recommended_target,
    pending_target,
    pending_count,
    ttt_k,
):
    """
    Apply TTT persistence to a recommended target.
    """
    if recommended_target == serving_index:
        return (
            serving_index,
            None,
            0,
            "stay",
        )

    if pending_target == recommended_target:
        pending_count += 1
    else:
        pending_target = recommended_target
        pending_count = 1

    if pending_count >= ttt_k:
        new_serving = recommended_target

        return (
            new_serving,
            None,
            0,
            "handover",
        )

    return (
        serving_index,
        pending_target,
        pending_count,
        "stay",
    )


def run_baseline_fold(
    fold,
    fold_index,
    baseline_name,
    model_name=None,
    initial_serving_index=4,
):
    """
    Evaluate one baseline on one LOTO fold.

    Supported baselines:
      - Event A3
      - Persistence
      - Fixed DP–TTT + forecaster
    """
    num_epochs = fold["test_target_db"].shape[0]

    serving_index = int(initial_serving_index)
    pending_target = None
    pending_count = 0
    handover_count = 0

    records = []

    if baseline_name.startswith("Fixed DP--TTT"):
        fixed_planner = RiskAwarePlanningAgent(
            horizon_options=(1,),
            beta_options=(0.5,),
            failure_penalty_options=(2.0,),
            ttt_options=(2,),
            handover_cost_db=2.0,
            reliability_penalty_db=0.0,
        )

        full_batch = get_forecast_batch(
            fold,
            "test",
            model_name,
        )

    for epoch_index in range(num_epochs):
        current_serving = serving_index

        # ---------------------------------------------------------
        # Event A3 baseline
        # ---------------------------------------------------------
        if baseline_name == "Event A3":
            observed_power = fold[
                "test_observed_power_db"
            ][epoch_index]

            candidate_indices = [
                index
                for index in range(NUM_GNBS)
                if index != serving_index
            ]

            strongest_candidate = max(
                candidate_indices,
                key=lambda index: observed_power[index],
            )

            a3_condition = (
                observed_power[strongest_candidate]
                - observed_power[serving_index]
                >= 2.0
            )

            if a3_condition:
                recommended_target = (
                    strongest_candidate
                )
            else:
                recommended_target = serving_index

            (
                new_serving,
                pending_target,
                pending_count,
                action,
            ) = apply_ttt_decision(
                serving_index=serving_index,
                recommended_target=recommended_target,
                pending_target=pending_target,
                pending_count=pending_count,
                ttt_k=2,
            )

        # ---------------------------------------------------------
        # Persistence baseline
        # ---------------------------------------------------------
        elif baseline_name == "Persistence":
            recommended_target = serving_index
            new_serving = serving_index
            action = "stay"
            pending_target = None
            pending_count = 0

        # ---------------------------------------------------------
        # Fixed DP–TTT baseline
        # ---------------------------------------------------------
        elif baseline_name.startswith("Fixed DP--TTT"):
            epoch_batch = slice_forecast_batch(
                full_batch,
                epoch_index,
            )

            fixed_plan = fixed_planner.plan(
                batch=epoch_batch,
                serving_index=serving_index,
                reliability=np.ones(NUM_GNBS),
                decision_epoch=epoch_index,
            )

            recommended_target = int(
                fixed_plan["recommended_target"]
            )

            (
                new_serving,
                pending_target,
                pending_count,
                action,
            ) = apply_ttt_decision(
                serving_index=serving_index,
                recommended_target=recommended_target,
                pending_target=pending_target,
                pending_count=pending_count,
                ttt_k=2,
            )

        else:
            raise ValueError(
                f"Unknown baseline: {baseline_name}"
            )

        if action == "handover":
            handover_count += 1

        records.append({
            "fold": fold_index,
            "epoch": epoch_index,
            "serving_before": serving_index,
            "serving_after": new_serving,
            "recommended_target": recommended_target,
            "executed_action": action,
            "handover": int(action == "handover"),
        })

        serving_index = new_serving

    trajectory_df = pd.DataFrame(records)

    safe_summary = recompute_safe_summary(
        fold=fold,
        trajectory_df=trajectory_df,
    )

    safe_summary["fold"] = fold_index
    safe_summary["baseline"] = baseline_name

    if model_name is not None:
        safe_summary["model"] = model_name

    return {
        "summary": safe_summary,
        "trajectory": trajectory_df,
    }


# Evaluate all baselines across all folds
baseline_runs = {}
baseline_summaries = []
baseline_trajectories = []

baseline_names = [
    ("Event A3", None),
    ("Persistence", None),
]

for model_name in FORECAST_MODELS:
    baseline_names.append(
        (
            f"Fixed DP--TTT + {model_name}",
            model_name,
        )
    )

for baseline_name, model_name in baseline_names:
    print(f"\nEvaluating {baseline_name}...")

    baseline_runs[baseline_name] = []

    for fold_index, fold in enumerate(fold_data):
        result = run_baseline_fold(
            fold=fold,
            fold_index=fold_index,
            baseline_name=baseline_name,
            model_name=model_name,
            initial_serving_index=4,
        )

        baseline_runs[baseline_name].append(result)
        baseline_summaries.append(result["summary"])

        trajectory = result["trajectory"].copy()
        trajectory["baseline"] = baseline_name

        if model_name is not None:
            trajectory["model"] = model_name

        baseline_trajectories.append(trajectory)

baseline_summary_df = pd.DataFrame(
    baseline_summaries
)

baseline_trajectory_df = pd.concat(
    baseline_trajectories,
    ignore_index=True,
)

print("\nBaseline evaluation completed.")
display(
    baseline_summary_df[
        [
            "baseline",
            "model",
            "fold",
            "handovers",
            "first_step_validity",
            "invalid_action_rate",
            "usable_oracle_gap_db",
        ]
    ].round(4)
)

In [ ]:
# Cell 14 — Aggregate six-fold comparison

def aggregate_method_results(
    summary_df,
    method_column="baseline",
):
    """
    Compute mean and standard deviation across LOTO folds.
    """
    metric_columns = [
        "handovers",
        "first_step_validity",
        "invalid_action_rate",
        "usable_oracle_gap_db",
        "mean_selected_power_valid_db",
    ]

    rows = []

    for method_name, group in summary_df.groupby(
        method_column,
        dropna=False,
    ):
        row = {
            "method": method_name,
            "folds": len(group),
        }

        for metric in metric_columns:
            values = group[metric].astype(float)

            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_std"] = values.std(
                ddof=1
            )

        rows.append(row)

    return pd.DataFrame(rows)


# Aggregate all fixed baselines
baseline_aggregate_df = aggregate_method_results(
    baseline_summary_df,
    method_column="baseline",
)

# Add the improved three-agent system
three_agent_aggregate = {
    "method": "Three-agent system",
    "folds": len(robust_summary_df),
}

for metric in [
    "handovers",
    "first_step_validity",
    "invalid_action_rate",
    "usable_oracle_gap_db",
    "mean_selected_power_valid_db",
]:
    values = robust_summary_df[metric].astype(float)

    three_agent_aggregate[
        f"{metric}_mean"
    ] = values.mean()

    three_agent_aggregate[
        f"{metric}_std"
    ] = values.std(ddof=1)

comparison_summary_df = pd.concat(
    [
        baseline_aggregate_df,
        pd.DataFrame([three_agent_aggregate]),
    ],
    ignore_index=True,
)

# Recommended display order
display_order = [
    "Event A3",
    "Persistence",
    "Fixed DP--TTT + persistence",
    "Fixed DP--TTT + deterministic",
    "Fixed DP--TTT + independent",
    "Fixed DP--TTT + joint",
    "Fixed DP--TTT + hurdle_joint",
    "Three-agent system",
]

comparison_summary_df["method"] = pd.Categorical(
    comparison_summary_df["method"],
    categories=display_order,
    ordered=True,
)

comparison_summary_df = (
    comparison_summary_df
    .sort_values("method")
    .reset_index(drop=True)
)

print("Six-fold comparison:")
display(
    comparison_summary_df[
        [
            "method",
            "handovers_mean",
            "handovers_std",
            "first_step_validity_mean",
            "first_step_validity_std",
            "invalid_action_rate_mean",
            "usable_oracle_gap_db_mean",
            "usable_oracle_gap_db_std",
        ]
    ].round(4)
)

# Save aggregate comparison
comparison_path = (
    REPORT_DIR / "robust_method_comparison.csv"
)

comparison_summary_df.to_csv(
    comparison_path,
    index=False,
)

print(f"\nSaved comparison to:\n{comparison_path}")

In [ ]:
# Cell 15 — Route-progress curves across the six LOTO folds

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def enrich_trajectory_metrics(
    trajectory_df,
    fold,
    rolling_window=15,
):
    """
    Add validity, usable oracle gap, cumulative handovers,
    and rolling metrics to a policy trajectory.
    """
    df = trajectory_df.copy().reset_index(drop=True)

    valid_values = []
    gap_values = []
    selected_power_values = []

    for epoch_index, row in df.iterrows():
        selected_index = int(
            row["serving_after"]
        )

        target_db = fold[
            "test_target_db"
        ][epoch_index, 0, :]

        validity = fold[
            "test_Yv"
        ][epoch_index, 0, :].astype(bool)

        selected_valid = bool(
            validity[selected_index]
        )

        valid_values.append(
            float(selected_valid)
        )

        if selected_valid:
            selected_power = float(
                target_db[selected_index]
            )
            oracle_power = float(
                np.max(target_db[validity])
            )

            selected_power_values.append(
                selected_power
            )
            gap_values.append(
                oracle_power - selected_power
            )
        else:
            selected_power_values.append(np.nan)
            gap_values.append(np.nan)

    df["selected_valid"] = valid_values
    df["usable_oracle_gap_db"] = gap_values
    df["selected_power_valid_db"] = (
        selected_power_values
    )

    df["cumulative_handovers"] = (
        df["handover"].cumsum()
    )

    df["rolling_validity"] = (
        df["selected_valid"]
        .rolling(
            rolling_window,
            min_periods=1,
        )
        .mean()
    )

    df["rolling_oracle_gap_db"] = (
        df["usable_oracle_gap_db"]
        .rolling(
            rolling_window,
            min_periods=1,
        )
        .mean()
    )

    df["progress"] = np.linspace(
        0.0,
        1.0,
        len(df),
    )

    return df


# Construct enriched trajectories for every method
method_trajectories = {}

for baseline_name, fold_runs in baseline_runs.items():
    method_trajectories[baseline_name] = []

    for fold_index, result in enumerate(fold_runs):
        enriched = enrich_trajectory_metrics(
            trajectory_df=result["trajectory"],
            fold=fold_data[fold_index],
        )

        enriched["fold"] = fold_index
        enriched["method"] = baseline_name

        method_trajectories[
            baseline_name
        ].append(enriched)

# Add robust three-agent trajectories
method_trajectories["Three-agent system"] = []

for fold_index, fold_run in (
    all_three_agent_runs.items()
):
    enriched = enrich_trajectory_metrics(
        trajectory_df=fold_run["trajectory"],
        fold=fold_data[fold_index],
    )

    enriched["fold"] = fold_index
    enriched["method"] = "Three-agent system"

    method_trajectories[
        "Three-agent system"
    ].append(enriched)


def interpolate_metric(
    fold_frames,
    metric,
    grid,
):
    """
    Interpolate one metric for each fold onto
    a common normalized route-progress grid.
    """
    interpolated = []

    for df in fold_frames:
        x = df["progress"].to_numpy()
        y = df[metric].to_numpy()

        valid = np.isfinite(y)

        if valid.sum() < 2:
            interpolated.append(
                np.full_like(grid, np.nan)
            )
            continue

        interpolated.append(
            np.interp(
                grid,
                x[valid],
                y[valid],
            )
        )

    return np.asarray(interpolated)


def plot_mean_std_curve(
    ax,
    metric,
    ylabel,
    title,
    methods,
    grid,
):
    colors = {
        "Event A3": "#1f77b4",
        "Fixed DP--TTT + independent": "#2ca02c",
        "Fixed DP--TTT + joint": "#9467bd",
        "Fixed DP--TTT + hurdle_joint": "#ff7f0e",
        "Three-agent system": "#d62728",
    }

    for method in methods:
        curves = interpolate_metric(
            method_trajectories[method],
            metric,
            grid,
        )

        mean_curve = np.nanmean(
            curves,
            axis=0,
        )

        std_curve = np.nanstd(
            curves,
            axis=0,
        )

        color = colors[method]

        ax.plot(
            grid,
            mean_curve,
            label=method,
            color=color,
            linewidth=2.0,
        )

        ax.fill_between(
            grid,
            mean_curve - std_curve,
            mean_curve + std_curve,
            color=color,
            alpha=0.14,
        )

    ax.set_xlabel("Normalized route progress")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(
        True,
        alpha=0.25,
        linewidth=0.7,
    )
    ax.set_xlim(0.0, 1.0)


plot_methods = [
    "Event A3",
    "Fixed DP--TTT + independent",
    "Fixed DP--TTT + joint",
    "Fixed DP--TTT + hurdle_joint",
    "Three-agent system",
]

progress_grid = np.linspace(
    0.0,
    1.0,
    101,
)

# Improved figure layout

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4.5),
    constrained_layout=False,
)

plot_mean_std_curve(
    ax=axes[0],
    metric="cumulative_handovers",
    ylabel="Cumulative handovers",
    title="Handover activity",
    methods=plot_methods,
    grid=progress_grid,
)

plot_mean_std_curve(
    ax=axes[1],
    metric="rolling_validity",
    ylabel="Rolling validity",
    title="Link-validity behavior",
    methods=plot_methods,
    grid=progress_grid,
)

plot_mean_std_curve(
    ax=axes[2],
    metric="rolling_oracle_gap_db",
    ylabel="Rolling usable oracle gap (dB)",
    title="Target-selection alignment",
    methods=plot_methods,
    grid=progress_grid,
)

# Improve axis limits
axes[1].set_ylim(0.0, 1.05)
axes[2].set_ylim(bottom=0.0)

for ax in axes:
    ax.set_xlabel("Normalized route progress")
    ax.grid(True, alpha=0.25, linewidth=0.7)

# Highlight the proposed method
for ax in axes:
    for line in ax.lines:
        if line.get_label() == "Three-agent system":
            line.set_linewidth(2.8)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.subplots_adjust(
    left=0.05,
    right=0.99,
    top=0.88,
    bottom=0.27,
    wspace=0.28,
)

figures_path = (
    FIGURES_DIR / "robust_route_progress_curves_clean.pdf"
)

figures_png_path = (
    FIGURES_DIR / "robust_route_progress_curves_clean.png"
)

fig.savefig(
    figures_path,
    bbox_inches="tight",
)

fig.savefig(
    figures_png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved:\n{figures_path}")
print(figures_png_path)

In [ ]:
# Cell 16 — Replot using the name "Proposed Agentic Twin"

PROPOSED_NAME = "Proposed Agentic Twin"

# Rename the trajectory entry
if "Three-agent system" in method_trajectories:
    method_trajectories[PROPOSED_NAME] = (
        method_trajectories.pop("Three-agent system")
    )

plot_methods_agentic = [
    "Event A3",
    "Fixed DP--TTT + independent",
    "Fixed DP--TTT + joint",
    "Fixed DP--TTT + hurdle_joint",
    PROPOSED_NAME,
]

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 13,
    "axes.labelsize": 16,
    "axes.titlesize": 17,
    "legend.fontsize": 11,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

colors = {
    "Event A3": "#1f77b4",
    "Fixed DP--TTT + independent": "#2ca02c",
    "Fixed DP--TTT + joint": "#9467bd",
    "Fixed DP--TTT + hurdle_joint": "#ff7f0e",
    PROPOSED_NAME: "#d62728",
}

markers = {
    "Event A3": "o",
    "Fixed DP--TTT + independent": "s",
    "Fixed DP--TTT + joint": "^",
    "Fixed DP--TTT + hurdle_joint": "D",
    PROPOSED_NAME: "*",
}

curve_specs = [
    {
        "metric": "cumulative_handovers",
        "ylabel": "Cumulative handovers",
        "title": "Handover activity",
        "filename": "proposed_agentic_twin_handover_activity",
        "ylim": None,
    },
    {
        "metric": "rolling_validity",
        "ylabel": "Rolling validity",
        "title": "Link-validity behavior",
        "filename": "proposed_agentic_twin_link_validity",
        "ylim": (0.0, 1.05),
    },
    {
        "metric": "rolling_oracle_gap_db",
        "ylabel": "Rolling usable oracle gap (dB)",
        "title": "Target-selection alignment",
        "filename": "proposed_agentic_twin_target_alignment",
        "ylim": (0.0, None),
    },
]

marker_positions = np.linspace(
    0,
    len(progress_grid) - 1,
    11,
).astype(int)

for spec in curve_specs:
    fig, ax = plt.subplots(
        figsize=(7.8, 5.4),
        constrained_layout=False,
    )

    for method in plot_methods_agentic:
        curves = interpolate_metric(
            method_trajectories[method],
            spec["metric"],
            progress_grid,
        )

        mean_curve = np.nanmean(
            curves,
            axis=0,
        )

        std_curve = np.nanstd(
            curves,
            axis=0,
        )

        lower_curve = np.maximum(
            mean_curve - std_curve,
            0.0,
        )

        upper_curve = mean_curve + std_curve

        if spec["metric"] == "rolling_validity":
            upper_curve = np.minimum(
                upper_curve,
                1.0,
            )

        is_proposed = method == PROPOSED_NAME

        ax.plot(
            progress_grid,
            mean_curve,
            color=colors[method],
            linewidth=3.2 if is_proposed else 2.0,
            marker=markers[method],
            markersize=8 if is_proposed else 5.5,
            markevery=marker_positions.tolist(),
            markeredgecolor="white",
            markeredgewidth=0.7,
            label=method,
            zorder=4 if is_proposed else 3,
        )

        ax.fill_between(
            progress_grid,
            lower_curve,
            upper_curve,
            color=colors[method],
            alpha=0.12,
            linewidth=0,
            zorder=1,
        )

    ax.set_xlabel("Normalized route progress")
    ax.set_ylabel(spec["ylabel"])
    ax.set_title(spec["title"], pad=12)
    ax.set_xlim(0.0, 1.0)

    if spec["ylim"] is not None:
        ax.set_ylim(*spec["ylim"])

    ax.grid(
        True,
        alpha=0.25,
        linewidth=0.8,
    )
    ax.set_axisbelow(True)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        frameon=False,
    )

    fig.subplots_adjust(
        left=0.15,
        right=0.97,
        top=0.88,
        bottom=0.32,
    )

    pdf_path = FIGURES_DIR / f"{spec['filename']}.pdf"
    png_path = FIGURES_DIR / f"{spec['filename']}.png"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    fig.savefig(
        png_path,
        dpi=400,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(f"Saved PDF: {pdf_path}")
    print(f"Saved PNG: {png_path}")

In [ ]:
# Cell 17 — Clear bar-plot comparison

import textwrap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROPOSED_NAME = "Proposed Agentic Twin"

bar_df = comparison_summary_df.copy()
bar_df["method"] = bar_df["method"].astype(str)

bar_df["method"] = bar_df["method"].replace({
    "Three-agent system": PROPOSED_NAME
})

# Remove duplicate persistence policy
bar_df = bar_df[
    bar_df["method"] != "Fixed DP--TTT + persistence"
].copy()

method_order = [
    "Event A3",
    "Persistence",
    "Fixed DP--TTT + deterministic",
    "Fixed DP--TTT + independent",
    "Fixed DP--TTT + joint",
    "Fixed DP--TTT + hurdle_joint",
    PROPOSED_NAME,
]

bar_df["method"] = pd.Categorical(
    bar_df["method"],
    categories=method_order,
    ordered=True,
)

bar_df = (
    bar_df
    .sort_values("method")
    .reset_index(drop=True)
)

short_labels = [
    "Event A3",
    "Persistence",
    "DP--TTT\n+ Deterministic",
    "DP--TTT\n+ Independent",
    "DP--TTT\n+ Joint",
    "DP--TTT\n+ Hurdle",
    "Proposed\nAgentic Twin",
]

colors = [
    "#8da0cb",
    "#bdbdbd",
    "#66c2a5",
    "#2ca25f",
    "#756bb1",
    "#fdae6b",
    "#d62728",
]

x = np.arange(len(bar_df))
width = 0.70

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 6.2),
)

metrics = [
    {
        "mean": "handovers_mean",
        "std": "handovers_std",
        "ylabel": "Mean handovers per trajectory",
        "title": "(a) Handover activity",
        "lower_better": True,
    },
    {
        "mean": "first_step_validity_mean",
        "std": "first_step_validity_std",
        "ylabel": "First-step validity",
        "title": "(b) Link validity",
        "lower_better": False,
    },
    {
        "mean": "usable_oracle_gap_db_mean",
        "std": "usable_oracle_gap_db_std",
        "ylabel": "Usable oracle gap (dB)",
        "title": "(c) Target-selection alignment",
        "lower_better": True,
    },
]

for ax, metric in zip(axes, metrics):
    means = bar_df[metric["mean"]].to_numpy()
    stds = bar_df[metric["std"]].to_numpy()

    bars = ax.bar(
        x,
        means,
        width=width,
        yerr=stds,
        capsize=4,
        color=colors,
        edgecolor="black",
        linewidth=0.7,
        error_kw={
            "elinewidth": 1.1,
            "ecolor": "#333333",
        },
    )

    # Emphasize Proposed Agentic Twin
    bars[-1].set_linewidth(1.8)
    bars[-1].set_edgecolor("#8b0000")

    ax.set_title(
        metric["title"],
        fontsize=16,
        pad=12,
    )

    ax.set_ylabel(
        metric["ylabel"],
        fontsize=14,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        short_labels,
        rotation=35,
        ha="right",
        fontsize=11,
    )

    ax.tick_params(
        axis="y",
        labelsize=12,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)

    # Add values above bars
    for index, (mean, std) in enumerate(
        zip(means, stds)
    ):
        if metric["mean"] == "first_step_validity_mean":
            label = f"{mean:.3f}"
        elif metric["mean"] == "usable_oracle_gap_db_mean":
            label = f"{mean:.2f}"
        else:
            label = f"{mean:.1f}"

        ax.text(
            index,
            mean + std + 0.015 * max(means),
            label,
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold" if index == len(means) - 1 else "normal",
        )

    if metric["mean"] == "first_step_validity_mean":
        ax.set_ylim(0.75, 1.03)

fig.suptitle(
    "Predictive handover performance across six LOTO folds",
    fontsize=18,
    fontweight="bold",
    y=1.02,
)

fig.text(
    0.5,
    -0.03,
    "Bars show mean values; whiskers show standard deviation. "
    "Lower is better for handovers and oracle gap, while higher is better for validity.",
    ha="center",
    fontsize=12,
)

fig.subplots_adjust(
    left=0.06,
    right=0.99,
    top=0.86,
    bottom=0.27,
    wspace=0.32,
)

bar_pdf = (
    FIGURES_DIR
    / "proposed_agentic_twin_bar_balance.pdf"
)

bar_png = (
    FIGURES_DIR
    / "proposed_agentic_twin_bar_balance.png"
)

fig.savefig(
    bar_pdf,
    bbox_inches="tight",
)

fig.savefig(
    bar_png,
    dpi=400,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved PDF: {bar_pdf}")
print(f"Saved PNG: {bar_png}")

In [ ]:
# Cell 18 — Save each bar plot as a separate PDF

import matplotlib.pyplot as plt
import numpy as np

separate_bar_specs = [
    {
        "mean": "handovers_mean",
        "std": "handovers_std",
        "ylabel": "Mean handovers per trajectory",
        "title": "Handover activity",
        "filename": "proposed_agentic_twin_handovers",
        "ylim": None,
    },
    {
        "mean": "first_step_validity_mean",
        "std": "first_step_validity_std",
        "ylabel": "First-step validity",
        "title": "Link validity",
        "filename": "proposed_agentic_twin_validity",
        "ylim": (0.75, 1.03),
    },
    {
        "mean": "usable_oracle_gap_db_mean",
        "std": "usable_oracle_gap_db_std",
        "ylabel": "Usable oracle gap (dB)",
        "title": "Target-selection alignment",
        "filename": "proposed_agentic_twin_oracle_gap",
        "ylim": None,
    },
]

for spec in separate_bar_specs:
    means = bar_df[spec["mean"]].to_numpy()
    stds = bar_df[spec["std"]].to_numpy()

    fig, ax = plt.subplots(
        figsize=(9.0, 6.4)
    )

    bars = ax.bar(
        x,
        means,
        width=0.68,
        yerr=stds,
        capsize=5,
        color=colors,
        edgecolor="black",
        linewidth=0.8,
        error_kw={
            "elinewidth": 1.2,
            "ecolor": "#333333",
        },
    )

    # Highlight Proposed Agentic Twin
    bars[-1].set_linewidth(2.0)
    bars[-1].set_edgecolor("#8b0000")

    ax.set_title(
        spec["title"],
        fontsize=19,
        fontweight="bold",
        pad=15,
    )

    ax.set_ylabel(
        spec["ylabel"],
        fontsize=16,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        short_labels,
        rotation=35,
        ha="right",
        fontsize=12,
    )

    ax.tick_params(
        axis="y",
        labelsize=13,
    )

    if spec["ylim"] is not None:
        ax.set_ylim(*spec["ylim"])

    ax.grid(
        axis="y",
        alpha=0.25,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)

    # Add numerical values
    value_range = max(means) - min(means)
    label_offset = (
        0.025 * max(means)
        if value_range > 0
        else 0.02
    )

    for index, (mean, std) in enumerate(
        zip(means, stds)
    ):
        if spec["mean"] == "first_step_validity_mean":
            label = f"{mean:.3f}"
        elif spec["mean"] == "usable_oracle_gap_db_mean":
            label = f"{mean:.2f}"
        else:
            label = f"{mean:.1f}"

        ax.text(
            index,
            mean + std + label_offset,
            label,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight=(
                "bold"
                if index == len(means) - 1
                else "normal"
            ),
        )

   
    fig.subplots_adjust(
        left=0.12,
        right=0.98,
        top=0.88,
        bottom=0.29,
    )

    pdf_path = (
        FIGURES_DIR
        / f"{spec['filename']}.pdf"
    )

    png_path = (
        FIGURES_DIR
        / f"{spec['filename']}.png"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    fig.savefig(
        png_path,
        dpi=400,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(f"Saved PDF: {pdf_path}")
    print(f"Saved PNG: {png_path}")

In [ ]:
# Cell 19 — Runtime-complexity comparison

import time
import gc

RUNTIME_REPEATS = 3
runtime_records = []

runtime_methods = [
    ("Event A3", None),
    ("Persistence", None),
    ("Fixed DP--TTT + deterministic", "deterministic"),
    ("Fixed DP--TTT + independent", "independent"),
    ("Fixed DP--TTT + joint", "joint"),
    ("Fixed DP--TTT + hurdle_joint", "hurdle_joint"),
    ("Proposed Agentic Twin", None),
]

for method_name, model_name in runtime_methods:
    print(f"Timing {method_name}...")

    for fold_index, fold in enumerate(fold_data):
        elapsed_times = []

        # Warm-up run
        if method_name == "Proposed Agentic Twin":
            run_three_agent_fold(
                fold=fold,
                fold_index=fold_index,
                initial_serving_index=4,
            )
        else:
            run_baseline_fold(
                fold=fold,
                fold_index=fold_index,
                baseline_name=method_name,
                model_name=model_name,
                initial_serving_index=4,
            )

        # Timed runs
        for _ in range(RUNTIME_REPEATS):
            gc.collect()

            start_time = time.perf_counter()

            if method_name == "Proposed Agentic Twin":
                run_three_agent_fold(
                    fold=fold,
                    fold_index=fold_index,
                    initial_serving_index=4,
                )
            else:
                run_baseline_fold(
                    fold=fold,
                    fold_index=fold_index,
                    baseline_name=method_name,
                    model_name=model_name,
                    initial_serving_index=4,
                )

            elapsed = time.perf_counter() - start_time
            elapsed_times.append(elapsed)

        num_epochs = fold[
            "test_target_db"
        ].shape[0]

        runtime_records.append({
            "method": method_name,
            "fold": fold_index,
            "total_time_s": np.mean(elapsed_times),
            "time_std_s": np.std(elapsed_times),
            "time_per_epoch_ms": (
                1000.0
                * np.mean(elapsed_times)
                / num_epochs
            ),
            "epochs": num_epochs,
        })

runtime_fold_df = pd.DataFrame(runtime_records)

runtime_summary_df = (
    runtime_fold_df
    .groupby("method", as_index=False)
    .agg(
        total_time_mean_s=(
            "total_time_s",
            "mean",
        ),
        total_time_std_s=(
            "total_time_s",
            "std",
        ),
        decision_time_mean_ms=(
            "time_per_epoch_ms",
            "mean",
        ),
        decision_time_std_ms=(
            "time_per_epoch_ms",
            "std",
        ),
    )
)

runtime_summary_df["relative_to_event_a3"] = (
    runtime_summary_df["decision_time_mean_ms"]
    / runtime_summary_df.loc[
        runtime_summary_df["method"] == "Event A3",
        "decision_time_mean_ms",
    ].iloc[0]
)

runtime_summary_df = runtime_summary_df.sort_values(
    "decision_time_mean_ms"
).reset_index(drop=True)

print("\nRuntime comparison:")
display(
    runtime_summary_df.round({
        "total_time_mean_s": 4,
        "total_time_std_s": 4,
        "decision_time_mean_ms": 4,
        "decision_time_std_ms": 4,
        "relative_to_event_a3": 2,
    })
)

# Save CSV
runtime_csv_path = (
    REPORT_DIR / "policy_runtime_complexity.csv"
)

runtime_summary_df.to_csv(
    runtime_csv_path,
    index=False,
)

# Save LaTeX table
runtime_latex_path = (
    REPORT_DIR / "policy_runtime_complexity.tex"
)

runtime_table = runtime_summary_df.copy()
runtime_table.columns = [
    "Method",
    "Total time mean [s]",
    "Total time std [s]",
    "Decision time mean [ms]",
    "Decision time std [ms]",
    "Relative to Event A3",
]

runtime_table.to_latex(
    runtime_latex_path,
    index=False,
    float_format="%.3f",
)

print(f"\nSaved CSV: {runtime_csv_path}")
print(f"Saved LaTeX table: {runtime_latex_path}")

## 5. Publication-quality figures and reports

This final section loads the saved forecasting and policy results, recomputes the common benchmark summaries, and produces the figures used for the manuscript.

The reported policy metrics are:

- handover activity;
- first-step ray-path availability;
- sustained validity; and
- usable target-selection oracle gap.

The normalized comparison should be interpreted as three separate criteria. It does not form a hidden weighted aggregate score.


## 5.1 Generate final figures and summary tables


# 04 — Publication-Quality Figures with Full Benchmark Names

This notebook **only reads results that already exist** in
`improved_experiment_v2/results/` (written by notebooks 02 and 03). Nothing is
retrained and no ray tracing is repeated. The per-fold policy metrics are
recomputed from the saved fold predictions (a few seconds) using exactly the
same policy code and settings as notebook 03, so the numbers match the paper.

Compared with the earlier all-bar-chart figures, the plots here

* use the **full benchmark names** (Persistence, Deterministic GRU, Independent
  Probabilistic GRU, Joint Probabilistic GRU, Hurdle-Aware Joint GRU) and full
  policy names instead of code identifiers such as `hurdle_joint`;
* use a plot type suited to each quantity: a **strip and mean plot** showing every
  fold for forecast error, **dumbbell plots** for the legacy versus proposed
  policy comparison, a **trade-off scatter** with arrows, and a **per-fold heatmap**;
* share one aligned label column, drop long in-figure titles (captions carry
  them), and use serif fonts, colour-blind-safe colours, and vector PDF output
  sized for IEEE double-column layout.

Outputs go to `improved_experiment_v2/figures_pro/`.


In [ ]:
from pathlib import Path
import csv
import json
import textwrap

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

PROJECT_ROOT = Path.home() / "wcnc_project"
V2_RESULTS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "results"
FIG_DIR = PROJECT_ROOT / "improved_experiment_v2" / "figures_pro"
FIG_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.dpi": 300,
})

# ---------- full names ----------
MODEL_ORDER = ["persistence", "deterministic", "independent", "joint", "hurdle_joint"]
FULL_NAME = {
    "persistence": "Persistence",
    "deterministic": "Deterministic GRU",
    "independent": "Independent Probabilistic GRU",
    "joint": "Joint Probabilistic GRU",
    "hurdle_joint": "Hurdle-Aware Joint GRU",
}
POLICY_ORDER = ["legacy_margin", "dp_ttt"]
POLICY_NAME = {
    "legacy_margin": "Legacy average-score margin policy",
    "dp_ttt": "Proposed DP-TTT policy",
}
# colour-blind-safe (Okabe-Ito): grey baselines, vermilion = non-hurdle joint, blue = proposed
COLOR = {
    "persistence": "#7F7F7F", "deterministic": "#009E73", "independent": "#E69F00",
    "joint": "#D55E00", "hurdle_joint": "#0072B2",
}

def wrap(name, width=17):
    return textwrap.fill(name, width=width)

Y_LABELS = [wrap(FULL_NAME[m]) for m in MODEL_ORDER]
print("Figures will be written to:", FIG_DIR)


In [ ]:
# ------------------------------------------------------------
# Load per-fold forecast metrics saved by notebook 02
# ------------------------------------------------------------
with open(V2_RESULTS_DIR / "forecaster_loto_summary.json") as f:
    forecast_summary = json.load(f)

NUM_FOLDS = forecast_summary["num_folds"]
mae_folds = {
    m: np.array([forecast_summary["per_fold_results"][str(k)][m]["masked_mae_db"]
                 for k in range(NUM_FOLDS)], dtype=float)
    for m in MODEL_ORDER
}
print("Loaded masked MAE for", NUM_FOLDS, "folds")


In [ ]:
# ------------------------------------------------------------
# Recompute per-fold policy metrics from saved predictions.
# Policy code and settings are identical to notebook 03.
# Two extra valid-link metrics are added (G_valid and I_invalid).
# ------------------------------------------------------------
RISK_AVERSION = 0.5
HANDOVER_MARGIN_DB = 2.0
HANDOVER_COST_DB = 2.0
TTT_K = 2
LINK_GATE_THRESHOLD = None


def build_score_matrices(mean_db, std_db=None, link_prob=None, risk_aversion=0.5, link_gate_threshold=None):
    score = mean_db.copy()
    if std_db is not None:
        score = score - risk_aversion * std_db
    if link_prob is not None:
        score = score + 10.0 * np.log1p(link_prob) - 10.0 * np.log1p(np.ones_like(link_prob))
        if link_gate_threshold is not None:
            score = np.where(link_prob < link_gate_threshold, -1e6, score)
    return score


def run_legacy_margin_policy(score_avg, current_power_db, margin_db=2.0):
    raw_actions = np.argmax(score_avg, axis=1).astype(np.int16)
    actions = np.zeros(len(score_avg), dtype=np.int16)
    serving = int(np.argmax(current_power_db[0]))
    for i in range(len(score_avg)):
        cand = int(raw_actions[i])
        if cand != serving and score_avg[i, cand] - score_avg[i, serving] >= margin_db:
            serving = cand
        actions[i] = serving
    return actions


def _dp_first_action(step_scores, current_serving, handover_cost):
    horizon, num_gnbs = step_scores.shape
    value_next = np.zeros(num_gnbs)
    first_policy = None
    for h in range(horizon - 1, -1, -1):
        value_cur = np.zeros(num_gnbs)
        action_cur = np.zeros(num_gnbs, dtype=np.int16)
        for g_prev in range(num_gnbs):
            best_v, best_g = -np.inf, g_prev
            for g_cur in range(num_gnbs):
                cost = handover_cost if g_cur != g_prev else 0.0
                v = step_scores[h, g_cur] - cost + value_next[g_cur]
                if v > best_v:
                    best_v, best_g = v, g_cur
            value_cur[g_prev] = best_v
            action_cur[g_prev] = best_g
        if h == 0:
            first_policy = action_cur
        value_next = value_cur
    return first_policy[current_serving]


def run_dp_ttt_policy(step_scores, current_power_db, handover_cost_db=2.0, ttt_k=2):
    n = step_scores.shape[0]
    actions = np.zeros(n, dtype=np.int16)
    serving = int(np.argmax(current_power_db[0]))
    pending, streak = None, 0
    for i in range(n):
        rec = _dp_first_action(step_scores[i], serving, handover_cost_db)
        if rec == serving:
            pending, streak = None, 0
        else:
            if rec == pending:
                streak += 1
            else:
                pending, streak = rec, 1
            if streak >= ttt_k:
                serving, pending, streak = rec, None, 0
        actions[i] = serving
    return actions


def evaluate_policy(actions, target_power_db, test_valid, future_best_gnb):
    n, horizon, _ = target_power_db.shape
    oi, si = np.arange(n), np.arange(horizon)
    sel_power = target_power_db[oi[:, None], si[None, :], actions[:, None]]
    sel_valid = test_valid[oi[:, None], si[None, :], actions[:, None]] > 0
    oracle_gap = np.max(target_power_db, axis=2) - sel_power

    # valid-link metrics: gap to the best *usable* link, and how often an unusable link is chosen
    usable_power = np.where(test_valid > 0, target_power_db, -np.inf)
    best_usable = usable_power.max(axis=2)
    ok = np.isfinite(best_usable) & sel_valid
    g_valid = float((best_usable - sel_power)[ok].mean()) if ok.any() else float("nan")

    return {
        "mean_oracle_gap_db": float(oracle_gap.mean()),
        "handover_count": int(np.sum(np.diff(actions) != 0)),
        "valid_link_gap_db": g_valid,
        "invalid_action_rate": float(1.0 - sel_valid.mean()),
    }


per_fold_policy = {m: {p: [] for p in POLICY_ORDER} for m in MODEL_ORDER}
for k in range(NUM_FOLDS):
    with np.load(V2_RESULTS_DIR / f"fold_{k}_summary_predictions.npz", allow_pickle=True) as d:
        files = list(d.files)
        target_db, test_valid, test_best = d["test_target_db"], d["test_Yv"], d["test_best"]
        for m in MODEL_ORDER:
            mean_db = d[f"{m}_mean_db"]
            std_db = d[f"{m}_std_db"] if f"{m}_std_db" in files else None
            link_prob = d[f"{m}_link_prob"] if f"{m}_link_prob" in files else None
            cur = mean_db[:, 0, :]
            score_step = build_score_matrices(mean_db, std_db, link_prob, RISK_AVERSION, LINK_GATE_THRESHOLD)
            a_leg = run_legacy_margin_policy(score_step.mean(axis=1), cur, HANDOVER_MARGIN_DB)
            a_dp = run_dp_ttt_policy(score_step, cur, HANDOVER_COST_DB, TTT_K)
            per_fold_policy[m]["legacy_margin"].append(evaluate_policy(a_leg, target_db, test_valid, test_best))
            per_fold_policy[m]["dp_ttt"].append(evaluate_policy(a_dp, target_db, test_valid, test_best))


def arr(model, policy, metric):
    return np.array([r[metric] for r in per_fold_policy[model][policy]], dtype=float)


def mean_std(x):
    x = x[~np.isnan(x)]
    return float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0


with open(V2_RESULTS_DIR / "policy_perfold_metrics.json", "w", encoding="utf-8") as f:
    json.dump(per_fold_policy, f, indent=2)
print("Recomputed per-fold policy metrics for", NUM_FOLDS, "folds x", len(MODEL_ORDER), "forecasters x 2 policies")


## Figure A — main result (three panels, one shared label column)

* **(a)** masked forecast MAE: every fold is a small dot, the diamond is the fold
  mean, whiskers are one standard deviation, and the stem from zero makes the
  ranking easy to read. A paired t-test of the proposed model against the
  non-hurdle joint model is printed inside the panel.
* **(b, c)** handover count and oracle gap as **dumbbells**: the hollow marker is the
  legacy margin policy, the filled marker is the proposed DP-TTT policy, and the
  connecting line shows the effect of the policy for that forecaster.


In [ ]:
ypos = np.arange(len(MODEL_ORDER))[::-1].astype(float)
rng = np.random.default_rng(7)

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.55), sharey=True,
                         gridspec_kw={"wspace": 0.10, "width_ratios": [1.0, 1.0, 1.0]})

# ---- (a) masked MAE: strip + mean +/- std lollipop ----
ax = axes[0]
for m, y in zip(MODEL_ORDER, ypos):
    vals = mae_folds[m]
    mu, sd = mean_std(vals)
    ax.hlines(y, 0, mu, color=COLOR[m], lw=1.2, alpha=0.55, zorder=1)
    ax.scatter(vals, y + rng.uniform(-0.13, 0.13, len(vals)), s=9, color=COLOR[m],
               alpha=0.45, linewidths=0, zorder=2)
    ax.errorbar(mu, y, xerr=sd, fmt="D", ms=5, color=COLOR[m], mec="black", mew=0.5,
                ecolor="black", elinewidth=0.8, capsize=2, zorder=3)
    ax.text(mu + sd + 0.35, y, f"{mu:.1f}", va="center", fontsize=6.5)
mu_h, _ = mean_std(mae_folds["hurdle_joint"])
mu_j, _ = mean_std(mae_folds["joint"])
t_stat, p_val = stats.ttest_rel(mae_folds["joint"], mae_folds["hurdle_joint"])
ax.text(0.98, 0.04,
        f"vs. Joint GRU: {100 * (1 - mu_h / mu_j):.0f}% lower\npaired $p$ = {p_val:.1e}",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=6.3,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", lw=0.5))
ax.set_xlim(0, max(mean_std(v)[0] + mean_std(v)[1] for v in mae_folds.values()) * 1.25)
ax.set_xlabel("Masked forecast MAE (dB)")
ax.set_title("(a)", loc="left", fontweight="bold")
ax.set_yticks(ypos)
ax.set_yticklabels(Y_LABELS)
ax.get_yticklabels()[-1].set_fontweight("bold")
ax.tick_params(axis="y", length=0)


def dumbbell(ax, metric, xlabel, title, decimals):
    for m, y in zip(MODEL_ORDER, ypos):
        mu_l, sd_l = mean_std(arr(m, "legacy_margin", metric))
        mu_d, sd_d = mean_std(arr(m, "dp_ttt", metric))
        ax.plot([mu_l, mu_d], [y, y], color=COLOR[m], lw=2.0, alpha=0.6, zorder=1,
                solid_capstyle="round")
        ax.errorbar(mu_l, y, xerr=sd_l, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.errorbar(mu_d, y, xerr=sd_d, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.scatter(mu_l, y, s=34, facecolor="white", edgecolor=COLOR[m], linewidths=1.3, zorder=3)
        ax.scatter(mu_d, y, s=34, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)
    ax.set_xlim(left=0)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.tick_params(axis="y", length=0)


dumbbell(axes[1], "handover_count", "Target-cell switches per trajectory", "(b)", 1)
dumbbell(axes[2], "mean_oracle_gap_db", "Mean oracle gap (dB)", "(c)", 1)

policy_handles = [
    Line2D([0], [0], marker="o", ls="", mfc="white", mec="black", mew=1.2, ms=5.5, label=POLICY_NAME["legacy_margin"]),
    Line2D([0], [0], marker="o", ls="", mfc="black", mec="black", ms=5.5, label=POLICY_NAME["dp_ttt"]),
]
fig.legend(handles=policy_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.56, -0.07), handletextpad=0.3, columnspacing=1.8)
fig.subplots_adjust(left=0.20, right=0.99, top=0.93, bottom=0.20)

fig.savefig(FIG_DIR / "fig_main_loto_diagnostics.pdf")
fig.savefig(FIG_DIR / "fig_main_loto_diagnostics.png")
plt.show()


## Figure B — policy trade-off scatter

Each forecaster appears twice, joined by an arrow from the legacy policy (hollow)
to the proposed DP-TTT policy (filled). Lower-left is better on both axes, so the
plot shows in one view whether a policy change trades handover count against
oracle gap. Crosshairs are one standard deviation across folds.


In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.9))
ax.grid(True, axis="both")
for m in MODEL_ORDER:
    x_l, xs_l = mean_std(arr(m, "legacy_margin", "handover_count"))
    y_l, ys_l = mean_std(arr(m, "legacy_margin", "mean_oracle_gap_db"))
    x_d, xs_d = mean_std(arr(m, "dp_ttt", "handover_count"))
    y_d, ys_d = mean_std(arr(m, "dp_ttt", "mean_oracle_gap_db"))
    for (x, y, xs, ys) in [(x_l, y_l, xs_l, ys_l), (x_d, y_d, xs_d, ys_d)]:
        ax.errorbar(x, y, xerr=xs, yerr=ys, fmt="none", ecolor=COLOR[m], elinewidth=0.6, alpha=0.5, capsize=0)
    ax.annotate("", xy=(x_d, y_d), xytext=(x_l, y_l),
                arrowprops=dict(arrowstyle="-|>", color=COLOR[m], lw=1.0, shrinkA=3, shrinkB=3, alpha=0.9))
    ax.scatter(x_l, y_l, s=38, facecolor="white", edgecolor=COLOR[m], linewidths=1.4, zorder=3)
    ax.scatter(x_d, y_d, s=38, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)

ax.set_xlabel("Target-cell switches per trajectory")
ax.set_ylabel("Mean oracle gap (dB)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.text(0.03, 0.03, "better", transform=ax.transAxes, fontsize=7, style="italic", color="0.35")

model_handles = [Line2D([0], [0], marker="s", ls="", color=COLOR[m], ms=5, label=FULL_NAME[m]) for m in MODEL_ORDER]
policy_handles = [
    Line2D([0], [0], marker="o", ls="", mfc="white", mec="black", mew=1.2, ms=5, label="Legacy policy"),
    Line2D([0], [0], marker="o", ls="", mfc="black", mec="black", ms=5, label="Proposed DP-TTT policy"),
]
ax.legend(handles=model_handles + policy_handles, loc="upper center", bbox_to_anchor=(0.5, -0.22),
          ncol=2, frameon=False, fontsize=6.3, columnspacing=1.2, handletextpad=0.3)
fig.savefig(FIG_DIR / "fig_policy_tradeoff_scatter.pdf")
fig.savefig(FIG_DIR / "fig_policy_tradeoff_scatter.png")
plt.show()


## Figure C — per-fold masked MAE heatmap

Shows every held-out trajectory for every forecaster, so a reader can see that the
ranking is consistent across folds and not driven by a single test trajectory.


In [ ]:
data = np.array([mae_folds[m] for m in MODEL_ORDER])          # models x folds
means = data.mean(axis=1, keepdims=True)
full = np.hstack([data, means])

fig, ax = plt.subplots(figsize=(3.5, 2.2))
ax.grid(False)
im = ax.imshow(full, cmap="viridis_r", aspect="auto")
ax.axvline(NUM_FOLDS - 0.5, color="white", lw=2.0)
for i in range(full.shape[0]):
    for j in range(full.shape[1]):
        v = full[i, j]
        r, g, b, _ = im.cmap(im.norm(v))
        col = "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.55 else "white"
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6.5, color=col,
                fontweight="bold" if j == NUM_FOLDS else "normal")
ax.set_xticks(range(NUM_FOLDS + 1))
ax.set_xticklabels([f"F{k}" for k in range(NUM_FOLDS)] + ["Mean"])
ax.set_yticks(range(len(MODEL_ORDER)))
ax.set_yticklabels([wrap(FULL_NAME[m], 20) for m in MODEL_ORDER])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xlabel("Held-out trajectory (fold)")
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("Masked MAE (dB)", fontsize=7)
cb.ax.tick_params(labelsize=6)
fig.savefig(FIG_DIR / "fig_masked_mae_fold_heatmap.pdf")
fig.savefig(FIG_DIR / "fig_masked_mae_fold_heatmap.png")
plt.show()


## Figure D — valid-link policy metrics (supplementary)

The paper notes that the sentinel-inclusive oracle gap can mislead when a policy
selects a link with no usable path. This figure separates the two effects:
**(a)** the gap to the best *usable* link when the chosen link is usable, and
**(b)** how often the policy chooses an unusable link.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.4, 2.4), sharey=True, gridspec_kw={"wspace": 0.10})
ypos = np.arange(len(MODEL_ORDER))[::-1].astype(float)


def dumbbell2(ax, metric, xlabel, title, scale=1.0):
    for m, y in zip(MODEL_ORDER, ypos):
        mu_l, sd_l = mean_std(arr(m, "legacy_margin", metric))
        mu_d, sd_d = mean_std(arr(m, "dp_ttt", metric))
        mu_l, sd_l, mu_d, sd_d = mu_l * scale, sd_l * scale, mu_d * scale, sd_d * scale
        ax.plot([mu_l, mu_d], [y, y], color=COLOR[m], lw=2.0, alpha=0.6, zorder=1, solid_capstyle="round")
        ax.errorbar(mu_l, y, xerr=sd_l, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.errorbar(mu_d, y, xerr=sd_d, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.scatter(mu_l, y, s=34, facecolor="white", edgecolor=COLOR[m], linewidths=1.3, zorder=3)
        ax.scatter(mu_d, y, s=34, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)
    ax.set_xlim(left=0)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.tick_params(axis="y", length=0)


dumbbell2(axes[0], "valid_link_gap_db", "Gap to best usable link (dB)", "(a)")
dumbbell2(axes[1], "invalid_action_rate", "Unusable-link selections (%)", "(b)", scale=100.0)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(Y_LABELS)
axes[0].get_yticklabels()[-1].set_fontweight("bold")
fig.legend(handles=policy_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.58, -0.08), handletextpad=0.3, columnspacing=1.8)
fig.subplots_adjust(left=0.30, right=0.99, top=0.92, bottom=0.22)
fig.savefig(FIG_DIR / "fig_valid_link_policy_metrics.pdf")
fig.savefig(FIG_DIR / "fig_valid_link_policy_metrics.png")
plt.show()


In [ ]:
# ------------------------------------------------------------
# Summary table with full names (CSV) for the paper
# ------------------------------------------------------------
rows = []
for m in MODEL_ORDER:
    mu, sd = mean_std(mae_folds[m])
    for p in POLICY_ORDER:
        rows.append({
            "Forecaster": FULL_NAME[m], "Policy": POLICY_NAME[p],
            "Masked MAE (dB)": f"{mu:.2f} +/- {sd:.2f}",
            "Switches per trajectory": "%.1f +/- %.1f" % mean_std(arr(m, p, "handover_count")),
            "Oracle gap (dB)": "%.2f +/- %.2f" % mean_std(arr(m, p, "mean_oracle_gap_db")),
            "Gap to best usable link (dB)": "%.2f +/- %.2f" % mean_std(arr(m, p, "valid_link_gap_db")),
            "Unusable-link selections (%)": "%.1f +/- %.1f" % tuple(np.array(mean_std(arr(m, p, "invalid_action_rate"))) * 100),
        })

with open(FIG_DIR / "results_summary_full_names.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

for r in rows:
    print(f"{r['Forecaster']:32s} | {r['Policy']:36s} | MAE {r['Masked MAE (dB)']:>14s} | "
          f"switches {r['Switches per trajectory']:>12s} | gap {r['Oracle gap (dB)']:>13s}")
print("\nSaved figures and table to:", FIG_DIR)


## 6. Reproducibility checklist

Before running the notebook, verify:

- the project root points to the local WCNC project;
- the KFUPM Sionna RT scene exists;
- CUDA, Mitsuba, Dr.Jit, Sionna, PyTorch, NumPy, SciPy, pandas, and Matplotlib are installed;
- the synchronized multimodal dataset exists if RF generation is skipped;
- the LOTO fold files exist before training; and
- the saved forecast files exist before running the agentic evaluation.

The final outputs are saved under the project results and figures directories.

**Interpretation limitation:** ray-path availability is a Digital Twin feasibility indicator. Full beamwise SINR, achievable rate, outage, and radio-link-failure evaluation require a future calibrated beam-level replay.
